# 📐 PartScan — iPhone LiDAR → accurate, measured 3D model of a part

Scan one small part (a bracket, housing, fixture, casting…) with an **iPhone 17 Pro / Pro Max** and get:

* a **true-scale 3D mesh** — `STL` / `OBJ` / `PLY` in millimetres plus `GLB`
* **dimensioned views** (top · front · side · isometric)
* automatic **measurements** — length × width × height, oriented bounding box, volume, area, mass,
  **hole diameters and positions**, **wall thicknesses**, **step heights**
* an **interactive 3D viewer** (phone friendly) — tap two points to measure anything
* an HTML **report** with capture-quality checks and an optional caliper cross-check

### How the accuracy is achieved
| Step | What | Why |
|---|---|---|
| Scale & poses | A printed **ChArUco board** under the part + bundle adjustment (with lens-distortion estimation) | Phone tracking alone is typically 0.5–2 % off in scale; the board fixes scale and gives drift-free camera poses |
| Photogrammetry | SIFT tie points triangulated across views | Independent, multi-view geometry to correct every depth map |
| AI depth | **Prompt Depth Anything** (CVPR 2025) — RGB + the phone's 256×192 LiDAR map → sharp metric depth | Raw iPhone LiDAR is ~±1 cm and blind to small details |
| AI silhouettes | **SAM 2.1**, prompted automatically from 3D | Pixel-sharp outlines carve away the edge smear that depth sensors produce |
| Fusion | GPU TSDF ∩ robust visual hull, closed on the support plane | Watertight, true-scale solid |
| Metrology | Plane fits, cross-section circle fits, min-area footprint, oriented box | Numbers you can compare with a caliper |

### You need
* **iPhone 17 Pro or 17 Pro Max** (iPhone 12 Pro – 16 Pro Max and LiDAR iPad Pros work too). ⚠️ The plain
  **iPhone 17 and iPhone Air have no LiDAR.**
* The **PartScan iPhone app** (TestFlight, built from this repository) — it records, uploads and shows the results.
  Without it: the free app **Stray Scanner** (App Store); Record3D `.r3d`, Polycam *raw data* and 3D Scanner App zips also work.
* The **PartScan board** — make the PDF in the app's tab *2 · Board* (or use the board PDFs that came with this
  notebook), print it at 100 % and measure its printed 100 mm ruler; you enter that value in the app.
* A **GPU runtime**: *Runtime → Change runtime type → T4 GPU* (free tier is fine).

### Quick start
1. **Runtime → Run all.** The app appears in section 3 (the first run downloads ≈2 GB of model weights).
2. Tab **2 · Board** → print at 100 % → measure the printed 100 mm ruler.
3. **With the PartScan iPhone app:** set `PUBLIC_LINK = True` in section 3 before running it — the cell shows a
   QR code; in the app's *Server* tab tap *Scan the QR code*, then record and *Upload & build* (roughly 10 min on a
   T4; *Fast* about half). **Without the app:** record with Stray Scanner → *Share → Save to Files → Google Drive ›
   PartScan* → press ↻ in tab 1, pick the scan → **Build 3D model**.

The public link needs Colab Pro / a compute-unit balance (Google's FAQ disallows using a **free** runtime mainly
through a web link) and the notebook tab must stay open. For a permanent address, see *Hosting it yourself* at the end.


In [ ]:
#@title 1 · Setup (≈1 min) { display-mode: "form" }
import os, sys, subprocess, importlib
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    gpu = None
print("GPU:", gpu or "NONE - choose Runtime > Change runtime type > T4 GPU (CPU works but is very slow)")
pkgs = ["trimesh", "av", "fast-simplification", "qrcode"]
missing = [p for p in pkgs if importlib.util.find_spec(p.replace("-", "_")) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import transformers, gradio, cv2, numpy
print("transformers", transformers.__version__, "| gradio", gradio.__version__, "| opencv", cv2.__version__,
      "| numpy", numpy.__version__)
os.makedirs("/content/partscan", exist_ok=True)
os.chdir("/content")
print("ready")

## 2 · Engine code
Each cell below writes one stage of the pipeline into the `partscan/` package. **Just run them** — or open
any stage to read or tweak it (re-run the cell, then re-run section 4).

| module | role |
|---|---|
| `io_scans` | readers for Stray Scanner / Record3D / Polycam / 3D Scanner App, sharp-frame & keyframe selection |
| `board` | ChArUco board generation (print-ready PDF), detection, robust PnP |
| `bundle`, `poses` | SIFT tracks, Levenberg-Marquardt bundle adjustment (Schur complement), lens distortion |
| `segment` | locating the part from LiDAR, SAM 2.1 silhouettes with guided-filter edge refinement |
| `depth` | Prompt Depth Anything (LiDAR-guided AI depth) + per-frame metric correction |
| `fusion` | GPU TSDF ∩ robust visual hull ∩ support plane, marching cubes |
| `meshproc`, `measure` | clean-up, canonical part frame, colour, planes / holes / dimensions |
| `drawings`, `viewer`, `report`, `export` | dimensioned views, interactive 3D viewer, HTML report, files |
| `pipeline`, `app`, `api` | orchestration, the Gradio interface and the REST API used by the iPhone app |


In [ ]:
%%writefile /content/partscan/__init__.py
"""PartScan: accurate 3D models + measurements of small parts from iPhone LiDAR captures."""
__version__ = "1.1.0"


In [ ]:
%%writefile /content/partscan/config.py
"""All tunable settings in one place, with sensible presets."""
from __future__ import annotations

from dataclasses import asdict, dataclass, field


@dataclass
class BoardSpec:
    """Printed ChArUco board. ``square_mm`` should be the size you MEASURED on the print."""
    squares_x: int = 8
    squares_y: int = 6
    square_mm: float = 30.0
    marker_mm: float = 22.0
    dictionary: str = "DICT_4X4_50"

    @property
    def marker_ratio(self) -> float:
        return self.marker_mm / self.square_mm


@dataclass
class Config:
    # ---- input ------------------------------------------------------------------------
    max_candidates: int = 360          # frames kept after sharpness screening (memory bound)
    n_keyframes: int = 90              # frames used for poses, depth and fusion
    min_sharpness_ratio: float = 0.35  # drop frames blurrier than this x median sharpness

    # ---- scale / poses ----------------------------------------------------------------
    use_board: bool = True             # auto-detected; disables itself if no board is seen
    board: BoardSpec = field(default_factory=BoardSpec)
    min_board_corners: int = 10
    refine_intrinsics: bool = True     # estimate radial distortion + focal correction from the board
    feature_ba: bool = True            # SIFT tracks + bundle adjustment (photogrammetric refinement)
    sift_features: int = 6000
    match_neighbors: int = 10          # pairs per keyframe for matching
    ba_max_points: int = 20000

    # ---- depth ------------------------------------------------------------------------
    depth_model: str = "promptda"      # "promptda" (AI, LiDAR-guided) | "lidar" (plain upsampling)
    promptda_checkpoint: str = "depth-anything/prompt-depth-anything-vitl-hf"
    promptda_size: int = 1008          # processing short side: 756 (fast), 1008, 1428 (sharpest)
    align_depth: bool = True           # per-frame scale/offset correction from triangulated points

    # ---- segmentation -----------------------------------------------------------------
    segmenter: str = "sam2"            # "sam2" | "geometry" (no AI; projected coarse shape)
    sam2_checkpoint: str = "facebook/sam2.1-hiera-large"
    carve_tolerance_px: float = 1.0    # how far outside a mask a voxel must project to be carved
    carve_min_views: int = 3           # robust visual hull: k-th strongest vote

    # ---- fusion / mesh ----------------------------------------------------------------
    voxel_mm: float = 0.0              # 0 = automatic (~max_dim/320, clamped 0.2..1.0 mm)
    trunc_voxels: float = 5.0          # TSDF truncation band in voxels
    trunc_back_ratio: float = 1.0      # band behind a surface relative to the front band. <1 helps very thin
                                       # walls but biases noisy depth outwards (erodes surfaces) - keep 1.0
    fusion_min_views: int = 2          # frames needed before a voxel counts as observed
    smooth_iterations: int = 8         # Taubin smoothing (volume preserving)
    max_faces: int = 400_000
    close_bottom: bool = True          # cap the unseen underside on the support plane

    # ---- measurement ------------------------------------------------------------------
    material: str = "Steel"
    reference_name: str = ""           # optional caliper check, e.g. "Length (X)"
    reference_mm: float = 0.0          # value you measured; 0 = no check
    apply_reference_scale: bool = False  # rescale the model so the reference matches exactly

    # ---- runtime ----------------------------------------------------------------------
    device: str = "auto"
    seed: int = 0
    debug: bool = False

    def to_dict(self):
        return asdict(self)


PRESETS = {
    "Fast (fewer frames)": dict(n_keyframes=45, promptda_size=756, sift_features=4000, max_candidates=240),
    "Accurate (recommended)": dict(),
    "Max detail (slow)": dict(n_keyframes=120, promptda_size=1428, sift_features=8000, voxel_mm=0.0,
                              max_candidates=480),
}


def make_config(preset: str = "Accurate (recommended)", **overrides) -> Config:
    cfg = Config()
    for k, v in {**PRESETS.get(preset, {}), **overrides}.items():
        if k == "board" and isinstance(v, dict):
            v = BoardSpec(**v)
        setattr(cfg, k, v)
    return cfg


MATERIAL_DENSITY_G_CM3 = {
    "Steel": 7.85, "Stainless steel": 8.00, "Aluminium 6061": 2.70, "Titanium Ti-6Al-4V": 4.43,
    "Brass": 8.50, "Copper": 8.96, "Cast iron": 7.20, "PLA": 1.24, "ABS": 1.04, "Nylon (PA12)": 1.01,
    "POM / Delrin": 1.41, "None": 0.0,
}


BOARDS = {
    "Standard 8x6 (Letter / A4)": dict(squares_x=8, squares_y=6, square_mm=30.0, marker_mm=22.0),
    "Large 11x7 (A3 / Tabloid)": dict(squares_x=11, squares_y=7, square_mm=35.0, marker_mm=26.0),
}
SCALE_CHECKS = {"printed 100 mm ruler": None, "6 squares between inner corners": 6}


def board_from_check(board_name: str, measured_mm: float, measured_on: str = "printed 100 mm ruler"):
    """BoardSpec scaled by what the user measured on the print. Returns (spec, note-or-empty)."""
    base = BOARDS.get(board_name, BOARDS["Standard 8x6 (Letter / A4)"])
    n_sq = SCALE_CHECKS.get(measured_on)
    nominal = 100.0 if n_sq is None else n_sq * base["square_mm"]
    note = ""
    if not measured_mm or measured_mm <= 0:
        ratio = 1.0
        note = "No print scale check entered - assuming the board printed at exactly 100 %."
    else:
        ratio = float(measured_mm) / nominal
        if abs(ratio - 1) > 0.03:
            note = (f"Your print check ({measured_mm:.2f} mm for {nominal:.1f} mm nominal) is {100 * (ratio - 1):+.1f} % off. "
                    "Printers are normally within 1 %; please re-check the measurement and that you printed at 100 %.")
    spec = BoardSpec(squares_x=base["squares_x"], squares_y=base["squares_y"],
                     square_mm=base["square_mm"] * ratio, marker_mm=base["marker_mm"] * ratio)
    return spec, note


In [ ]:
%%writefile /content/partscan/geometry.py
"""Small, dependency-light geometry helpers shared by every stage.

Conventions used throughout PartScan
------------------------------------
* Units are metres internally; reports and exported meshes use millimetres.
* Camera poses are 4x4 camera-to-world matrices ``T_wc`` with OpenCV camera axes
  (x right, y down, z forward).  ``T_cw = inv(T_wc)``.
* Pixel coordinates follow OpenCV: the centre of the top-left pixel is (0, 0).
  When an image is resized by (sx, sy) the intrinsics change as
  ``fx' = fx*sx`` and ``cx' = (cx + 0.5)*sx - 0.5`` (half-pixel convention).
"""
from __future__ import annotations

import numpy as np
from scipy.spatial.transform import Rotation


# --------------------------------------------------------------------------- poses
def quat_pose(qx, qy, qz, qw, tx, ty, tz) -> np.ndarray:
    T = np.eye(4)
    T[:3, :3] = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()  # scalar-last
    T[:3, 3] = [tx, ty, tz]
    return T


def inv_T(T: np.ndarray) -> np.ndarray:
    """Inverse of one (4,4) or many (N,4,4) rigid transforms."""
    R = T[..., :3, :3]
    t = T[..., :3, 3]
    Ti = np.zeros_like(T)
    Rt = np.swapaxes(R, -1, -2)
    Ti[..., :3, :3] = Rt
    Ti[..., :3, 3] = -np.einsum("...ij,...j->...i", Rt, t)
    Ti[..., 3, 3] = 1.0
    return Ti


def rt_to_T(rvec, tvec) -> np.ndarray:
    T = np.eye(4)
    T[:3, :3] = Rotation.from_rotvec(np.asarray(rvec, float).ravel()).as_matrix()
    T[:3, 3] = np.asarray(tvec, float).ravel()
    return T


def T_to_rt(T):
    return Rotation.from_matrix(T[:3, :3]).as_rotvec(), T[:3, 3].copy()


def rot_angle_deg(R1, R2) -> float:
    c = (np.trace(R1.T @ R2) - 1.0) / 2.0
    return float(np.degrees(np.arccos(np.clip(c, -1.0, 1.0))))


def transform_points(T: np.ndarray, X: np.ndarray) -> np.ndarray:
    return X @ T[:3, :3].T + T[:3, 3]


def umeyama_sim3(src: np.ndarray, dst: np.ndarray, with_scale: bool = True):
    """Least-squares similarity transform with dst ~= s * R @ src + t.  Returns (s, R, t)."""
    src = np.asarray(src, float)
    dst = np.asarray(dst, float)
    mu_s, mu_d = src.mean(0), dst.mean(0)
    xs, xd = src - mu_s, dst - mu_d
    cov = xd.T @ xs / len(src)
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    var_s = (xs ** 2).sum() / len(src)
    s = float(np.trace(np.diag(D) @ S) / var_s) if with_scale else 1.0
    t = mu_d - s * R @ mu_s
    return s, R, t


def sim3_matrix(s, R, t) -> np.ndarray:
    T = np.eye(4)
    T[:3, :3] = s * R
    T[:3, 3] = t
    return T


def apply_sim3_to_pose(s, R, t, T_wc: np.ndarray) -> np.ndarray:
    """Map a camera-to-world pose through x' = s R x + t (rotation part stays orthonormal)."""
    out = np.eye(4)
    out[:3, :3] = R @ T_wc[:3, :3]
    out[:3, 3] = s * R @ T_wc[:3, 3] + t
    return out


# --------------------------------------------------------------------------- intrinsics
def scale_K(K: np.ndarray, sx: float, sy: float | None = None) -> np.ndarray:
    """Intrinsics after resizing an image by (sx, sy) with the half-pixel convention."""
    sy = sx if sy is None else sy
    K2 = np.array(K, dtype=np.float64, copy=True)
    K2[0, 0] *= sx
    K2[0, 1] *= sx
    K2[1, 1] *= sy
    K2[0, 2] = (K[0, 2] + 0.5) * sx - 0.5
    K2[1, 2] = (K[1, 2] + 0.5) * sy - 0.5
    return K2


def K_for_size(K: np.ndarray, src_wh, dst_wh) -> np.ndarray:
    return scale_K(K, dst_wh[0] / src_wh[0], dst_wh[1] / src_wh[1])


def project(K: np.ndarray, T_cw: np.ndarray, X: np.ndarray, dist=None):
    """Project world points. Returns (uv (N,2), z (N,)).  dist = (k1, k2) radial or None."""
    Xc = X @ T_cw[:3, :3].T + T_cw[:3, 3]
    z = Xc[:, 2]
    zs = np.where(np.abs(z) < 1e-9, 1e-9, z)
    x = Xc[:, 0] / zs
    y = Xc[:, 1] / zs
    if dist is not None and (dist[0] != 0 or dist[1] != 0):
        r2 = x * x + y * y
        f = 1 + dist[0] * r2 + dist[1] * r2 * r2
        x, y = x * f, y * f
    u = K[0, 0] * x + K[0, 1] * y + K[0, 2]
    v = K[1, 1] * y + K[1, 2]
    return np.stack([u, v], 1), z


def backproject_depth(depth: np.ndarray, K: np.ndarray, T_wc: np.ndarray, mask=None, stride: int = 1):
    """Depth map (z, metres) -> world points (N,3) plus the pixel coords used."""
    d = depth[::stride, ::stride]
    valid = np.isfinite(d) & (d > 0)
    if mask is not None:
        valid &= mask[::stride, ::stride]
    v, u = np.nonzero(valid)
    z = d[v, u].astype(np.float64)
    u = u * stride
    v = v * stride
    x = (u - K[0, 2]) / K[0, 0] * z
    y = (v - K[1, 2]) / K[1, 1] * z
    Xc = np.stack([x, y, z], 1)
    return transform_points(T_wc, Xc), np.stack([u, v], 1)


def undistort_points_norm(xd: np.ndarray, k1: float, k2: float, iters: int = 8) -> np.ndarray:
    """Invert the 2-term radial model on normalised coordinates (N,2)."""
    xu = xd.copy()
    for _ in range(iters):
        r2 = (xu ** 2).sum(1, keepdims=True)
        xu = xd / (1 + k1 * r2 + k2 * r2 * r2)
    return xu


# --------------------------------------------------------------------------- fitting
def fit_plane(P: np.ndarray, w=None):
    """Weighted least-squares plane. Returns (n, d) with n.x = d and |n| = 1."""
    if w is None:
        c = P.mean(0)
        Q = P - c
    else:
        w = np.asarray(w, float)
        c = (P * w[:, None]).sum(0) / w.sum()
        Q = (P - c) * np.sqrt(w)[:, None]
    _, _, Vt = np.linalg.svd(Q, full_matrices=False)
    n = Vt[-1]
    return n, float(n @ c)


def ransac_plane(P: np.ndarray, thresh: float, iters: int = 400, normal_hint=None,
                 max_angle_deg: float = 25.0, rng=None):
    """Robust plane fit. Optionally only accept planes whose normal is near ``normal_hint``."""
    rng = np.random.default_rng(0) if rng is None else rng
    n_pts = len(P)
    if n_pts < 3:
        raise ValueError("need >= 3 points for a plane")
    best, best_cnt = None, -1
    cos_lim = np.cos(np.radians(max_angle_deg))
    for _ in range(iters):
        idx = rng.choice(n_pts, 3, replace=False)
        a, b, c = P[idx]
        n = np.cross(b - a, c - a)
        nn = np.linalg.norm(n)
        if nn < 1e-12:
            continue
        n /= nn
        if normal_hint is not None:
            if n @ normal_hint < 0:
                n = -n
            if n @ normal_hint < cos_lim:
                continue
        d = n @ a
        cnt = int((np.abs(P @ n - d) < thresh).sum())
        if cnt > best_cnt:
            best, best_cnt = (n, d), cnt
    if best is None:
        raise RuntimeError("RANSAC found no plane")
    n, d = best
    inl = np.abs(P @ n - d) < thresh
    n, d = fit_plane(P[inl])
    if normal_hint is not None and n @ normal_hint < 0:
        n, d = -n, -d
    inl = np.abs(P @ n - d) < thresh
    return n, d, inl


def fit_circle_2d(pts: np.ndarray):
    """Algebraic (Kasa) circle fit refined by Gauss-Newton on geometric distance.
    Returns (cx, cy, r, rms_residual)."""
    pts = np.asarray(pts, float)
    x, y = pts[:, 0], pts[:, 1]
    A = np.stack([x, y, np.ones_like(x)], 1)
    b = x * x + y * y
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0] / 2, sol[1] / 2
    r = np.sqrt(max(sol[2] + cx * cx + cy * cy, 1e-18))
    p = np.array([cx, cy, r])
    for _ in range(20):
        dx, dy = x - p[0], y - p[1]
        dist = np.sqrt(dx * dx + dy * dy) + 1e-15
        res = dist - p[2]
        J = np.stack([-dx / dist, -dy / dist, -np.ones_like(dist)], 1)
        step, *_ = np.linalg.lstsq(J, -res, rcond=None)
        p += step
        if np.abs(step).max() < 1e-12:
            break
    dx, dy = x - p[0], y - p[1]
    rms = float(np.sqrt(np.mean((np.sqrt(dx * dx + dy * dy) - p[2]) ** 2)))
    return float(p[0]), float(p[1]), float(abs(p[2])), rms


def closest_point_to_lines(origins: np.ndarray, dirs: np.ndarray) -> np.ndarray:
    """Least-squares point closest to a bundle of 3D lines (e.g. camera optical axes)."""
    dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)
    A = np.zeros((3, 3))
    b = np.zeros(3)
    for o, d in zip(origins, dirs):
        M = np.eye(3) - np.outer(d, d)
        A += M
        b += M @ o
    return np.linalg.lstsq(A, b, rcond=None)[0]


def min_area_rect(xy: np.ndarray):
    """Minimum-area enclosing rectangle of 2D points (rotating calipers over the hull).
    Returns dict(center, size=(long, short), angle_rad of the long side, corners)."""
    from scipy.spatial import ConvexHull

    xy = np.asarray(xy, float)
    hull = xy[ConvexHull(xy).vertices]
    edges = np.diff(np.vstack([hull, hull[:1]]), axis=0)
    angles = np.unique(np.mod(np.arctan2(edges[:, 1], edges[:, 0]), np.pi / 2))
    best = None
    for a in angles:
        c, s = np.cos(a), np.sin(a)
        R = np.array([[c, s], [-s, c]])  # rotate points by -a
        p = hull @ R.T
        mn, mx = p.min(0), p.max(0)
        area = np.prod(mx - mn)
        if best is None or area < best[0]:
            best = (area, a, mn, mx)
    _, a, mn, mx = best
    size = mx - mn
    c, s = np.cos(a), np.sin(a)
    R = np.array([[c, s], [-s, c]])
    center = R.T @ ((mn + mx) / 2)
    if size[1] > size[0]:  # make the first axis the long side
        a += np.pi / 2
        size = size[::-1]
    a = float(np.mod(a + np.pi / 2, np.pi) - np.pi / 2)  # long axis angle in (-90, 90] deg
    u = np.array([np.cos(a), np.sin(a)])
    v = np.array([-u[1], u[0]])
    h = size / 2
    corners = np.array([center + sx * h[0] * u + sy * h[1] * v for sx, sy in ((-1, -1), (1, -1), (1, 1), (-1, 1))])
    return {"center": center, "size": (float(size[0]), float(size[1])), "angle": a, "corners": corners}


# --------------------------------------------------------------------------- upright rotation
def upright_k(up_cam) -> int:
    """Number of 90-degree counter-clockwise rotations (np.rot90 k) that make the image upright,
    given the world 'up' direction in camera coordinates (x right, y down)."""
    v = np.array([up_cam[0], up_cam[1]], float)
    best, best_k = -2.0, 0
    for k in range(4):
        score = -v[1] / (np.linalg.norm(v) + 1e-12)   # alignment with image-up (0, -1)
        if score > best:
            best, best_k = score, k
        v = np.array([v[1], -v[0]])                      # direction after one more CCW rotation
    return best_k


def rot_points(pts, k: int, W: int, H: int):
    """Pixel coordinates (x, y) in an image of size W x H after np.rot90(img, k)."""
    out = []
    for x, y in pts:
        w, h = W, H                      # per point: every point starts from the original size
        for _ in range(k % 4):
            x, y, w, h = y, w - 1 - x, h, w
        out.append((x, y))
    return out


def fill_triangles(img: np.ndarray, tri_px: np.ndarray, value: int = 1, shift: int = 3) -> np.ndarray:
    """Rasterise many triangles (N,3,2) as a UNION. (cv2.fillPoly with a list of polygons uses the
    even-odd rule, so overlapping front/back faces would cancel each other out.)
    tri_px are fixed-point pixel coordinates scaled by 2**shift."""
    import cv2
    tri_px = np.ascontiguousarray(tri_px, np.int32)
    for t in tri_px:
        cv2.fillConvexPoly(img, t, value, lineType=cv2.LINE_8, shift=shift)
    return img


In [ ]:
%%writefile /content/partscan/io_scans.py
"""Readers for iPhone LiDAR capture apps -> one common in-memory format.

Supported exports (auto-detected from a .zip, a .r3d or a folder):
  * Stray Scanner  (free, recommended)   odometry.csv + rgb.mp4 + depth/*.png + confidence/*.png
  * Record3D       (.r3d)                metadata + rgbd/{i}.jpg/.depth/.conf (LZFSE)
  * Polycam        (Raw data export)     keyframes/{images,cameras,depth,confidence}
  * 3D Scanner App ("All Data" export)   frame_XXXXX.{jpg,json} + depth_XXXXX.png + conf_XXXXX.png

Every frame ends up with: RGB image, intrinsics K (for that RGB), camera-to-world pose in
OpenCV camera axes inside the app's ARKit world (gravity aligned, +Y up, metres), and the
LiDAR depth (float32 metres) + confidence (0/1/2) at the sensor's native 256x192.
"""
from __future__ import annotations

import glob
import io
import json
import logging
import math
import os
import shutil
import subprocess
import sys
import zipfile
from dataclasses import dataclass, field
from typing import Optional

import cv2
import numpy as np
from PIL import Image

from .geometry import quat_pose, rot_angle_deg

log = logging.getLogger("partscan")
GL2CV = np.diag([1.0, -1.0, -1.0, 1.0])  # ARKit/OpenGL camera axes -> OpenCV camera axes


@dataclass
class Frame:
    idx: int                        # frame number inside the recording
    t: float                        # timestamp [s]
    K: np.ndarray                   # 3x3 intrinsics of the RGB image
    T_wc: np.ndarray                # 4x4 camera->world, OpenCV axes, ARKit world
    depth: Optional[np.ndarray] = None   # LiDAR depth [m], (h, w) float32, 0 = invalid
    conf: Optional[np.ndarray] = None    # LiDAR confidence 0/1/2 (uint8)
    sharpness: float = 0.0
    rgb_bytes: Optional[bytes] = None    # encoded image kept in memory
    rgb_path: Optional[str] = None       # or a path on disk
    rgb: Optional[np.ndarray] = None     # decoded (H, W, 3) uint8, only for keyframes
    extra: dict = field(default_factory=dict)

    def load_rgb(self) -> np.ndarray:
        if self.rgb is None:
            if self.rgb_bytes is not None:
                arr = cv2.imdecode(np.frombuffer(self.rgb_bytes, np.uint8),
                                   cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)   # K refers to stored pixels
                self.rgb = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
            elif self.rgb_path is not None:
                self.rgb = np.array(Image.open(self.rgb_path).convert("RGB"))
            else:
                raise RuntimeError(f"frame {self.idx} has no image")
        return self.rgb

    @property
    def rgb_size(self):
        if self.rgb is not None:
            return self.rgb.shape[1], self.rgb.shape[0]
        return self.extra["rgb_size"]


@dataclass
class Scan:
    frames: list
    source: str
    rgb_size: tuple
    n_total: int
    fps: float = 0.0
    notes: list = field(default_factory=list)


# --------------------------------------------------------------------------- helpers
def sharpness(gray_small: np.ndarray) -> float:
    """Variance of the Laplacian on a downscaled grayscale image (higher = sharper)."""
    return float(cv2.Laplacian(gray_small, cv2.CV_32F, ksize=3).var())


def _encode(rgb: np.ndarray) -> bytes:
    ok, buf = cv2.imencode(".jpg", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 96])
    assert ok
    return buf.tobytes()


def _read_png(path) -> np.ndarray:
    return np.array(Image.open(path))


def _pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


def _lzfse_decompress(data: bytes) -> bytes:
    try:
        import lzfse  # pip install lzfse
        return lzfse.decompress(data)
    except ImportError:
        pass
    try:
        import liblzfse  # pip install pyliblzfse
        return liblzfse.decompress(data)
    except ImportError:
        log.info("installing lzfse (needed for Record3D files)...")
        _pip_install("lzfse")
        import lzfse
        return lzfse.decompress(data)


def _window_pick(n: int, max_keep: int):
    """Split range(n) into consecutive windows so at most max_keep survive."""
    stride = max(1, math.ceil(n / max_keep))
    return stride


# --------------------------------------------------------------------------- Stray Scanner
def load_stray(root: str, max_candidates: int = 360, progress=None) -> Scan:
    import pandas as pd
    import av

    odo = pd.read_csv(os.path.join(root, "odometry.csv"), skipinitialspace=True)
    odo.columns = [c.strip() for c in odo.columns]
    n = len(odo)
    K_last = None
    cm = os.path.join(root, "camera_matrix.csv")
    if os.path.exists(cm):
        K_last = np.loadtxt(cm, delimiter=",").reshape(3, 3)
    has_k = all(c in odo.columns for c in ("fx", "fy", "cx", "cy")) and odo["fx"].notna().all()
    stride = _window_pick(n, max_candidates)
    notes = []

    frames, best = [], None
    vpath = os.path.join(root, "rgb.mp4")
    # ignore_editlist: Stray gives the first frame a negative timestamp that an edit list hides;
    # without this flag the decoder silently drops it and every image is paired with the wrong pose.
    with av.open(vpath, options={"ignore_editlist": "1"}) as cont:
        stream = cont.streams.video[0]
        stream.thread_type = "AUTO"
        W, H = stream.codec_context.width, stream.codec_context.height
        fps = float(stream.average_rate or 0) if stream.average_rate else 0.0
        n_video, extra_video = 0, False
        for k, vf in enumerate(cont.decode(stream)):
            if k >= n:
                extra_video = True  # more video frames than poses
                break
            n_video = k + 1
            small = vf.reformat(width=W // 4, height=H // 4, format="gray").to_ndarray()
            s = sharpness(small)
            if best is None or s > best[0]:
                best = (s, k, vf.to_ndarray(format="rgb24"))
            if (k + 1) % stride == 0:
                frames.append(_stray_frame(root, odo, best, has_k, K_last))
                best = None
            if progress is not None and k % 50 == 0:
                progress(k / n, f"decoding video {k}/{n}")
        if best is not None:
            frames.append(_stray_frame(root, odo, best, has_k, K_last))
    if n_video != n or extra_video:
        msg = (f"video has {'more than ' + str(n) if extra_video else n_video} frames but odometry has {n} rows;"
               f" frames were paired in order")
        log.warning(msg)
        notes.append(msg)
    if not has_k:
        notes.append("old Stray Scanner version: one intrinsics matrix for all frames (update the app to v1.4+)")
    return Scan(frames, "Stray Scanner", (W, H), n, fps, notes)


def _stray_frame(root, odo, best, has_k, K_last) -> Frame:
    s, k, rgb = best
    r = odo.iloc[k]
    fid = int(r["frame"])
    T = quat_pose(r["qx"], r["qy"], r["qz"], r["qw"], r["x"], r["y"], r["z"])  # already OpenCV axes
    if has_k:
        K = np.array([[r["fx"], 0, r["cx"]], [0, r["fy"], r["cy"]], [0, 0, 1.0]])
    else:
        K = K_last
    depth = _read_png(os.path.join(root, "depth", f"{fid:06d}.png")).astype(np.float32) / 1000.0
    cpath = os.path.join(root, "confidence", f"{fid:06d}.png")
    conf = _read_png(cpath).astype(np.uint8) if os.path.exists(cpath) else np.full(depth.shape, 2, np.uint8)
    f = Frame(fid, float(r["timestamp"]), K, T, depth, conf, s, rgb_bytes=_encode(rgb))
    f.extra["rgb_size"] = (rgb.shape[1], rgb.shape[0])
    return f


# --------------------------------------------------------------------------- Record3D
def load_r3d(path: str, max_candidates: int = 360, progress=None) -> Scan:
    if os.path.isdir(path):
        def read(name):
            with open(os.path.join(path, name), "rb") as fh:
                return fh.read()
        names = set(os.path.relpath(p, path).replace(os.sep, "/") for p in glob.glob(os.path.join(path, "**"), recursive=True))
    else:
        z = zipfile.ZipFile(path)
        read = z.read
        names = set(z.namelist())
    md = json.loads(read("metadata"))
    W, H, dw, dh = int(md["w"]), int(md["h"]), int(md["dw"]), int(md["dh"])
    K = np.array(md["K"], dtype=np.float64).reshape(3, 3).T  # stored column-major
    poses = md["poses"]
    ts = md.get("frameTimestamps") or [i / float(md.get("fps", 30)) for i in range(len(poses))]
    n = len(poses)
    stride = _window_pick(n, max_candidates)
    frames = []
    for w0 in range(0, n, stride):
        best = None
        for i in range(w0, min(n, w0 + stride)):
            name = f"rgbd/{i}.jpg"
            if name not in names:
                continue
            im = Image.open(io.BytesIO(read(name)))
            im.draft("L", (im.width // 4, im.height // 4))
            s = sharpness(np.array(im.convert("L")))
            if best is None or s > best[0]:
                best = (s, i)
        if best is None:
            continue
        s, i = best
        raw = read(f"rgbd/{i}.jpg")
        depth = np.frombuffer(_lzfse_decompress(read(f"rgbd/{i}.depth")), np.float32).reshape(dh, dw).copy()
        depth[~np.isfinite(depth)] = 0
        cname = f"rgbd/{i}.conf"
        conf = (np.frombuffer(_lzfse_decompress(read(cname)), np.uint8).reshape(dh, dw).copy()
                if cname in names else np.full((dh, dw), 2, np.uint8))
        f = Frame(i, float(ts[i]) if i < len(ts) else float(i), K.copy(), quat_pose(*poses[i]) @ GL2CV,
                  depth, conf, s, rgb_bytes=raw)
        f.extra["rgb_size"] = (W, H)
        frames.append(f)
        if progress is not None:
            progress(w0 / n, f"reading Record3D frame {i}/{n}")
    notes = []
    if W < 1000:
        notes.append(f"Record3D stored only {W}x{H} RGB - enable the highest RGB quality in the app for better detail")
    return Scan(frames, "Record3D", (W, H), n, float(md.get("fps", 0)), notes)


# --------------------------------------------------------------------------- Polycam
def load_polycam(root: str, max_candidates: int = 360, progress=None) -> Scan:
    kf = os.path.join(root, "keyframes")
    # original images + their intrinsics (the depth maps belong to these); loop-closed poses if available
    corrected = os.path.isdir(os.path.join(kf, "corrected_cameras"))
    cam_dir = os.path.join(kf, "cameras") if os.path.isdir(os.path.join(kf, "cameras")) else os.path.join(kf, "corrected_cameras")
    pose_dir = os.path.join(kf, "corrected_cameras") if corrected else cam_dir
    img_dir = os.path.join(kf, "images") if os.path.isdir(os.path.join(kf, "images")) else os.path.join(kf, "corrected_images")
    files = sorted(glob.glob(os.path.join(cam_dir, "*.json")), key=lambda p: int(os.path.basename(p)[:-5]))
    stride = _window_pick(len(files), max_candidates)
    frames, W, H = [], None, None
    for k in range(0, len(files), stride):
        jf = files[k]
        ts = int(os.path.basename(jf)[:-5])
        c = json.load(open(jf))
        pj = os.path.join(pose_dir, os.path.basename(jf))
        cp = json.load(open(pj)) if os.path.exists(pj) else c
        T = np.eye(4)
        T[:3, :] = [[cp[f"t_{r}{q}"] for q in range(4)] for r in range(3)]
        K = np.array([[c["fx"], 0, c["cx"]], [0, c["fy"], c["cy"]], [0, 0, 1.0]])
        W, H = int(c["width"]), int(c["height"])
        dpath = os.path.join(kf, "depth", f"{ts}.png")
        cpath = os.path.join(kf, "confidence", f"{ts}.png")
        if not os.path.exists(dpath):
            continue
        depth = _read_png(dpath).astype(np.float32) / 1000.0
        conf = (_read_png(cpath) // 127).astype(np.uint8) if os.path.exists(cpath) else np.full(depth.shape, 2, np.uint8)
        ipath = os.path.join(img_dir, f"{ts}.jpg")
        im = Image.open(ipath)
        im.draft("L", (im.width // 4, im.height // 4))
        f = Frame(ts, ts * 1e-6, K, T @ GL2CV, depth, conf, sharpness(np.array(im.convert("L"))), rgb_path=ipath)
        f.extra["rgb_size"] = (W, H)
        frames.append(f)
    notes = ["Polycam keyframes" + (" with loop-closed (corrected) poses" if corrected else "")]
    return Scan(frames, "Polycam (raw data)", (W, H), len(files), 0.0, notes)


# --------------------------------------------------------------------------- 3D Scanner App
def load_3dscannerapp(root: str, max_candidates: int = 360, progress=None) -> Scan:
    jsons = sorted(glob.glob(os.path.join(root, "frame_*.json")))
    with_img = [j for j in jsons if os.path.exists(j[:-5] + ".jpg")]
    stride = _window_pick(len(with_img), max_candidates)
    frames, W, H = [], None, None
    for jf in with_img[::stride]:
        i = int(os.path.basename(jf)[6:11])
        d = json.load(open(jf))
        T_gl = np.array(d["cameraPoseARFrame"], dtype=np.float64).reshape(4, 4)
        K = np.array(d["intrinsics"], dtype=np.float64).reshape(3, 3)
        jpg = jf[:-5] + ".jpg"
        im = Image.open(jpg)
        W, H = im.size
        im.draft("L", (im.width // 4, im.height // 4))
        depth = _read_png(os.path.join(root, f"depth_{i:05d}.png")).astype(np.float32) / 1000.0
        cp = os.path.join(root, f"conf_{i:05d}.png")
        conf = _read_png(cp).astype(np.uint8) if os.path.exists(cp) else np.full(depth.shape, 2, np.uint8)
        f = Frame(i, float(d.get("time", i)), K, T_gl @ GL2CV, depth, conf,
                  sharpness(np.array(im.convert("L"))), rgb_path=jpg)
        f.extra["rgb_size"] = (W, H)
        frames.append(f)
    return Scan(frames, "3D Scanner App", (W, H), len(jsons), 0.0, [])


# --------------------------------------------------------------------------- auto-detect
def _find(root: str, test, max_depth: int = 4):
    root = os.path.abspath(root)
    base_depth = root.rstrip(os.sep).count(os.sep)
    for d, dirs, files in os.walk(root):
        if d.count(os.sep) - base_depth > max_depth:
            dirs[:] = []
            continue
        dirs[:] = sorted(x for x in dirs if not x.startswith(("__MACOSX", ".")))
        if test(d, set(files), set(dirs)):
            return d
    return None


def load_scan(path: str, work_dir: str, max_candidates: int = 360, progress=None) -> Scan:
    """Load any supported capture (zip / .r3d / folder)."""
    path = os.path.abspath(path)
    if os.path.isfile(path) and path.lower().endswith(".r3d"):
        return load_r3d(path, max_candidates, progress)
    if os.path.isfile(path) and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            if "metadata" in z.namelist() and any(n.startswith("rgbd/") for n in z.namelist()):
                return load_r3d(path, max_candidates, progress)
        out = os.path.join(work_dir, "extracted")
        shutil.rmtree(out, ignore_errors=True)
        os.makedirs(out, exist_ok=True)
        with zipfile.ZipFile(path) as z:
            z.extractall(out)
        path = out
    if not os.path.isdir(path):
        raise ValueError(f"Unsupported input: {path}. Upload the zip exported by Stray Scanner, "
                         "a Record3D .r3d, a Polycam raw-data zip, or a 3D Scanner App 'All Data' zip.")
    tests = [
        ("stray", lambda d, f, s: "odometry.csv" in f and "rgb.mp4" in f),
        ("r3d", lambda d, f, s: "metadata" in f and "rgbd" in s),
        ("polycam", lambda d, f, s: "keyframes" in s and (
            os.path.isdir(os.path.join(d, "keyframes", "cameras")) or
            os.path.isdir(os.path.join(d, "keyframes", "corrected_cameras")))),
        ("3dsa", lambda d, f, s: any(x.startswith("frame_") and x.endswith(".json") for x in f)),
    ]
    for kind, test in tests:
        d = _find(path, test)
        if d is None:
            continue
        log.info("detected %s capture in %s", kind, d)
        if kind == "stray":
            return load_stray(d, max_candidates, progress)
        if kind == "r3d":
            return load_r3d(d, max_candidates, progress)
        if kind == "polycam":
            return load_polycam(d, max_candidates, progress)
        return load_3dscannerapp(d, max_candidates, progress)
    raise ValueError("Could not recognise the capture format. Expected a Stray Scanner folder "
                     "(odometry.csv + rgb.mp4), Record3D .r3d, Polycam raw data or 3D Scanner App 'All Data'.")


# --------------------------------------------------------------------------- keyframes
def select_keyframes(frames: list, n: int, min_sharpness_ratio: float = 0.35) -> list:
    """Pick ~n sharp frames spread evenly along the camera path (by rotation + translation)."""
    if not frames:
        return []
    frames = sorted(frames, key=lambda f: f.t)
    s = np.array([f.sharpness for f in frames])
    keep = s >= min_sharpness_ratio * np.median(s)
    cand = [f for f, k in zip(frames, keep) if k]
    if len(cand) <= n:
        return cand
    motion = [0.0]
    for a, b in zip(cand[:-1], cand[1:]):
        dr = rot_angle_deg(a.T_wc[:3, :3], b.T_wc[:3, :3]) / 5.0          # 5 deg  ~ 1 unit
        dt = np.linalg.norm(a.T_wc[:3, 3] - b.T_wc[:3, 3]) / 0.02          # 2 cm   ~ 1 unit
        motion.append(motion[-1] + dr + dt)
    motion = np.array(motion)
    edges = np.linspace(0, motion[-1] + 1e-9, n + 1)
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        idx = np.nonzero((motion >= lo) & (motion < hi))[0]
        if len(idx):
            out.append(cand[int(idx[np.argmax([cand[i].sharpness for i in idx])])])
    return out


In [ ]:
%%writefile /content/partscan/board.py
"""Printable ChArUco calibration board: generation, detection and per-frame pose.

Putting the part on this board gives three things LiDAR cannot: an exact metric scale
(from the printed square size you measure), drift-free camera poses, and a perfectly
known support plane.
"""
from __future__ import annotations

import logging

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

from .config import BoardSpec

log = logging.getLogger("partscan")

PAPER_MM = {"A4": (297.0, 210.0), "Letter": (279.4, 215.9), "A3": (420.0, 297.0), "Tabloid": (431.8, 279.4)}


def make_board(spec: BoardSpec):
    d = cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, spec.dictionary))
    return cv2.aruco.CharucoBoard((spec.squares_x, spec.squares_y), spec.square_mm / 1000.0,
                                  spec.marker_mm / 1000.0, d)


def make_detector(board):
    dp = cv2.aruco.DetectorParameters()
    dp.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
    dp.adaptiveThreshWinSizeMax = 53
    cp = cv2.aruco.CharucoParameters()
    cp.tryRefineMarkers = True
    return cv2.aruco.CharucoDetector(board, cp, dp)


def board_to_part(spec: BoardSpec) -> np.ndarray:
    """4x4 transform from OpenCV board coordinates (origin at a corner, y down the print,
    z into the table) to the PartScan world: origin at the board centre, z up."""
    cx = spec.squares_x * spec.square_mm / 2000.0
    cy = spec.squares_y * spec.square_mm / 2000.0
    T = np.diag([1.0, -1.0, -1.0, 1.0])
    T[:3, 3] = [-cx, cy, 0.0]
    return T


def board_extent_part(spec: BoardSpec):
    """(xmin, xmax, ymin, ymax) of the printed squares in the part frame [m]."""
    hx = spec.squares_x * spec.square_mm / 2000.0
    hy = spec.squares_y * spec.square_mm / 2000.0
    return -hx, hx, -hy, hy


def detect(gray: np.ndarray, board, detector):
    """Returns (img_pts (N,2) float64, ids (N,), obj_pts (N,3) float64) or None."""
    cc, ci, _, _ = detector.detectBoard(gray)
    if ci is None or len(ci) < 4:
        return None
    obj, img = board.matchImagePoints(cc, ci)
    return img.reshape(-1, 2).astype(np.float64), ci.ravel().astype(int), obj.reshape(-1, 3).astype(np.float64)


def pnp(obj: np.ndarray, img: np.ndarray, K: np.ndarray, dist=None):
    """Board pose T_cb (board -> camera) and reprojection RMS [px]."""
    dist = np.zeros(5) if dist is None else np.asarray(dist, float)
    ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist, flags=cv2.SOLVEPNP_IPPE)
    if not ok:
        ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist, flags=cv2.SOLVEPNP_ITERATIVE)
        if not ok:
            return None, np.inf
    rvec, tvec = cv2.solvePnPRefineLM(obj, img, K, dist, rvec, tvec)
    proj, _ = cv2.projectPoints(obj, rvec, tvec, K, dist)
    rms = float(np.sqrt(np.mean(np.sum((proj.reshape(-1, 2) - img) ** 2, 1))))
    T = np.eye(4)
    T[:3, :3] = cv2.Rodrigues(rvec)[0]
    T[:3, 3] = tvec.ravel()
    return T, rms


def robust_pnp(obj: np.ndarray, img: np.ndarray, K: np.ndarray, dist=None, min_pts: int = 8):
    """PnP with iterative rejection of mis-detected corners (e.g. partly hidden by the part).
    Returns (T_cb, keep_mask, rms_px) or (None, None, inf)."""
    keep = np.ones(len(img), bool)
    T, rms = None, np.inf
    for _ in range(4):
        if keep.sum() < min_pts:
            return None, None, np.inf
        T, rms = pnp(obj[keep], img[keep], K, dist)
        if T is None:
            return None, None, np.inf
        Xc = obj @ T[:3, :3].T + T[:3, 3]
        uv = Xc[:, :2] / Xc[:, 2:3] * [K[0, 0], K[1, 1]] + [K[0, 2], K[1, 2]]
        err = np.linalg.norm(uv - img, axis=1)
        thr = max(0.8, 3.0 * 1.4826 * np.median(np.abs(err[keep])))
        new_keep = err < thr
        if (new_keep == keep).all():
            break
        keep = new_keep
    return T, keep, rms


# --------------------------------------------------------------------------- printable PDF
def _font(px: int):
    try:
        import matplotlib
        import os
        return ImageFont.truetype(os.path.join(matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSans.ttf"), px)
    except Exception:
        return ImageFont.load_default()


def render_board_pdf(spec: BoardSpec, path: str, paper: str = "A4", px_per_mm: int = 24, text: bool = True) -> str:
    """Write a print-ready PDF (landscape) with the board at exact physical size plus
    reference rulers. px_per_mm=24 -> 609.6 dpi so every square is an integer number of pixels.
    text=False leaves out the captions (the bare 100 mm ruler stays)."""
    pw, ph = PAPER_MM[paper]
    W, H = int(round(pw * px_per_mm)), int(round(ph * px_per_mm))
    bw_mm, bh_mm = spec.squares_x * spec.square_mm, spec.squares_y * spec.square_mm
    if bw_mm > pw - 16 or bh_mm > ph - 16:
        raise ValueError(f"{bw_mm:.0f}x{bh_mm:.0f} mm board does not fit on {paper}; use smaller squares or bigger paper")
    page = Image.new("L", (W, H), 255)
    board = make_board(spec)
    bw, bh = int(round(bw_mm * px_per_mm)), int(round(bh_mm * px_per_mm))
    img = board.generateImage((bw, bh), marginSize=0, borderBits=1)
    x0 = int(round((pw - bw_mm) / 2 * px_per_mm))
    y0 = int(round((ph - bh_mm) / 2 * px_per_mm))
    page.paste(Image.fromarray(img), (x0, y0))
    dr = ImageDraw.Draw(page)
    mm = px_per_mm
    f_small = _font(int(2.6 * mm))
    top = y0 / mm  # mm of margin above the board
    txt = (f"PartScan ChArUco board  |  {spec.squares_x}x{spec.squares_y} squares of {spec.square_mm:g} mm, "
           f"markers {spec.marker_mm:g} mm, {spec.dictionary}  |  PRINT AT 100% / ACTUAL SIZE, glue flat to a rigid board")
    if text:
        dr.text((x0, max(1.5, top / 2 - 1.5) * mm), txt, fill=0, font=f_small)
    # reference ruler under the board: 100 mm with 10 mm ticks
    by = y0 + bh + int(min(4, (ph - bh_mm) / 2 - 6) * mm)
    L = 100 * mm
    dr.line([(x0, by), (x0 + L, by)], fill=0, width=max(2, mm // 6))
    for k in range(11):
        tick = int((2.5 if k % 5 == 0 else 1.5) * mm)
        dr.line([(x0 + k * 10 * mm, by - tick), (x0 + k * 10 * mm, by)], fill=0, width=max(2, mm // 6))
    if text:
        dr.text((x0 + L + 2 * mm, by - 3 * mm),
                f"100 mm check ruler: measure it (or 6 squares between inner corners = {6 * spec.square_mm:g} mm) "
                f"and enter the value in the app.", fill=0, font=f_small)
    page.save(path, "PDF", resolution=px_per_mm * 25.4)
    return path


def render_board_png(spec: BoardSpec, path: str, px_per_mm: int = 10) -> str:
    board = make_board(spec)
    img = board.generateImage((int(spec.squares_x * spec.square_mm * px_per_mm),
                               int(spec.squares_y * spec.square_mm * px_per_mm)), marginSize=0, borderBits=1)
    Image.fromarray(img).save(path)
    return path


In [ ]:
%%writefile /content/partscan/bundle.py
"""Photogrammetric pose refinement.

* SIFT (RootSIFT) features, mutual-NN + ratio matching (GPU if available), MAGSAC verification
* multi-view tracks (union-find) and DLT triangulation
* sparse bundle adjustment (scipy) mixing three kinds of evidence:
    - ChArUco corners (known 3D points -> metric scale, no drift)
    - feature tracks on/around the part
    - ARKit relative motion between consecutive keyframes (keeps weakly observed frames sane)
  optionally refining a global focal correction, principal point shift and radial distortion.
"""
from __future__ import annotations

import logging
from dataclasses import dataclass, field

import cv2
import numpy as np
from scipy.optimize import least_squares
from scipy.spatial.transform import Rotation

from .geometry import inv_T

log = logging.getLogger("partscan")


# --------------------------------------------------------------------------- features
def extract_features(gray: np.ndarray, n_features: int, mask=None):
    sift = cv2.SIFT_create(nfeatures=n_features, contrastThreshold=0.015, edgeThreshold=12)
    kps, desc = sift.detectAndCompute(gray, mask)
    if desc is None or len(kps) == 0:
        return np.zeros((0, 2)), np.zeros((0, 128), np.float32)
    desc = desc / (desc.sum(1, keepdims=True) + 1e-7)        # RootSIFT
    desc = np.sqrt(desc).astype(np.float32)
    pts = np.array([k.pt for k in kps], np.float64)
    return pts, desc


def _match_pair(d1, d2, ratio=0.82, device=None):
    """Mutual nearest neighbours with Lowe ratio on unit-norm descriptors."""
    if len(d1) < 2 or len(d2) < 2:
        return np.zeros((0, 2), int)
    try:
        import torch
        dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        a = torch.from_numpy(d1).to(dev)
        b = torch.from_numpy(d2).to(dev)
        sim = a @ b.T
        top2 = sim.topk(2, dim=1)
        nn12 = top2.indices[:, 0]
        nn21 = sim.argmax(0)
        # ||a-b||^2 = 2 - 2 sim  (unit vectors)
        d_best = (2 - 2 * top2.values[:, 0]).clamp_min(0).sqrt()
        d_second = (2 - 2 * top2.values[:, 1]).clamp_min(0).sqrt()
        ok = (d_best < ratio * d_second) & (nn21[nn12] == torch.arange(len(a), device=dev))
        i1 = torch.nonzero(ok).ravel()
        return torch.stack([i1, nn12[i1]], 1).cpu().numpy()
    except ImportError:
        sim = d1 @ d2.T
        nn12 = sim.argmax(1)
        nn21 = sim.argmax(0)
        s_sorted = np.sort(sim, 1)
        d_best = np.sqrt(np.clip(2 - 2 * s_sorted[:, -1], 0, None))
        d_second = np.sqrt(np.clip(2 - 2 * s_sorted[:, -2], 0, None))
        ok = (d_best < ratio * d_second) & (nn21[nn12] == np.arange(len(d1)))
        i1 = np.nonzero(ok)[0]
        return np.stack([i1, nn12[i1]], 1)


def select_pairs(T_wc: list, n_neighbors: int, max_angle: float = 40.0):
    """Frame pairs with similar viewing directions (likely to share features)."""
    F = len(T_wc)
    z = np.array([T[:3, 2] for T in T_wc])
    c = np.array([T[:3, 3] for T in T_wc])
    ang = np.degrees(np.arccos(np.clip(z @ z.T, -1, 1)))
    pairs = set()
    for i in range(F):
        order = np.argsort(ang[i])
        cnt = 0
        for j in order:
            if j == i or ang[i, j] > max_angle:
                continue
            if np.linalg.norm(c[i] - c[j]) < 0.004 and ang[i, j] < 1.0:
                continue  # nearly identical viewpoint: no triangulation value
            pairs.add((min(i, j), max(i, j)))
            cnt += 1
            if cnt >= n_neighbors:
                break
    return sorted(pairs)


class _UF:
    def __init__(self, n):
        self.p = np.arange(n)

    def find(self, a):
        p = self.p
        root = a
        while p[root] != root:
            root = p[root]
        while p[a] != root:
            p[a], a = root, p[a]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[max(ra, rb)] = min(ra, rb)


@dataclass
class Tracks:
    obs_cam: np.ndarray        # (M,) frame index
    obs_pt: np.ndarray         # (M,) track index
    obs_uv: np.ndarray         # (M,2) pixel
    X: np.ndarray              # (P,3) world points
    n_points: int = 0
    stats: dict = field(default_factory=dict)


def build_tracks(feats: list, pairs: list, device=None, min_inliers: int = 20, progress=None):
    """Match features over the pairs and chain the verified matches into multi-view tracks."""
    offs = np.cumsum([0] + [len(p) for p, _ in feats])
    uf = _UF(int(offs[-1]))
    n_ok = 0
    for k, (i, j) in enumerate(pairs):
        (p1, d1), (p2, d2) = feats[i], feats[j]
        m = _match_pair(d1, d2, device=device)
        if len(m) < min_inliers:
            continue
        F, inl = cv2.findFundamentalMat(p1[m[:, 0]], p2[m[:, 1]], cv2.USAC_MAGSAC, 1.5, 0.999, 5000)
        if F is None or inl is None:
            continue
        m = m[inl.ravel().astype(bool)]
        if len(m) < min_inliers:
            continue
        n_ok += 1
        for a, b in m:
            uf.union(offs[i] + a, offs[j] + b)
        if progress is not None and k % 20 == 0:
            progress(k / max(1, len(pairs)), f"matching pair {k}/{len(pairs)}")
    roots = np.array([uf.find(a) for a in range(int(offs[-1]))])
    cam_of = np.searchsorted(offs, np.arange(int(offs[-1])), side="right") - 1
    kp_of = np.arange(int(offs[-1])) - offs[cam_of]
    order = np.argsort(roots, kind="stable")
    r_sorted = roots[order]
    starts = np.r_[0, np.nonzero(np.diff(r_sorted))[0] + 1]
    ends = np.r_[starts[1:], len(order)]
    tracks = []
    for s, e in zip(starts, ends):
        if e - s < 2:
            continue
        members = order[s:e]
        cams = cam_of[members]
        if len(np.unique(cams)) != len(cams):
            continue  # inconsistent track (two features of one image)
        tracks.append([(int(c), feats[c][0][kp_of[mm]]) for c, mm in zip(cams, members)])
    return tracks, n_ok


# --------------------------------------------------------------------------- triangulation
def triangulate_tracks(tracks: list, Ks: np.ndarray, T_cw: np.ndarray, min_views: int = 3,
                       max_reproj: float = 3.0, min_angle_deg: float = 2.0):
    """Multi-view DLT for every track; drops bad observations / points. Returns Tracks."""
    obs_cam, obs_pt, obs_uv, Xs = [], [], [], []
    centers = inv_T(T_cw)[:, :3, 3]
    P = np.einsum("fij,fjk->fik", Ks, T_cw[:, :3, :])
    by_len = {}
    for t in tracks:
        if len(t) >= min_views:
            by_len.setdefault(len(t), []).append(t)
    for L, group in by_len.items():
        cams = np.array([[c for c, _ in t] for t in group])            # (T,L)
        uv = np.array([[p for _, p in t] for t in group])               # (T,L,2)
        Pm = P[cams]                                                    # (T,L,3,4)
        A = np.concatenate([uv[..., 0:1] * Pm[..., 2, :] - Pm[..., 0, :],
                            uv[..., 1:2] * Pm[..., 2, :] - Pm[..., 1, :]], axis=1)  # (T,2L,4)
        A = A / (np.linalg.norm(A, axis=2, keepdims=True) + 1e-12)
        _, _, Vt = np.linalg.svd(A)
        Xh = Vt[:, -1, :]
        X = Xh[:, :3] / np.where(np.abs(Xh[:, 3:]) < 1e-12, 1e-12, Xh[:, 3:])
        # reprojection per observation
        Xc = np.einsum("tlij,tj->tli", T_cw[cams][..., :3, :3], X) + T_cw[cams][..., :3, 3]
        z = Xc[..., 2]
        proj = np.einsum("tlij,tlj->tli", Ks[cams], Xc / np.where(np.abs(z) < 1e-9, 1e-9, z)[..., None])[..., :2]
        err = np.linalg.norm(proj - uv, axis=2)
        good = (err < max_reproj) & (z > 0.02)
        # triangulation angle
        rays = X[:, None, :] - centers[cams]
        rays /= np.linalg.norm(rays, axis=2, keepdims=True) + 1e-12
        cosang = np.einsum("tli,tmi->tlm", rays, rays)
        cosang = np.where(good[:, :, None] & good[:, None, :], cosang, 1.0)
        max_ang = np.degrees(np.arccos(np.clip(cosang.min(axis=(1, 2)), -1, 1)))
        keep = (good.sum(1) >= min_views) & (max_ang >= min_angle_deg)
        for ti in np.nonzero(keep)[0]:
            pid = len(Xs)
            Xs.append(X[ti])
            for l in np.nonzero(good[ti])[0]:
                obs_cam.append(cams[ti, l])
                obs_pt.append(pid)
                obs_uv.append(uv[ti, l])
    if not Xs:
        return Tracks(np.zeros(0, int), np.zeros(0, int), np.zeros((0, 2)), np.zeros((0, 3)), 0)
    return Tracks(np.array(obs_cam), np.array(obs_pt), np.array(obs_uv), np.array(Xs), len(Xs))


# --------------------------------------------------------------------------- bundle adjustment
@dataclass
class BAResult:
    T_wc: np.ndarray
    X: np.ndarray
    intr: np.ndarray            # [dfocal, dcx, dcy, k1, k2]
    rms_board: float
    rms_tracks: float
    n_board_obs: int
    n_track_obs: int
    iterations: int
    cost0: float
    cost: float
    track_err: np.ndarray = None   # per track observation reprojection error [px]
    board_err: np.ndarray = None


def _skew(v):
    O = np.zeros(v.shape[:-1] + (3, 3))
    O[..., 0, 1], O[..., 0, 2] = -v[..., 2], v[..., 1]
    O[..., 1, 0], O[..., 1, 2] = v[..., 2], -v[..., 0]
    O[..., 2, 0], O[..., 2, 1] = -v[..., 1], v[..., 0]
    return O


def _right_jac(r):
    th = np.linalg.norm(r, axis=1)
    S = _skew(r)
    small = th < 1e-7
    t = np.where(small, 1.0, th)
    a = np.where(small, 0.5, (1 - np.cos(t)) / t ** 2)
    b = np.where(small, 1.0 / 6.0, (t - np.sin(t)) / t ** 3)
    return np.eye(3)[None] - a[:, None, None] * S + b[:, None, None] * (S @ S)


INTR_NAMES = ("dfocal", "dcx", "dcy", "k1", "k2")


def _scatter2(cols, blocks, N):
    """Sum blocks[m] (C,C) into an (N,N) matrix at rows/cols cols[m] (C,)."""
    idx = cols[:, :, None] * N + cols[:, None, :]
    return np.bincount(idx.ravel(), blocks.ravel(), minlength=N * N).reshape(N, N)


def _scatter_blocks(index, blocks, P):
    """Sum (M,3,3) blocks into (P,3,3) by point index."""
    flat = (index[:, None] * 9 + np.arange(9)).ravel()
    return np.bincount(flat, blocks.reshape(-1, 9).ravel(), minlength=P * 9).reshape(P, 3, 3)


def bundle_adjust(Ks: np.ndarray, T_wc0: np.ndarray, board_obs: dict | None = None,
                  tracks: Tracks | None = None, rel_priors: list | None = None,
                  abs_priors: np.ndarray | None = None, refine_intrinsics=False,
                  intr0=None, intr_sigma=None, huber_px: float = 1.0, max_iter: int = 40,
                  verbose: bool = False, fix_poses=False) -> BAResult:
    """Levenberg-Marquardt bundle adjustment with analytic Jacobians and a Schur complement
    (points eliminated), so it stays fast with tens of thousands of points.

    Ks          (F,3,3) per-frame intrinsics (ARKit; can vary with focus)
    T_wc0       (F,4,4) initial camera->world poses
    board_obs   {frame: (uv (N,2), XYZ_world (N,3))}  known 3D points (ChArUco corners)
    tracks      Tracks; its world points X are optimised
    rel_priors  [(i, j, T_ji, sigma_rot_rad, sigma_t_m)]  soft relative-motion constraints
    abs_priors  (F,3) expected camera centres (2 cm sigma) to fix the gauge without a board
    refine_intrinsics  False | True (all five) | iterable of names from INTR_NAMES
    intr_sigma  prior sigmas for the five global intrinsic corrections
    fix_poses   False | True (only points move) | (F,) bool mask of frames whose pose stays fixed |
                (F, 6) mask per pose parameter [rx ry rz tx ty tz]
    """
    from scipy.sparse import csr_matrix
    from scipy.linalg import cho_factor, cho_solve

    F = len(T_wc0)
    T_cw0 = inv_T(np.asarray(T_wc0, float))
    r = Rotation.from_matrix(T_cw0[:, :3, :3]).as_rotvec()
    t = T_cw0[:, :3, 3].copy()
    intr = np.zeros(5) if intr0 is None else np.asarray(intr0, float).copy()
    if refine_intrinsics is True:
        active = list(range(5))
    elif refine_intrinsics:
        active = [INTR_NAMES.index(n) for n in refine_intrinsics]
    else:
        active = []
    nI = len(active)
    isig = np.array([0.003, 4.0, 4.0, 0.05, 0.01]) if intr_sigma is None else np.asarray(intr_sigma, float)
    intr_prior = intr.copy()

    # flatten observations
    bc, buv, bX = [], [], []
    for i, (uv, X) in (board_obs or {}).items():
        bc.append(np.full(len(uv), i))
        buv.append(uv)
        bX.append(X)
    bc = np.concatenate(bc).astype(int) if bc else np.zeros(0, int)
    buv = np.concatenate(buv) if buv else np.zeros((0, 2))
    bX = np.concatenate(bX) if bX else np.zeros((0, 3))
    use_t = tracks is not None and tracks.n_points > 0
    tc = tracks.obs_cam.astype(int) if use_t else np.zeros(0, int)
    tp = tracks.obs_pt.astype(int) if use_t else np.zeros(0, int)
    tuv = tracks.obs_uv if use_t else np.zeros((0, 2))
    X = tracks.X.copy() if use_t else np.zeros((0, 3))
    P = len(X)
    rel = rel_priors or []
    fxa, fya, cxa, cya = Ks[:, 0, 0], Ks[:, 1, 1], Ks[:, 0, 2], Ks[:, 1, 2]
    Nc = 6 * F + nI
    if fix_poses is False or fix_poses is True:
        fixed = np.full((F, 6), bool(fix_poses))
    else:                            # (F,) per frame or (F, 6) per parameter [rx ry rz tx ty tz]
        fixed = np.asarray(fix_poses, bool)
        fixed = np.repeat(fixed[:, None], 6, 1) if fixed.ndim == 1 else fixed
    free = np.r_[~fixed.ravel(), np.ones(nI, bool)]     # camera unknowns that move

    def reproj(R, t_, Xw_b, Xw_t, intr_, with_jac):
        out = []
        for cams, Xw, uv, has_pt in ((bc, Xw_b, buv, False), (tc, Xw_t, tuv, True)):
            if len(cams) == 0:
                out.append(None)
                continue
            Rc = R[cams]
            Xc = np.einsum("nij,nj->ni", Rc, Xw) + t_[cams]
            z = Xc[:, 2]
            z = np.where(np.abs(z) < 1e-6, 1e-6, z)
            x, y = Xc[:, 0] / z, Xc[:, 1] / z
            r2 = x * x + y * y
            k1, k2 = intr_[3], intr_[4]
            d = 1 + k1 * r2 + k2 * r2 * r2
            S = 1 + intr_[0]
            fx, fy = fxa[cams] * S, fya[cams] * S
            u = fx * x * d + cxa[cams] + intr_[1]
            v = fy * y * d + cya[cams] + intr_[2]
            res = np.stack([u, v], 1) - uv
            if not with_jac:
                out.append((res, None, None, None))
                continue
            dd = 2 * (k1 + 2 * k2 * r2)
            A = np.empty((len(cams), 2, 2))
            A[:, 0, 0] = fx * (d + x * dd * x)
            A[:, 0, 1] = fx * x * dd * y
            A[:, 1, 0] = fy * y * dd * x
            A[:, 1, 1] = fy * (d + y * dd * y)
            B = np.zeros((len(cams), 2, 3))
            B[:, 0, 0] = 1 / z
            B[:, 0, 2] = -x / z
            B[:, 1, 1] = 1 / z
            B[:, 1, 2] = -y / z
            G = A @ B                                                   # d(uv)/dXc
            dXdr = -Rc @ _skew(Xw) @ Jr[cams]
            Jpose = np.concatenate([G @ dXdr, G], 2)                    # (M,2,6)
            Jint = np.zeros((len(cams), 2, 5))
            Jint[:, 0, 0] = fxa[cams] * x * d
            Jint[:, 1, 0] = fya[cams] * y * d
            Jint[:, 0, 1] = 1
            Jint[:, 1, 2] = 1
            Jint[:, 0, 3] = fx * x * r2
            Jint[:, 1, 3] = fy * y * r2
            Jint[:, 0, 4] = fx * x * r2 * r2
            Jint[:, 1, 4] = fy * y * r2 * r2
            Jpt = G @ Rc if has_pt else None
            out.append((res, Jpose, Jint[:, :, active], Jpt))
        return out

    def rel_res(ri, ti, rj, tj):
        Ri = Rotation.from_rotvec(ri).as_matrix()
        Rj = Rotation.from_rotvec(rj).as_matrix()
        Rest = Rj @ np.swapaxes(Ri, 1, 2)
        test = tj - np.einsum("nij,nj->ni", Rest, ti)
        e_r = Rotation.from_matrix(Rest @ np.swapaxes(rel_R, 1, 2)).as_rotvec() / rel_sr[:, None]
        e_t = (test - rel_t) / rel_st[:, None]
        return np.concatenate([e_r, e_t], 1)

    def abs_res(r_, t_):
        R = Rotation.from_rotvec(r_).as_matrix()
        C = -np.einsum("nji,nj->ni", R, t_)
        return (C - abs_priors) / 0.02

    if rel:
        rel_i = np.array([p[0] for p in rel])
        rel_j = np.array([p[1] for p in rel])
        rel_R = np.array([p[2][:3, :3] for p in rel])
        rel_t = np.array([p[2][:3, 3] for p in rel])
        rel_sr = np.array([p[3] for p in rel])
        rel_st = np.array([p[4] for p in rel])

    def robust(res):
        e = np.linalg.norm(res, axis=1)
        w = np.where(e <= huber_px, 1.0, huber_px / np.maximum(e, 1e-12))
        cost = np.where(e <= huber_px, 0.5 * e * e, huber_px * (e - 0.5 * huber_px))
        return w, cost.sum()

    def total_cost(r_, t_, X_, intr_):
        R = Rotation.from_rotvec(r_).as_matrix()
        c = 0.0
        for item in reproj(R, t_, bX, X_[tp] if use_t else None, intr_, False):
            if item is not None:
                c += robust(item[0])[1]
        if rel:
            c += 0.5 * (rel_res(r_[rel_i], t_[rel_i], r_[rel_j], t_[rel_j]) ** 2).sum()
        if abs_priors is not None:
            c += 0.5 * (abs_res(r_, t_) ** 2).sum()
        if nI:
            c += 0.5 * (((intr_ - intr_prior)[active] / isig[active]) ** 2).sum()
        return c

    lam = 1e-3
    cost = total_cost(r, t, X, intr)
    cost0 = cost
    it = 0
    for it in range(max_iter):
        R = Rotation.from_rotvec(r).as_matrix()
        Jr = _right_jac(r)
        t_cur = t
        items = reproj(R, t, bX, X[tp] if use_t else None, intr, True)
        Hcc = np.zeros((Nc, Nc))
        gc = np.zeros(Nc)
        V = np.zeros((P, 3, 3))
        gp = np.zeros((P, 3))
        Hcp = None
        for k, item in enumerate(items):
            if item is None:
                continue
            res, Jpose, Jint, Jpt = item
            cams = bc if k == 0 else tc
            w, _ = robust(res)
            sw = np.sqrt(w)[:, None]
            res_w = res * sw
            Jpose = Jpose * sw[:, :, None]
            Jint = Jint * sw[:, :, None]
            Jc = np.concatenate([Jpose, Jint], 2)                     # (M,2,6+nI)
            cols = np.concatenate([6 * cams[:, None] + np.arange(6), np.broadcast_to(6 * F + np.arange(nI), (len(cams), nI))], 1)
            blk = np.einsum("mai,maj->mij", Jc, Jc)
            Hcc += _scatter2(cols, blk, Nc)
            gc += np.bincount(cols.ravel(), np.einsum("mai,ma->mi", Jc, res_w).ravel(), minlength=Nc)
            if k == 1:
                Jp = Jpt * sw[:, :, None]
                V += _scatter_blocks(tp, np.einsum("mai,maj->mij", Jp, Jp), P)
                gp += np.bincount((3 * tp[:, None] + np.arange(3)).ravel(), np.einsum("mai,ma->mi", Jp, res_w).ravel(),
                                  minlength=3 * P).reshape(P, 3)
                hcp = np.einsum("mai,maj->mij", Jc, Jp)                 # (M, 6+nI, 3)
                rows = np.repeat(cols[:, :, None], 3, 2).ravel()
                cl = np.repeat((3 * tp[:, None] + np.arange(3))[:, None, :], cols.shape[1], 1).ravel()
                Hcp = csr_matrix((hcp.ravel(), (rows, cl)), shape=(Nc, 3 * P))
        # priors (numeric Jacobians, vectorised)
        if rel:
            e0 = rel_res(r[rel_i], t[rel_i], r[rel_j], t[rel_j])
            Jrel = np.zeros((len(rel), 6, 12))
            h = 1e-6
            for s in range(12):
                args = [r[rel_i].copy(), t[rel_i].copy(), r[rel_j].copy(), t[rel_j].copy()]
                slot, comp = divmod(s, 3)
                args[slot][:, comp] += h
                ep = rel_res(*args)
                args[slot][:, comp] -= 2 * h
                em = rel_res(*args)
                Jrel[:, :, s] = (ep - em) / (2 * h)
            # parameter order inside one pose block is [r(3), t(3)]
            colmap = np.concatenate([6 * rel_i[:, None] + np.arange(3), 6 * rel_i[:, None] + 3 + np.arange(3),
                                     6 * rel_j[:, None] + np.arange(3), 6 * rel_j[:, None] + 3 + np.arange(3)], 1)
            Hcc += _scatter2(colmap, np.einsum("nai,naj->nij", Jrel, Jrel), Nc)
            gc += np.bincount(colmap.ravel(), np.einsum("nai,na->ni", Jrel, e0).ravel(), minlength=Nc)
        if abs_priors is not None:
            e0 = abs_res(r, t)
            Jab = np.zeros((F, 3, 6))
            h = 1e-6
            for s in range(6):
                rr, tt = r.copy(), t.copy()
                (rr if s < 3 else tt)[:, s % 3] += h
                ep = abs_res(rr, tt)
                (rr if s < 3 else tt)[:, s % 3] -= 2 * h
                em = abs_res(rr, tt)
                Jab[:, :, s] = (ep - em) / (2 * h)
            colmap = 6 * np.arange(F)[:, None] + np.arange(6)
            Hcc += _scatter2(colmap, np.einsum("nai,naj->nij", Jab, Jab), Nc)
            gc += np.bincount(colmap.ravel(), np.einsum("nai,na->ni", Jab, e0).ravel(), minlength=Nc)
        if nI:
            ii = 6 * F + np.arange(nI)
            Hcc[ii, ii] += 1.0 / isig[active] ** 2
            gc[ii] += (intr - intr_prior)[active] / isig[active] ** 2

        # LM step with Schur complement
        improved = False
        for _try in range(10):
            Hd = Hcc + lam * np.diag(np.diag(Hcc) + 1e-9)
            if P:
                Vd = V + lam * (V * np.eye(3)[None]) + 1e-12 * np.eye(3)[None]
                Vinv = np.linalg.inv(Vd)
                rows = (3 * np.arange(P)[:, None, None] + np.arange(3)[None, :, None]).repeat(3, 2).ravel()
                cols = (3 * np.arange(P)[:, None, None] + np.arange(3)[None, None, :]).repeat(3, 1).ravel()
                Vi = csr_matrix((Vinv.ravel(), (rows, cols)), shape=(3 * P, 3 * P))
                HV = Hcp @ Vi
                Sm = Hd - (HV @ Hcp.T).toarray()
                rhs = -gc + HV @ gp.ravel()
            else:
                Sm, rhs = Hd, -gc
            dc = np.zeros(Nc)
            if free.any():          # fixed poses: solve the reduced camera system for the free unknowns only
                try:
                    dc[free] = cho_solve(cho_factor(Sm[np.ix_(free, free)]), rhs[free])
                except np.linalg.LinAlgError:
                    lam *= 10
                    continue
            dp = (Vi @ (-gp.ravel() - Hcp.T @ dc)).reshape(P, 3) if P else np.zeros((0, 3))
            r_n = r + dc[:6 * F].reshape(F, 6)[:, :3]
            t_n = t + dc[:6 * F].reshape(F, 6)[:, 3:]
            intr_n = intr.copy()
            if nI:
                intr_n[active] += dc[6 * F:]
            X_n = X + dp
            c_n = total_cost(r_n, t_n, X_n, intr_n)
            if c_n < cost:
                rel_dec = (cost - c_n) / max(cost, 1e-12)
                r, t, X, intr, cost = r_n, t_n, X_n, intr_n, c_n
                lam = max(lam / 3, 1e-9)
                improved = True
                break
            lam *= 4
        if verbose:
            log.info("BA it %d cost %.4g lam %.1e", it, cost, lam)
        if not improved or rel_dec < 1e-7:
            break

    R = Rotation.from_rotvec(r).as_matrix()
    t_cur = t
    items = reproj(R, t, bX, X[tp] if use_t else None, intr, False)
    berr = np.linalg.norm(items[0][0], axis=1) if items[0] is not None else np.zeros(0)
    terr = np.linalg.norm(items[1][0], axis=1) if items[1] is not None else np.zeros(0)
    T_cw = np.tile(np.eye(4), (F, 1, 1))
    T_cw[:, :3, :3] = R
    T_cw[:, :3, 3] = t
    rms = lambda e: float(np.sqrt(np.mean(e ** 2))) if len(e) else float("nan")
    return BAResult(inv_T(T_cw), X, intr, rms(berr), rms(np.minimum(terr, 5.0)), len(berr), len(terr),
                    it + 1, float(cost0), float(cost), terr, berr)


def relative_priors(T_wc_arkit: np.ndarray, scale: float, sigma_rot_deg: float = 0.3, sigma_t_mm: float = 1.5):
    """ARKit relative motion between consecutive keyframes as soft BA constraints.
    Sigma grows with the size of the step (VIO error accumulates with motion)."""
    pri = []
    T_cw = inv_T(T_wc_arkit)
    for i in range(len(T_wc_arkit) - 1):
        j = i + 1
        Tji = T_cw[j] @ T_wc_arkit[i]
        Tji = Tji.copy()
        Tji[:3, 3] *= scale
        step = np.linalg.norm(Tji[:3, 3])
        st = sigma_t_mm / 1000.0 + 0.02 * step
        sr = np.radians(sigma_rot_deg) + 0.01 * np.linalg.norm(Rotation.from_matrix(Tji[:3, :3]).as_rotvec())
        pri.append((i, j, Tji, sr, st))
    return pri


def undistort_maps(K_src: np.ndarray, intr: np.ndarray, size_wh, K_dst: np.ndarray | None = None):
    """Remap tables that turn an image taken with (K_src, intr) into an ideal pinhole image
    with intrinsics K_dst (default: the corrected focal/principal point, no distortion)."""
    s = 1 + intr[0]
    Kc = K_src.copy()
    Kc[0, 0] *= s
    Kc[1, 1] *= s
    Kc[0, 2] += intr[1]
    Kc[1, 2] += intr[2]
    K_dst = Kc if K_dst is None else K_dst
    dist = np.array([intr[3], intr[4], 0, 0, 0], float)
    m1, m2 = cv2.initUndistortRectifyMap(Kc, dist, None, K_dst, tuple(int(v) for v in size_wh), cv2.CV_32FC1)
    return m1, m2, K_dst


In [ ]:
%%writefile /content/partscan/poses.py
"""Camera pose estimation: ChArUco board (if present) + photogrammetric refinement."""
from __future__ import annotations

import logging
from dataclasses import dataclass, field

import cv2
import numpy as np

from . import bundle
from .board import board_to_part, detect, make_board, make_detector, robust_pnp
from .config import Config
from .geometry import apply_sim3_to_pose, inv_T, project, umeyama_sim3

log = logging.getLogger("partscan")


@dataclass
class PoseState:
    world: str                       # "board" (part frame from the board) or "arkit"
    T_wc: np.ndarray                 # (F,4,4) current best poses
    T_wc_arkit: np.ndarray           # (F,4,4) raw ARKit poses
    board_obs: dict = field(default_factory=dict)   # i -> (uv, XYZ_world)
    arkit_scale: float = 1.0         # world metres per ARKit metre (from the board)
    intr: np.ndarray = field(default_factory=lambda: np.zeros(5))
    undistorted: bool = False
    stats: dict = field(default_factory=dict)
    tracks: object = None            # bundle.Tracks after feature BA


def detect_board(kf, cfg: Config, progress=None) -> dict:
    spec = cfg.board
    board = make_board(spec)
    det = make_detector(board)
    T_pb = board_to_part(spec)
    out = {}
    for i, f in enumerate(kf):
        gray = cv2.cvtColor(f.load_rgb(), cv2.COLOR_RGB2GRAY)
        d = detect(gray, board, det)
        if progress is not None and i % 10 == 0:
            progress(i / len(kf), f"looking for the calibration board {i}/{len(kf)}")
        if d is None or len(d[0]) < cfg.min_board_corners:
            continue
        uv, ids, obj = d
        T_cb, keep, rms = robust_pnp(obj, uv, f.K)
        if T_cb is None or keep.sum() < cfg.min_board_corners or rms > 2.5:
            continue
        out[i] = dict(uv=uv[keep], X=obj[keep] @ T_pb[:3, :3].T + T_pb[:3, 3], T_wc=T_pb @ inv_T(T_cb), rms=rms)
    return out


def init_poses(kf, cfg: Config, progress=None) -> PoseState:
    T_ark = np.array([f.T_wc for f in kf])
    det = detect_board(kf, cfg, progress) if cfg.use_board else {}
    n_ok = len(det)
    if n_ok >= max(6, 0.25 * len(kf)):
        ids = sorted(det)
        s, R, t = umeyama_sim3(T_ark[ids, :3, 3], np.array([det[i]["T_wc"][:3, 3] for i in ids]))
        T0 = np.array([apply_sim3_to_pose(s, R, t, T) for T in T_ark])
        for i in ids:
            T0[i] = det[i]["T_wc"]
        st = PoseState("board", T0, T_ark, {i: (det[i]["uv"], det[i]["X"]) for i in ids}, float(s))
        st.stats.update(board_frames=n_ok, board_pnp_rms_px=float(np.median([det[i]["rms"] for i in ids])),
                        arkit_scale_error_pct=float((1.0 / s - 1.0) * 100.0))
        log.info("board found in %d/%d keyframes; phone scale error vs board %.2f%%", n_ok, len(kf),
                 st.stats["arkit_scale_error_pct"])
        return st
    if cfg.use_board and n_ok:
        log.warning("board seen in only %d/%d keyframes - ignoring it", n_ok, len(kf))
    st = PoseState("arkit", T_ark.copy(), T_ark, {}, 1.0)
    st.stats.update(board_frames=n_ok)
    return st


def refine_with_board(kf, st: PoseState, cfg: Config) -> PoseState:
    """Board-only BA (+ global focal/principal-point correction and radial distortion)."""
    Ks = np.array([f.K for f in kf])
    W = kf[0].rgb_size[0]
    pri = bundle.relative_priors(st.T_wc_arkit, st.arkit_scale)
    kw = dict(board_obs=st.board_obs, rel_priors=pri)
    res0 = bundle.bundle_adjust(Ks, st.T_wc, **kw)
    res = res0
    if cfg.refine_intrinsics:
        sig = np.array([0.003, 4.0 * W / 1920, 4.0 * W / 1920, 0.05, 0.01])
        res1 = bundle.bundle_adjust(Ks, st.T_wc, refine_intrinsics=("dfocal", "dcx", "dcy", "k1"),
                                    intr_sigma=sig, **kw)
        if res1.rms_board < 0.97 * res0.rms_board or abs(res1.intr[3]) > 0.004:
            res = res1
    st.T_wc = res.T_wc
    st.intr = res.intr
    st.stats.update(board_ba_rms_px=res.rms_board, board_ba_rms_px_no_intr=res0.rms_board,
                    k1=float(res.intr[3]), focal_correction_pct=float(res.intr[0] * 100))
    log.info("board BA: %.3f px (fixed intrinsics %.3f px), k1=%.4f", res.rms_board, res0.rms_board, res.intr[3])
    return st


def undistort_keyframes(kf, st: PoseState, min_k1: float = 0.002) -> None:
    """Resample RGB, LiDAR depth and confidence to an ideal pinhole camera (in place)."""
    intr = st.intr
    if abs(intr[3]) < min_k1 and abs(intr[4]) < min_k1 and abs(intr[0]) < 1e-4 and abs(intr[1]) < 0.05 and abs(intr[2]) < 0.05:
        return
    for f in kf:
        rgb = f.load_rgb()
        H, W = rgb.shape[:2]
        m1, m2, Kn = bundle.undistort_maps(f.K, intr, (W, H))
        f.extra["K_orig"] = f.K.copy()
        f.extra["undist_intr"] = np.asarray(intr, float).copy()
        valid = cv2.remap(np.ones((H, W), np.uint8), m1, m2, cv2.INTER_NEAREST, borderValue=0)
        f.rgb = cv2.remap(rgb, m1, m2, cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT)
        f.extra["valid"] = valid.astype(bool)
        if f.depth is not None:
            dh, dw = f.depth.shape
            from .geometry import K_for_size
            Knd = K_for_size(Kn, (W, H), (dw, dh))   # corrected pinhole model at depth resolution
            d1, d2, _ = bundle.undistort_maps(Knd, np.r_[0, 0, 0, intr[3:]], (dw, dh), K_dst=Knd)
            f.depth = cv2.remap(f.depth, d1, d2, cv2.INTER_NEAREST, borderValue=0)
            f.conf = cv2.remap(f.conf, d1, d2, cv2.INTER_NEAREST, borderValue=0)
        f.K = Kn
    # board observations must now be in the undistorted image
    for i, (uv, X) in list(st.board_obs.items()):
        st.board_obs[i] = (_undistort_px(uv, kf[i].K, st.intr), X)
    st.undistorted = True
    st.intr = np.zeros(5)


def _undistort_px(uv, K_new, intr):
    # K_new is the corrected pinhole matrix; invert the radial model in normalised coordinates
    xd = (uv - K_new[:2, 2]) / [K_new[0, 0], K_new[1, 1]]
    xu = xd.copy()
    for _ in range(10):
        r2 = (xu ** 2).sum(1, keepdims=True)
        xu = xd / (1 + intr[3] * r2 + intr[4] * r2 * r2)
    return xu * [K_new[0, 0], K_new[1, 1]] + K_new[:2, 2]


def roi_mask(f, T_wc, roi_min, roi_max, grow_px: int = 25):
    """Image mask covering the projection of an axis-aligned box (part frame)."""
    W, H = f.rgb_size
    c = np.array([[x, y, z] for x in (roi_min[0], roi_max[0]) for y in (roi_min[1], roi_max[1])
                  for z in (roi_min[2], roi_max[2])])
    uv, z = project(f.K, inv_T(T_wc), c)
    m = np.zeros((H, W), np.uint8)
    if (z <= 0.01).any():
        m[:] = 1
        return m
    hull = cv2.convexHull(uv.astype(np.float32)).astype(np.int32)
    cv2.fillConvexPoly(m, hull, 1)
    if grow_px:
        m = cv2.dilate(m, np.ones((2 * grow_px + 1, 2 * grow_px + 1), np.uint8))
    return m


def feature_refine(kf, st: PoseState, roi_min, roi_max, cfg: Config, progress=None) -> PoseState:
    """SIFT tracks around the part + joint bundle adjustment."""
    feats = []
    grow = np.array([0.04, 0.04, 0.0])
    for i, f in enumerate(kf):
        gray = cv2.cvtColor(f.load_rgb(), cv2.COLOR_RGB2GRAY)
        m = roi_mask(f, st.T_wc[i], roi_min - grow, roi_max + grow * [1, 1, 0])
        if "valid" in f.extra:
            m &= cv2.erode(f.extra["valid"].astype(np.uint8), np.ones((9, 9), np.uint8))
        feats.append(bundle.extract_features(gray, cfg.sift_features, m))
        if progress is not None and i % 10 == 0:
            progress(0.3 * i / len(kf), f"features {i}/{len(kf)}")
    pairs = bundle.select_pairs(list(st.T_wc), cfg.match_neighbors)
    raw_tracks, n_pairs = bundle.build_tracks(
        feats, pairs, progress=None if progress is None else (lambda p, m: progress(0.3 + 0.4 * p, m)))
    Ks = np.array([f.K for f in kf])
    T_cw = inv_T(st.T_wc)
    tracks = bundle.triangulate_tracks(raw_tracks, Ks, T_cw, min_views=3, max_reproj=6.0)
    if tracks.n_points > cfg.ba_max_points:
        keep_pts = _subsample_points(tracks, cfg.ba_max_points)
        tracks = _filter_tracks(tracks, np.isin(tracks.obs_pt, keep_pts))
    log.info("feature tracks: %d points (%d obs) from %d/%d verified pairs", tracks.n_points, len(tracks.obs_cam),
             n_pairs, len(pairs))
    if tracks.n_points < 50:
        st.stats["feature_points"] = int(tracks.n_points)
        log.warning("too few feature tracks for photogrammetric refinement")
        return st
    pri = bundle.relative_priors(st.T_wc_arkit, st.arkit_scale)
    absp = None if st.world == "board" else np.array([T[:3, 3] for T in st.T_wc])
    # With the board in (almost) every frame, the board-only solution is already the most accurate:
    # keep those poses and refine the tie points plus the few frames that did not see the board
    # (e.g. low views straight into a side hole) - those only have the phone's drifting tracking.
    # Their orientation from the board BA (board frames + the phone's relative motion) is already good to
    # a few hundredths of a degree; a handful of tie points can fix their position but would spoil it.
    fix = False
    if st.world == "board" and len(st.board_obs) >= 0.8 * len(kf):
        on_board = np.array([i in st.board_obs for i in range(len(kf))])
        fix = np.zeros((len(kf), 6), bool)
        fix[on_board] = True
        fix[~on_board, :3] = True
    st.stats["poses_fixed_to_board"] = fix is not False
    st.stats["poses_refined_without_board"] = int(len(kf) - len(st.board_obs)) if fix is not False else 0
    if progress is not None:
        progress(0.75, "bundle adjustment")
    res = bundle.bundle_adjust(Ks, st.T_wc, board_obs=st.board_obs, tracks=tracks, rel_priors=pri,
                               abs_priors=absp, huber_px=1.5, fix_poses=fix)
    # drop outliers, re-triangulate with the refined poses, polish
    good = res.track_err < 2.0
    tracks.X = res.X
    tracks = _filter_tracks(tracks, good)
    if progress is not None:
        progress(0.9, "bundle adjustment (polish)")
    res = bundle.bundle_adjust(Ks, res.T_wc, board_obs=st.board_obs, tracks=tracks, rel_priors=pri,
                               abs_priors=absp, huber_px=1.0, fix_poses=fix)
    tracks.X = res.X
    st.T_wc = res.T_wc
    st.tracks = tracks
    st.stats.update(feature_points=int(tracks.n_points), feature_obs=int(len(tracks.obs_cam)),
                    ba_rms_tracks_px=res.rms_tracks, ba_rms_board_px=res.rms_board)
    log.info("feature BA: tracks %.3f px, board %.3f px", res.rms_tracks, res.rms_board)
    return st


def _subsample_points(tracks, n):
    cnt = np.bincount(tracks.obs_pt, minlength=tracks.n_points)
    order = np.argsort(-cnt + np.random.default_rng(0).random(len(cnt)) * 0.5)
    return order[:n]


def _filter_tracks(tracks, obs_keep):
    """Keep selected observations; drop points left with < 2 views; re-index."""
    cam, pt, uv = tracks.obs_cam[obs_keep], tracks.obs_pt[obs_keep], tracks.obs_uv[obs_keep]
    cnt = np.bincount(pt, minlength=tracks.n_points)
    ok_pt = cnt >= 2
    m = ok_pt[pt]
    cam, pt, uv = cam[m], pt[m], uv[m]
    new_id = -np.ones(tracks.n_points, int)
    new_id[np.nonzero(ok_pt)[0]] = np.arange(ok_pt.sum())
    return bundle.Tracks(cam, new_id[pt], uv, tracks.X[ok_pt], int(ok_pt.sum()))


In [ ]:
%%writefile /content/partscan/segment.py
"""Finding the part: coarse 3D localisation from LiDAR, then per-frame silhouettes.

Silhouettes matter for accuracy: depth sensors smear depth across object edges, while a
good image mask is sharp to a pixel. The fusion step carves away anything that projects
outside the silhouettes (a robust visual hull), which pins down the outline of the part.
"""
from __future__ import annotations

import logging

import cv2
import numpy as np
from scipy import ndimage

from .geometry import (K_for_size, backproject_depth, closest_point_to_lines, inv_T, project, ransac_plane,
                       transform_points)

log = logging.getLogger("partscan")


# --------------------------------------------------------------------------- coarse 3D
def look_at_point(T_wc: np.ndarray) -> np.ndarray:
    return closest_point_to_lines(T_wc[:, :3, 3], T_wc[:, :3, 2])


def lidar_points(kf, T_wc, stride: int = 2, min_conf: int = 2, max_range: float = 1.5, align=None):
    """World points from the LiDAR maps of all keyframes (optionally per-frame corrected)."""
    pts = []
    for i, f in enumerate(kf):
        if f.depth is None:
            continue
        d = f.depth.copy()
        if align is not None and align[i] is not None:
            a, b = align[i]
            d = np.where(d > 0, a * d + b, 0)
        Kd = K_for_size(f.K, f.rgb_size, (d.shape[1], d.shape[0]))
        m = (f.conf >= min_conf) & (d < max_range) & (d > 0.05)
        P, _ = backproject_depth(d, Kd, T_wc[i], m, stride)
        pts.append(P)
    return np.concatenate(pts) if pts else np.zeros((0, 3))


def plane_lidar_alignment(kf, T_wc, n=np.array([0, 0, 1.0]), d0: float = 0.0, region=None, stride: int = 4):
    """Per-frame scale/offset of the LiDAR depth so that the support plane lands exactly on
    n.x = d0. Uses a trimmed fit (the part itself is an outlier). Returns list of (a, b)."""
    out = []
    for i, f in enumerate(kf):
        if f.depth is None:
            out.append(None)
            continue
        dh, dw = f.depth.shape
        Kd = K_for_size(f.K, f.rgb_size, (dw, dh))
        v, u = np.mgrid[0:dh:stride, 0:dw:stride]
        u, v = u.ravel(), v.ravel()
        dm = f.depth[v, u]
        ok = (f.conf[v, u] >= 1) & (dm > 0.05)
        rays_c = np.stack([(u - Kd[0, 2]) / Kd[0, 0], (v - Kd[1, 2]) / Kd[1, 1], np.ones(len(u))], 1)
        R, C = T_wc[i][:3, :3], T_wc[i][:3, 3]
        rw = rays_c @ R.T
        den = rw @ n
        lam = (d0 - C @ n) / np.where(np.abs(den) < 1e-9, 1e-9, den)   # z-depth since rays_c.z = 1
        P = C + rw * lam[:, None]
        ok &= (lam > 0.05) & (lam < 1.5)
        if region is not None:
            xmin, xmax, ymin, ymax = region
            ok &= (P[:, 0] > xmin) & (P[:, 0] < xmax) & (P[:, 1] > ymin) & (P[:, 1] < ymax)
        if ok.sum() < 30:
            out.append(None)
            continue
        x, y = dm[ok], lam[ok]
        a, b = 1.0, 0.0
        keep = np.ones(len(x), bool)
        for _ in range(4):
            A = np.stack([x[keep], np.ones(keep.sum())], 1)
            a, b = np.linalg.lstsq(A, y[keep], rcond=None)[0]
            res = np.abs(a * x + b - y)
            keep = res <= np.quantile(res, 0.7) + 1e-4
        out.append((float(a), float(b)) if 0.8 < a < 1.2 and abs(b) < 0.05 else None)
    return out


def coarse_part(kf, T_wc, world: str, board_extent=None, align=None):
    """Locate the part. Returns dict with T_pw (part-from-world), roi_min/max in part frame,
    coarse part points (part frame) and the look-at prior."""
    look = look_at_point(T_wc)
    pts_w = lidar_points(kf, T_wc, align=align)
    if world == "board":
        T_pw = np.eye(4)
    else:
        near = pts_w[np.linalg.norm(pts_w - look, axis=1) < 0.3]
        if len(near) < 100:
            raise RuntimeError("Not enough LiDAR points near the scanned area. Was the object at least 25 cm away?")
        n, d, inl = ransac_plane(near, thresh=0.004, iters=600, normal_hint=np.array([0, 1.0, 0]), max_angle_deg=25)
        z = n
        x = np.cross([0, 0, 1.0], z)
        if np.linalg.norm(x) < 1e-6:
            x = np.cross([1.0, 0, 0], z)
        x /= np.linalg.norm(x)
        y = np.cross(z, x)
        o = look - (look @ n - d) * n
        T_wp = np.eye(4)
        T_wp[:3, :3] = np.stack([x, y, z], 1)
        T_wp[:3, 3] = o
        T_pw = inv_T(T_wp)
    P = transform_points(T_pw, pts_w)
    lk = transform_points(T_pw, look[None])[0]
    # height threshold from the spread of the plane points
    ring = P[np.hypot(P[:, 0] - lk[0], P[:, 1] - lk[1]) < 0.2]
    zz = ring[:, 2]
    plane_z = zz[np.abs(zz) < 0.01]
    sig = 1.4826 * np.median(np.abs(plane_z - np.median(plane_z))) if len(plane_z) > 50 else 0.003
    zmin = float(np.clip(4 * sig, 0.0025, 0.008))
    cand = P[(P[:, 2] > zmin) & (P[:, 2] < 0.35) & (np.hypot(P[:, 0] - lk[0], P[:, 1] - lk[1]) < 0.25)]
    if board_extent is not None:
        xmin, xmax, ymin, ymax = board_extent
        m = 0.08   # parts may overhang the board
        cand = cand[(cand[:, 0] > xmin - m) & (cand[:, 0] < xmax + m) & (cand[:, 1] > ymin - m) & (cand[:, 1] < ymax + m)]
    info = dict(T_pw=T_pw, look_at=lk, plane_sigma=sig, z_threshold=zmin)
    if len(cand) < 30:
        log.warning("LiDAR barely sees the part; using a default search box around the scan centre")
        c = lk.copy()
        c[2] = 0
        info.update(roi_min=c + [-0.07, -0.07, -0.003], roi_max=c + [0.07, 0.07, 0.12], points=np.zeros((0, 3)))
        return info
    vox = 0.003
    lo = cand.min(0)
    idx = np.floor((cand - lo) / vox).astype(int)
    dims = idx.max(0) + 1
    occ = np.zeros(dims, np.int32)
    np.add.at(occ, tuple(idx.T), 1)
    lab, nlab = ndimage.label(occ >= 2, structure=np.ones((3, 3, 3)))
    if nlab == 0:
        lab, nlab = ndimage.label(occ >= 1, structure=np.ones((3, 3, 3)))
    pl = lab[tuple(idx.T)]
    best, best_s = None, -1
    for k in range(1, nlab + 1):
        sel = pl == k
        cnt = sel.sum()
        if cnt < 10:
            continue
        cxy = cand[sel, :2].mean(0)
        s = cnt / (1 + (np.linalg.norm(cxy - lk[:2]) / 0.05) ** 2)
        if s > best_s:
            best, best_s = sel, s
    if best is None:
        best = np.ones(len(cand), bool)
    part = cand[best]
    lo_p, hi_p = np.percentile(part, 0.5, 0), np.percentile(part, 99.5, 0)
    size = hi_p - lo_p
    margin = np.maximum(0.012, 0.25 * size)
    margin[2] = max(0.012, 0.25 * size[2])
    info.update(roi_min=np.array([lo_p[0] - margin[0], lo_p[1] - margin[1], -0.003]),
                roi_max=np.array([hi_p[0] + margin[0], hi_p[1] + margin[1], hi_p[2] + margin[2]]),
                points=part)
    return info


# --------------------------------------------------------------------------- masks helpers
def splat_mask(points_w, K, T_cw, size_wh, radius_m: float = 0.002):
    """Render points as discs -> coarse silhouette."""
    W, H = size_wh
    m = np.zeros((H, W), np.uint8)
    if len(points_w) == 0:
        return m
    uv, z = project(K, T_cw, points_w)
    ok = (z > 0.02) & (uv[:, 0] >= -50) & (uv[:, 0] < W + 50) & (uv[:, 1] >= -50) & (uv[:, 1] < H + 50)
    uv, z = uv[ok], z[ok]
    rad = np.clip(radius_m * K[0, 0] / z, 1, 60).astype(int)
    for (u, v), r in zip(uv.astype(int), rad):
        cv2.circle(m, (int(u), int(v)), int(r), 1, -1)
    return m


def mesh_silhouette(V, F, K, T_cw, size_wh):
    """Binary silhouette of a mesh (world vertices V) in an image."""
    W, H = size_wh
    Xc = V @ T_cw[:3, :3].T + T_cw[:3, 3]
    z = np.maximum(Xc[:, 2], 1e-4)
    uv = Xc[:, :2] / z[:, None] * [K[0, 0], K[1, 1]] + K[:2, 2]
    sil = np.zeros((H, W), np.uint8)
    tri = np.round(uv[F] * 8).astype(np.int64)
    ok = (Xc[F][:, :, 2] > 1e-3).all(1)
    tri = np.clip(tri[ok], -(2 ** 28), 2 ** 28).astype(np.int32)
    from .geometry import fill_triangles
    fill_triangles(sil, tri, 1, shift=3)
    return sil > 0


def mask_sdt(mask: np.ndarray) -> np.ndarray:
    """Signed distance (pixels) to the mask boundary: >0 outside, <0 inside."""
    m = mask.astype(np.uint8)
    out = cv2.distanceTransform(1 - m, cv2.DIST_L2, 5)
    ins = cv2.distanceTransform(m, cv2.DIST_L2, 5)
    return (out - ins).astype(np.float32)


def guided_filter(I: np.ndarray, p: np.ndarray, r: int, eps: float) -> np.ndarray:
    ksz = (2 * r + 1, 2 * r + 1)
    mean = lambda x: cv2.boxFilter(x, cv2.CV_32F, ksz)
    mI, mp = mean(I), mean(p)
    a = (mean(I * p) - mI * mp) / (mean(I * I) - mI * mI + eps)
    b = mp - a * mI
    return mean(a) * I + mean(b)


# --------------------------------------------------------------------------- segmenters
class GeometrySegmenter:
    """No AI: silhouette = projection of the coarse 3D points (blurry; not used for carving)."""
    name = "geometry"
    trusted = False

    def segment(self, f, coarse_mask, box, pos, neg):
        return coarse_mask.astype(bool), 0.5


class Sam2Segmenter:
    """SAM 2.1 (Apache-2.0) prompted with a box + points that come from the 3D localisation."""
    name = "sam2"
    trusted = True

    def __init__(self, checkpoint: str, device: str):
        import torch
        from transformers import Sam2Model, Sam2Processor
        self.torch = torch
        self.device = device
        dtype = torch.float32
        self.model = Sam2Model.from_pretrained(checkpoint, torch_dtype=dtype).to(device).eval()
        self.proc = Sam2Processor.from_pretrained(checkpoint)

    def segment(self, f, coarse_mask, box, pos, neg):
        torch = self.torch
        rgb0 = f.load_rgb()
        H0, W0 = rgb0.shape[:2]
        rk = int(f.extra.get("upright_k", 0))           # SAM sees an upright image
        rgb = np.ascontiguousarray(np.rot90(rgb0, rk))
        from .geometry import rot_points
        pos = rot_points(pos, rk, W0, H0)
        neg = rot_points(neg, rk, W0, H0)
        bx = rot_points([(box[0], box[1]), (box[2], box[3])], rk, W0, H0)
        box = [min(bx[0][0], bx[1][0]), min(bx[0][1], bx[1][1]), max(bx[0][0], bx[1][0]), max(bx[0][1], bx[1][1])]
        coarse_mask = np.rot90(coarse_mask, rk)
        pts = [list(map(float, p)) for p in pos] + [list(map(float, p)) for p in neg]
        lbl = [1] * len(pos) + [0] * len(neg)
        kw = dict(images=rgb, input_boxes=[[list(map(float, box))]], return_tensors="pt")
        if pts:
            kw.update(input_points=[[pts]], input_labels=[[lbl]])
        inputs = self.proc(**kw).to(self.device)
        with torch.inference_mode():
            out = self.model(**inputs, multimask_output=True)
        logits = self.proc.post_process_masks(out.pred_masks.float().cpu(), inputs["original_sizes"].cpu(),
                                              binarize=False)[0][0]           # (3,H,W) logits
        scores = out.iou_scores[0, 0].float().cpu().numpy()
        ref = coarse_mask.astype(bool)
        best, best_s = None, -1
        for k in range(logits.shape[0]):
            m = logits[k].numpy() > 0
            inter = (m & ref).sum()
            union = (m | ref).sum() + 1
            s = 0.6 * inter / union + 0.4 * float(scores[k])
            if s > best_s:
                best, best_s = k, s
        prob = 1 / (1 + np.exp(-np.clip(logits[best].numpy(), -30, 30)))
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
        r = max(2, int(round(rgb.shape[1] / 480)))
        refined = guided_filter(gray, prob.astype(np.float32), r, 1e-3) > 0.5
        # keep the component(s) overlapping the coarse prior
        n, lab = cv2.connectedComponents(refined.astype(np.uint8))
        if n > 2:
            keep = np.zeros(n, bool)
            for k in range(1, n):
                if (ref & (lab == k)).sum() > 0.02 * max(1, ref.sum()) or (lab == k).sum() > 0.2 * refined.sum():
                    keep[k] = True
            refined = keep[lab]
        return np.ascontiguousarray(np.rot90(refined, -rk)), float(best_s)


def make_prompts(f, T_wc, part_pts, look_at, coarse_radius=0.002):
    """Coarse mask + box + positive/negative points for one frame."""
    T_cw = inv_T(T_wc)
    W, H = f.rgb_size
    cm = splat_mask(part_pts, f.K, T_cw, (W, H), coarse_radius)
    cm = cv2.morphologyEx(cm, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    ys, xs = np.nonzero(cm)
    if len(xs) == 0:
        return None
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    pad = 0.08 * max(x1 - x0, y1 - y0) + 4
    box = [max(0, x0 - pad), max(0, y0 - pad), min(W - 1, x1 + pad), min(H - 1, y1 + pad)]
    dt = cv2.distanceTransform(cm, cv2.DIST_L2, 5)
    pos = []
    for _ in range(2):
        v, u = np.unravel_index(np.argmax(dt), dt.shape)
        if dt[v, u] < 3:
            break
        pos.append((u, v))
        cv2.circle(dt, (int(u), int(v)), int(max(6, dt[v, u])), 0, -1)
    # negatives: a ring on the support plane around the part
    if len(part_pts):
        c = part_pts[:, :2].mean(0)
        rad = np.percentile(np.linalg.norm(part_pts[:, :2] - c, axis=1), 98) + 0.02
    else:
        c, rad = look_at[:2], 0.08
    ang = np.linspace(0, 2 * np.pi, 8, endpoint=False)
    ring = np.stack([c[0] + rad * np.cos(ang), c[1] + rad * np.sin(ang), np.zeros(8)], 1)
    uv, z = project(f.K, T_cw, ring)
    neg = [(u, v) for (u, v), zz in zip(uv, z) if zz > 0 and 0 <= u < W and 0 <= v < H and not cm[int(v), int(u)]]
    if len(neg) > 4:   # spread the negative clicks around the part instead of clustering on one side
        neg = [neg[int(round(i))] for i in np.linspace(0, len(neg) - 1, 4)]
    return dict(coarse=cm, box=box, pos=pos, neg=neg)


In [ ]:
%%writefile /content/partscan/depth.py
"""Dense depth per keyframe + per-frame metric correction.

PromptDepthAnything (CVPR 2025) is a depth foundation model that takes the RGB image AND the
iPhone's low-resolution LiDAR map as a "prompt" and returns sharp, metric depth at up to
~1428x1904. It was trained on ARKit LiDAR data, i.e. exactly this sensor.

Even so, a single depth map can carry a few millimetres of scale/offset bias (the LiDAR
anchor). Each frame is therefore corrected with a robust affine fit against geometry that
is known more precisely from multi-view photogrammetry: the board / support plane and the
triangulated feature points.
"""
from __future__ import annotations

import logging

import cv2
import numpy as np

from .geometry import K_for_size, inv_T

log = logging.getLogger("partscan")


def fill_invalid(depth: np.ndarray) -> np.ndarray:
    """Nearest-valid fill (PromptDA min/max-normalises its prompt, so zeros must not leak in)."""
    bad = ~(depth > 0)
    if not bad.any():
        return depth
    if bad.all():
        return np.full_like(depth, 1.0)
    _, idx = cv2.distanceTransformWithLabels(bad.astype(np.uint8), cv2.DIST_L2, 5, labelType=cv2.DIST_LABEL_PIXEL)
    vals = np.zeros(idx.max() + 1, depth.dtype)
    good_idx = idx[~bad]
    vals[good_idx] = depth[~bad]
    return np.where(bad, vals[idx], depth)


class LidarUpsampler:
    """No-AI fallback: bilinear upsampling of the 256x192 LiDAR map to half the RGB resolution."""
    name = "lidar"

    def __init__(self, scale: float = 0.5):
        self.scale = scale

    def predict(self, f):
        W, H = f.rgb_size
        w, h = int(round(W * self.scale)), int(round(H * self.scale))
        d = cv2.resize(fill_invalid(f.depth), (w, h), interpolation=cv2.INTER_LINEAR)
        valid = cv2.resize((f.depth > 0).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) > 0
        if "valid" in f.extra:
            valid &= cv2.resize(f.extra["valid"].astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) > 0
        return np.where(valid, d, 0).astype(np.float32), K_for_size(f.K, (W, H), (w, h))


class PromptDADepth:
    name = "promptda"

    def __init__(self, checkpoint: str, size: int, device: str):
        import torch
        from transformers import AutoImageProcessor, AutoModelForDepthEstimation
        self.torch = torch
        self.device = device
        self.proc = AutoImageProcessor.from_pretrained(checkpoint)
        # fp32 on purpose: the model rescales its output to metres internally, and fp16 would
        # quantise depth to ~0.2 mm at 30 cm
        dtype = torch.float32
        self.dtype = dtype
        self.model = AutoModelForDepthEstimation.from_pretrained(checkpoint, torch_dtype=dtype).to(device).eval()
        self.size = int(size)

    def predict(self, f):
        torch = self.torch
        rgb = f.load_rgb()
        H, W = rgb.shape[:2]
        prompt = fill_invalid(f.depth.astype(np.float32))
        rk = int(f.extra.get("upright_k", 0))           # feed the model an upright image
        inputs = self.proc(images=np.ascontiguousarray(np.rot90(rgb, rk)),
                           prompt_depth=np.ascontiguousarray(np.rot90(prompt, rk)), prompt_scale_to_meter=1.0,
                           size={"height": self.size, "width": self.size}, return_tensors="pt")
        inputs = {kk: (v.to(self.device, self.dtype) if v.is_floating_point() else v.to(self.device))
                  for kk, v in inputs.items()}
        with torch.inference_mode():
            out = self.model(**inputs)
        d = np.ascontiguousarray(np.rot90(out.predicted_depth[0].float().cpu().numpy(), -rk))
        h, w = d.shape
        valid = np.ones((h, w), bool)
        if "valid" in f.extra:
            valid = cv2.resize(f.extra["valid"].astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) > 0
        return np.where(valid, d, 0).astype(np.float32), K_for_size(f.K, (W, H), (w, h))


# --------------------------------------------------------------------------- alignment
def plane_samples(D, Kd, T_wc, n, d0, exclude=None, region=None, step: int = 6):
    """(u, v, z_true) for pixels whose ray hits the plane n.x = d0 (optionally inside region,
    outside the exclusion mask)."""
    h, w = D.shape
    v, u = np.mgrid[step // 2:h:step, step // 2:w:step]
    u, v = u.ravel(), v.ravel()
    ok = D[v, u] > 0
    if exclude is not None:
        ok &= ~exclude[v, u]
    rays = np.stack([(u - Kd[0, 2]) / Kd[0, 0], (v - Kd[1, 2]) / Kd[1, 1], np.ones(len(u))], 1)
    R, C = T_wc[:3, :3], T_wc[:3, 3]
    rw = rays @ R.T
    den = rw @ n
    lam = (d0 - C @ n) / np.where(np.abs(den) < 1e-9, 1e-9, den)
    ok &= (lam > 0.05) & (lam < 1.2)
    if region is not None:
        P = C + rw * lam[:, None]
        xmin, xmax, ymin, ymax = region
        ok &= (P[:, 0] > xmin) & (P[:, 0] < xmax) & (P[:, 1] > ymin) & (P[:, 1] < ymax)
    return u[ok], v[ok], lam[ok]


def track_samples(i, tracks, T_wc, K_rgb, Kd, shape):
    """(u, v, z_true) in depth-map pixels from triangulated points observed in frame i."""
    if tracks is None or tracks.n_points == 0:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0)
    sel = tracks.obs_cam == i
    if not sel.any():
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0)
    X = tracks.X[tracks.obs_pt[sel]]
    T_cw = inv_T(T_wc)
    Xc = X @ T_cw[:3, :3].T + T_cw[:3, 3]
    uv = tracks.obs_uv[sel]
    # RGB pixel -> depth-map pixel (half-pixel convention)
    sx, sy = Kd[0, 0] / K_rgb[0, 0], Kd[1, 1] / K_rgb[1, 1]
    u = np.round((uv[:, 0] + 0.5) * sx - 0.5).astype(int)
    v = np.round((uv[:, 1] + 0.5) * sy - 0.5).astype(int)
    h, w = shape
    ok = (u >= 0) & (u < w) & (v >= 0) & (v < h) & (Xc[:, 2] > 0.02)
    return u[ok], v[ok], Xc[ok, 2]


def robust_affine(x, y, w=None, delta: float = 0.001, iters: int = 8, scale_sigma: float = 0.02,
                  relief=None, relief_sigma: float = 0.05):
    """Huber IRLS for  y ~ a*x + b (+ c*relief)  with weak priors a ~ 1 +- scale_sigma and c ~ 0 +- relief_sigma.
    The scale prior keeps the fit well posed when all samples sit at nearly the same depth (looking straight
    down at the board). 'relief' = predicted depth minus the support-plane depth along the same ray: the c
    term corrects a depth model that flattens (or exaggerates) the part relative to the table.
    Returns (a, b, c, robust residual sigma)."""
    w = np.ones_like(x) if w is None else w
    r_ = np.zeros_like(x) if relief is None else relief
    use_c = relief is not None
    a, b, c = 1.0, float(np.median(y - x)), 0.0
    sig = delta
    for _ in range(iters):
        r = a * x + b + c * r_ - y
        wr = w * np.where(np.abs(r) <= delta, 1.0, delta / np.maximum(np.abs(r), 1e-12))
        cols = [x, np.ones_like(x)] + ([r_] if use_c else [])
        A = np.stack(cols, 1) * np.sqrt(wr)[:, None] / sig
        rhs = y * np.sqrt(wr) / sig
        pri = [[1.0 / scale_sigma, 0.0] + ([0.0] if use_c else [])]
        prhs = [1.0 / scale_sigma]
        if use_c:
            pri.append([0.0, 0.0, 1.0 / relief_sigma])
            prhs.append(0.0)
        A = np.vstack([A, pri])
        rhs = np.r_[rhs, prhs]
        sol = np.linalg.lstsq(A, rhs, rcond=None)[0]
        a, b = sol[0], sol[1]
        c = sol[2] if use_c else 0.0
    r = a * x + b + c * r_ - y
    return float(a), float(b), float(c), float(1.4826 * np.median(np.abs(r)))


def plane_depth_map(shape, Kd, T_wc, n, d0, max_depth: float = 2.0):
    """Depth at which every pixel's ray meets the support plane n.x = d0 (0 where it does not)."""
    h, w = shape
    v, u = np.mgrid[0:h, 0:w]
    rays = np.stack([(u - Kd[0, 2]) / Kd[0, 0], (v - Kd[1, 2]) / Kd[1, 1], np.ones((h, w))], -1)
    R, C = T_wc[:3, :3], T_wc[:3, 3]
    den = rays @ (R.T @ n)
    lam = (d0 - C @ n) / np.where(np.abs(den) < 1e-9, 1e-9, den)
    return np.where((lam > 0.05) & (lam < max_depth), lam, 0.0).astype(np.float32)


def align_depth(D, Kd, T_wc, K_rgb, i, tracks=None, plane=None, part_mask=None, plane_region=None,
                part_weight: float = 4.0):
    """Correct one dense depth map so it agrees with the photogrammetric geometry."""
    us, vs, zs, ws = [], [], [], []
    if plane is not None:
        excl = None
        if part_mask is not None:
            pm = cv2.resize(part_mask.astype(np.uint8), (D.shape[1], D.shape[0]), interpolation=cv2.INTER_NEAREST)
            excl = cv2.dilate(pm, np.ones((15, 15), np.uint8)) > 0
        u, v, z = plane_samples(D, Kd, T_wc, plane[0], plane[1], excl, plane_region)
        us.append(u), vs.append(v), zs.append(z), ws.append(np.ones(len(z)))
    u, v, z = track_samples(i, tracks, T_wc, K_rgb, Kd, D.shape)
    if len(z):
        near = np.ones(len(z), bool)
        if part_mask is not None:
            pm = cv2.resize(part_mask.astype(np.uint8), (D.shape[1], D.shape[0]), interpolation=cv2.INTER_NEAREST)
            near = cv2.dilate(pm, np.ones((9, 9), np.uint8))[v, u] > 0
        us.append(u), vs.append(v), zs.append(z), ws.append(np.where(near, part_weight, 1.0))
    if not us:
        return D, dict(a=1.0, b=0.0, n=0, sigma_mm=float("nan"), ok=False)
    u, v, z, w = (np.concatenate(x) for x in (us, vs, zs, ws))
    d = D[v, u]
    ok = d > 0
    u, v, z, w, d = u[ok], v[ok], z[ok], w[ok], d[ok]
    if len(z) < 25:
        return D, dict(a=1.0, b=0.0, c=0.0, n=int(len(z)), sigma_mm=float("nan"), ok=False)
    # relief term: only ON the part (inside its outline). Off the part the true depth IS the plane depth,
    # so a relief computed from the prediction there would just echo the prediction's own bias.
    Dp, on_part, relief = None, None, None
    if plane is not None and part_mask is not None:
        Dp = plane_depth_map(D.shape, Kd, T_wc, *plane)
        on_part = cv2.resize(part_mask.astype(np.uint8), (D.shape[1], D.shape[0]),
                             interpolation=cv2.INTER_NEAREST) > 0
        on_part &= Dp > 0
        sel = on_part[v, u]
        relief = np.where(sel, d - Dp[v, u], 0.0)
        if sel.sum() < 15:          # too few points on the part to say anything about its relief
            relief, on_part = None, None
    a, b, c, sig = robust_affine(d, z, w, relief=relief)
    good = 0.85 < a < 1.15 and abs(b) < 0.03 and abs(c) < 0.2
    if not good:
        log.warning("frame %d: depth correction rejected (a=%.3f, b=%.1f mm, c=%.3f)", i, a, b * 1000, c)
        return D, dict(a=a, b=b, c=c, n=int(len(z)), sigma_mm=sig * 1000, ok=False)
    Dc = a * D + b
    if on_part is not None and c != 0.0:
        Dc = Dc + c * np.where(on_part, D - Dp, 0.0)
    Dc = np.where(D > 0, Dc, 0).astype(np.float32)
    return Dc, dict(a=a, b=b, c=c, n=int(len(z)), sigma_mm=sig * 1000, ok=True)


In [ ]:
%%writefile /content/partscan/fusion.py
"""Volumetric fusion: TSDF of the corrected depth maps intersected with a robust visual hull.

    solid = TSDF-solid  AND  visual-hull-solid  AND  {z >= 0 (support plane)}
    sdf   = max(sdf_tsdf, sdf_hull, -z)

* TSDF: part and background pixels are integrated (the background proves free space); mixed
  pixels at depth edges only contribute free space in front of the near surface. Voxels with too
  little evidence (the inside of the part) count as solid.
* Visual hull: for every view, the distance of a voxel outside the silhouette is
  (pixels outside the mask - tolerance) * depth / focal. The hull SDF is the k-th largest of
  those values, so a single bad mask can never cut into the part.
* Support plane: the underside the camera never saw is closed exactly at z = 0.
"""
from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Optional

import numpy as np

log = logging.getLogger("partscan")


@dataclass
class FusionFrame:
    T_cw: np.ndarray                 # 4x4 world(part frame)->camera
    depth: Optional[np.ndarray]      # (h,w) metres, 0 = invalid
    Kd: Optional[np.ndarray]         # intrinsics of the depth map
    depth_mask: Optional[np.ndarray]  # (h,w) bool: pixels allowed into the TSDF
    sdt: Optional[np.ndarray] = None  # (H,W) float32 signed distance to silhouette [px], >0 outside
    K: Optional[np.ndarray] = None    # intrinsics of the silhouette image
    weight: float = 1.0
    weight_map: Optional[np.ndarray] = None  # (h,w) per-pixel confidence for the TSDF
    free_depth: Optional[np.ndarray] = None  # (h,w) for excluded (edge) pixels: space closer than this is empty


def auto_voxel(roi_min, roi_max, voxel_mm: float = 0.0) -> float:
    if voxel_mm and voxel_mm > 0:
        return voxel_mm / 1000.0
    ext = float(np.max(np.asarray(roi_max) - np.asarray(roi_min)))
    return float(np.clip(ext / 320.0, 0.0002, 0.001))


def fuse(frames, roi_min, roi_max, voxel: float, trunc: float, k_carve: int = 3, tol_px: float = 1.0,
         close_bottom: bool = True, device: str = "cpu", progress=None, chunk: int | None = None,
         min_weight: float = 0.8, min_views: int = 2, trunc_back: float | None = None):
    """trunc: band in front of a surface; trunc_back: (shorter) band behind it, so that views of the far
    side of a thin wall do not thicken it. min_views / min_weight: evidence needed before a voxel counts
    as observed - one wrong depth map can then never hollow out the part."""
    if trunc_back is None:
        trunc_back = trunc
    import torch

    roi_min = np.asarray(roi_min, float)
    roi_max = np.asarray(roi_max, float)
    dims = np.maximum(np.ceil((roi_max - roi_min) / voxel).astype(int), 2)
    N = int(np.prod(dims))
    dx, dy, dz = (int(v) for v in dims)
    if chunk is None:
        chunk = 6_000_000 if device.startswith("cuda") else 1_500_000
    dev = torch.device(device)
    f32 = torch.float32
    # move frames to the device once; torch.from_numpy shares memory (no CPU copy), and frames may be a
    # generator so the caller never needs to hold every frame's buffers at the same time
    def T(a, dtype=None):
        t = torch.from_numpy(np.ascontiguousarray(a if dtype is None else np.asarray(a, dtype)))
        return t.to(dev) if dev.type != "cpu" else t

    up = []
    for fr in frames:
        e = {"R": torch.tensor(fr.T_cw[:3, :3], dtype=f32, device=dev),
             "t": torch.tensor(fr.T_cw[:3, 3], dtype=f32, device=dev), "w": fr.weight}
        if fr.depth is not None:
            h, w = fr.depth.shape
            e.update(D=T(fr.depth, np.float32).reshape(-1), M=T(fr.depth_mask, np.bool_).reshape(-1),
                     Wm=T(fr.weight_map, np.float16).reshape(-1) if fr.weight_map is not None else None,
                     Fd=T(fr.free_depth, np.float32).reshape(-1) if fr.free_depth is not None else None,
                     Kd=[float(fr.Kd[0, 0]), float(fr.Kd[1, 1]), float(fr.Kd[0, 2]), float(fr.Kd[1, 2])], hw=(h, w))
        if fr.sdt is not None:
            H, W = fr.sdt.shape
            e.update(S=T(fr.sdt, np.float16)[None, None],
                     K=[float(fr.K[0, 0]), float(fr.K[1, 1]), float(fr.K[0, 2]), float(fr.K[1, 2])], HW=(H, W))
        up.append(e)
    n_sil = sum("S" in e for e in up)
    k_eff = int(min(k_carve, max(1, n_sil)))
    out = np.empty(N, np.float32)
    seen = np.zeros(N, np.uint8)
    o = torch.tensor(roi_min, dtype=f32, device=dev)
    neg_inf = torch.tensor(float("-inf"), device=dev)
    for c0 in range(0, N, chunk):
        idx = torch.arange(c0, min(N, c0 + chunk), device=dev)
        iz = idx % dz
        iy = (idx // dz) % dy
        ix = idx // (dy * dz)
        P = o + (torch.stack([ix, iy, iz], 1).to(f32) + 0.5) * voxel
        n = len(idx)
        tsdf = torch.ones(n, dtype=f32, device=dev)
        wsum = torch.zeros(n, dtype=f32, device=dev)
        nobs = torch.zeros(n, dtype=torch.int16, device=dev)
        vh = torch.full((k_eff, n), float("-inf"), dtype=f32, device=dev) if n_sil else None
        for e in up:
            Xc = P @ e["R"].T + e["t"]
            z = Xc[:, 2]
            front = z > 0.03
            zs = torch.where(front, z, torch.ones_like(z))
            xn, yn = Xc[:, 0] / zs, Xc[:, 1] / zs
            if "D" in e:
                fx, fy, cx, cy = e["Kd"]
                h, w = e["hw"]
                u = torch.round(fx * xn + cx).long()
                v = torch.round(fy * yn + cy).long()
                ins = front & (u >= 0) & (u < w) & (v >= 0) & (v < h)
                lin = v.clamp(0, h - 1) * w + u.clamp(0, w - 1)
                d = e["D"][lin]
                ok = ins & e["M"][lin] & (d > 0)
                sdf = (d - z) * torch.sqrt(1 + xn * xn + yn * yn)
                upd = ok & (sdf > -trunc_back)
                obs = torch.clamp(sdf / trunc, -1.0, 1.0)
                wo = upd.to(f32) * e["w"]
                if e["Wm"] is not None:
                    wo = wo * e["Wm"][lin].to(f32)
                if e["Fd"] is not None:
                    # pixels excluded at depth edges: the NEAREST neighbouring depth is certainly a
                    # surface or in front of one, so space in front of it is observed as outside
                    # (never as inside - behind it could be either side of the edge)
                    fd = e["Fd"][lin]
                    sdf_f = (fd - z) * torch.sqrt(1 + xn * xn + yn * yn)
                    fupd = ins & ~e["M"][lin] & (fd > 0) & (sdf_f > 0)
                    obs = torch.where(fupd, torch.clamp(sdf_f / trunc, 0.0, 1.0), obs)
                    wo = torch.where(fupd, torch.full_like(wo, 0.5 * e["w"]), wo)
                    upd = upd | fupd
                tsdf = torch.where(upd, (tsdf * wsum + obs * wo) / (wsum + wo).clamp_min(1e-6), tsdf)
                wsum = wsum + wo
                nobs = nobs + (upd & (wo > 0)).to(torch.int16)
            if "S" in e:
                fx, fy, cx, cy = e["K"]
                H, W = e["HW"]
                u2 = fx * xn + cx
                v2 = fy * yn + cy
                ins2 = front & (u2 >= 0) & (u2 <= W - 1) & (v2 >= 0) & (v2 <= H - 1)
                grid = torch.stack([u2 / (W - 1) * 2 - 1, v2 / (H - 1) * 2 - 1], 1)[None, None]
                s = torch.nn.functional.grid_sample(e["S"].to(f32), grid, mode="bilinear", padding_mode="border",
                                                    align_corners=True)[0, 0, 0]
                val = torch.where(ins2, (s - tol_px) * z / fx, neg_inf)
                vh = torch.topk(torch.cat([vh, val[None]], 0), k_eff, dim=0).values
        # a voxel counts as observed only with ~one frontal view's worth of evidence: a single biased
        # frame can then never hollow out the inside of the part
        observed = (wsum >= min_weight) & (nobs >= min_views)
        sdf_t = torch.where(observed, tsdf * trunc, torch.full_like(tsdf, -trunc))
        sdf = sdf_t
        if vh is not None:
            sdf = torch.maximum(sdf, vh[k_eff - 1])
        if close_bottom:
            sdf = torch.maximum(sdf, -P[:, 2])
        out[c0:c0 + n] = torch.clamp(sdf, -trunc, trunc).cpu().numpy()
        seen[c0:c0 + n] = observed.cpu().numpy()
        if progress is not None:
            progress(min(1.0, (c0 + n) / N), f"fusing volume {100 * (c0 + n) // N}%")
    vol = out.reshape(dx, dy, dz)
    n_void = fill_enclosed_voids(vol, trunc)
    stats = dict(dims=[dx, dy, dz], voxel_mm=voxel * 1000, observed_fraction=float(seen.mean()),
                 silhouette_views=n_sil, carve_k=k_eff, voids_filled=int(n_void))
    return vol, stats


def fill_enclosed_voids(vol: np.ndarray, trunc: float) -> int:
    """Empty pockets that do not connect to the outside of the volume cannot be seen by any camera:
    they are noise, so make them solid (in place). Holes, slots and pockets stay open because they
    connect to the outside. Returns the number of voxels filled."""
    from scipy import ndimage
    lab, n = ndimage.label(vol > 0)
    if n <= 1:
        return 0
    border = np.unique(np.concatenate([lab[0].ravel(), lab[-1].ravel(), lab[:, 0].ravel(), lab[:, -1].ravel(),
                                       lab[:, :, 0].ravel(), lab[:, :, -1].ravel()]))
    enclosed = np.ones(n + 1, bool)
    enclosed[0] = False
    enclosed[border] = False
    fill = enclosed[lab]
    vol[fill] = -trunc
    return int(fill.sum())


def extract_mesh(vol: np.ndarray, roi_min, voxel: float):
    """Marching cubes at sdf = 0 -> (vertices [m, part frame], faces) with outward normals."""
    from skimage.measure import marching_cubes
    import trimesh

    if not (vol.min() < 0 < vol.max()):
        raise RuntimeError("The fused volume contains no surface - the part was not reconstructed. "
                           "Check that the part is visible in the scan and the board/support plane was found.")
    v, f, _, _ = marching_cubes(vol, level=0.0, spacing=(voxel, voxel, voxel), allow_degenerate=False)
    v = v + np.asarray(roi_min) + 0.5 * voxel
    m = trimesh.Trimesh(v, f, process=True)
    if m.volume < 0:
        m.invert()
    return m


def sample_sdf(vol, roi_min, voxel, pts):
    """Trilinear SDF lookup at world points (NaN outside the grid)."""
    from scipy.ndimage import map_coordinates
    g = (np.asarray(pts) - np.asarray(roi_min)) / voxel - 0.5
    val = map_coordinates(vol, g.T, order=1, mode="constant", cval=np.nan)
    return val


In [ ]:
%%writefile /content/partscan/meshproc.py
"""Mesh clean-up, alignment to a canonical part frame, and vertex colouring."""
from __future__ import annotations

import logging

import numpy as np
import trimesh

from .geometry import inv_T, min_area_rect

log = logging.getLogger("partscan")


def clean_mesh(m: trimesh.Trimesh, smooth_iterations: int = 8, max_faces: int = 400_000) -> trimesh.Trimesh:
    parts = m.split(only_watertight=False)
    if len(parts) > 1:
        sizes = np.array([len(p.faces) for p in parts])
        keep = [p for p, s in zip(parts, sizes) if s >= 0.05 * sizes.max()]
        m = trimesh.util.concatenate(keep)
        log.info("removed %d small floating pieces", len(parts) - len(keep))
    if smooth_iterations:
        trimesh.smoothing.filter_taubin(m, lamb=0.5, nu=0.53, iterations=int(smooth_iterations))
    if len(m.faces) > max_faces:
        try:
            m = m.simplify_quadric_decimation(face_count=int(max_faces))
        except Exception as ex:  # fast_simplification missing
            log.warning("decimation skipped: %s", ex)
    m.remove_unreferenced_vertices()
    m.fix_normals()
    return m


def canonical_frame(m: trimesh.Trimesh):
    """Rotate about the support-plane normal (z) so the footprint's minimum-area rectangle is
    axis aligned (length along X), and centre it. Returns (T_cp: part frame -> canonical, rect)."""
    rect = min_area_rect(m.vertices[:, :2])
    a = rect["angle"]
    c, s = np.cos(-a), np.sin(-a)
    T = np.eye(4)
    T[:3, :3] = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    ctr = np.r_[rect["center"], 0.0]
    T[:3, 3] = -T[:3, :3] @ ctr
    return T, rect


def colorize(m: trimesh.Trimesh, views, max_views: int = 60):
    """Average colour of each vertex over views where it is visible.
    views: list of (rgb (H,W,3), K, T_cw, depth (h,w) or None, Kd)."""
    V = m.vertices
    N = m.vertex_normals
    acc = np.zeros((len(V), 3))
    wsum = np.zeros(len(V))
    step = max(1, len(views) // max_views)
    for rgb, K, T_cw, D, Kd in views[::step]:
        H, W = rgb.shape[:2]
        Xc = V @ T_cw[:3, :3].T + T_cw[:3, 3]
        z = Xc[:, 2]
        ok = z > 0.03
        u = K[0, 0] * Xc[:, 0] / np.where(ok, z, 1) + K[0, 2]
        v = K[1, 1] * Xc[:, 1] / np.where(ok, z, 1) + K[1, 2]
        ok &= (u >= 0) & (u <= W - 1) & (v >= 0) & (v <= H - 1)
        cam_c = inv_T(T_cw)[:3, 3]
        view = V - cam_c
        view /= np.linalg.norm(view, axis=1, keepdims=True) + 1e-12
        cosang = -(N * view).sum(1)
        ok &= cosang > 0.25
        if D is not None:
            ud = np.round(Kd[0, 0] * Xc[:, 0] / np.where(ok, z, 1) + Kd[0, 2]).astype(int)
            vdp = np.round(Kd[1, 1] * Xc[:, 1] / np.where(ok, z, 1) + Kd[1, 2]).astype(int)
            h, w = D.shape
            inside = ok & (ud >= 0) & (ud < w) & (vdp >= 0) & (vdp < h)
            dd = np.zeros(len(V))
            dd[inside] = D[vdp[inside], ud[inside]]
            ok &= inside & (np.abs(dd - z) < 0.003) & (dd > 0)
        idx = np.nonzero(ok)[0]
        if not len(idx):
            continue
        ui = np.clip(np.round(u[idx]).astype(int), 0, W - 1)
        vi = np.clip(np.round(v[idx]).astype(int), 0, H - 1)
        col = rgb[vi, ui].astype(np.float64)
        wt = cosang[idx] ** 2
        acc[idx] += col * wt[:, None]
        wsum[idx] += wt
    out = np.full((len(V), 3), 170.0)
    has = wsum > 0
    out[has] = acc[has] / wsum[has, None]
    rgba = np.concatenate([np.clip(out, 0, 255), np.full((len(V), 1), 255)], 1).astype(np.uint8)
    m.visual = trimesh.visual.ColorVisuals(m, vertex_colors=rgba)
    return float(has.mean())


In [ ]:
%%writefile /content/partscan/measure.py
"""Measurements on the final mesh (canonical frame, millimetres).

* overall size as placed: length (X) x width (Y) x height (Z) of the footprint rectangle
* minimum-volume oriented bounding box (any orientation)
* volume, surface area, mass for a chosen material, centre of mass
* holes / round bosses: circles fitted to cross-sections along X, Y and Z, merged across levels
* planar faces: wall thicknesses (opposite faces) and step heights (same-facing faces)
"""
from __future__ import annotations

import logging

import numpy as np
import trimesh

from .config import MATERIAL_DENSITY_G_CM3
from .geometry import fit_circle_2d, fit_plane

log = logging.getLogger("partscan")
AXES = {"X": np.array([1.0, 0, 0]), "Y": np.array([0, 1.0, 0]), "Z": np.array([0, 0, 1.0])}


def overall(m: trimesh.Trimesh) -> dict:
    lo, hi = m.bounds
    ext = hi - lo
    out = dict(length_x=float(ext[0]), width_y=float(ext[1]), height_z=float(hi[2]),
               bbox_min=lo.tolist(), bbox_max=hi.tolist())
    try:
        T, e = trimesh.bounds.oriented_bounds(m)
        e = np.sort(np.asarray(e))[::-1]
        out["obb_extents"] = e.tolist()
    except Exception:
        out["obb_extents"] = sorted(ext.tolist(), reverse=True)
    return out


def mass_props(m: trimesh.Trimesh, material: str, sdf_volume_mm3=None) -> dict:
    wt = bool(m.is_watertight)
    vol = float(abs(m.volume)) if wt else float(sdf_volume_mm3 if sdf_volume_mm3 is not None else "nan")
    dens = MATERIAL_DENSITY_G_CM3.get(material, 0.0)
    ok = vol == vol
    return dict(volume_mm3=vol, area_mm2=float(m.area), watertight=wt,
                mass_g=(vol / 1000.0 * dens) if (ok and dens) else float("nan"),
                material=material, density_g_cm3=dens,
                center_of_mass=(m.center_mass.tolist() if wt else m.centroid.tolist()))


# --------------------------------------------------------------------------- circles
def _section_polys(m, origin, normal):
    try:
        sec = m.section(plane_origin=origin, plane_normal=normal)
    except Exception:
        return [], None
    if sec is None:
        return [], None
    try:
        p2, T = sec.to_2D()
        return list(p2.polygons_full), T
    except Exception:
        return [], None


def _ring_harmonics(pts, cx, cy, r, nb=72):
    """Shape of a ring's radial deviation: (largest share of it in ONE harmonic 3..8, 2nd harmonic / r).
    A polygon (square, hexagon...) concentrates its deviation in one harmonic; a rough scanned bore doesn't."""
    th = np.arctan2(pts[:, 1] - cy, pts[:, 0] - cx)
    e = np.hypot(pts[:, 0] - cx, pts[:, 1] - cy) - r
    b = ((th + np.pi) / (2 * np.pi) * nb).astype(int) % nb
    s, c = np.bincount(b, e, nb), np.bincount(b, None, nb)
    ok = c > 0
    prof = np.interp(np.arange(nb), np.flatnonzero(ok), s[ok] / c[ok], period=nb)
    A = np.abs(np.fft.rfft(prof - prof.mean())) / nb * 2
    var = max((A[1:] ** 2).sum() / 2, 1e-12)
    return float((A[3:9] ** 2 / 2).max() / var), float(A[2] / r)


def _circle_from_ring(coords, min_r=0.4, allow_rough=False):
    """(cx, cy, r, rms, rough) for a closed, round ring, else None. 'rough' = round opening whose
    outline is too ragged for a precise diameter (e.g. a side hole seen only at an angle)."""
    pts = np.asarray(coords)[:-1]
    if len(pts) < 10:
        return None
    cx, cy, r, rms = fit_circle_2d(pts)
    if r < min_r:
        return None
    ang = np.sort(np.arctan2(pts[:, 1] - cy, pts[:, 0] - cx))
    gap = np.max(np.diff(np.r_[ang, ang[0] + 2 * np.pi]))
    # scanned surfaces are noisy: accept up to 0.12 mm or 4% of r RMS deviation, no big gaps,
    # and the ring must close (area of the ring ~ area of the circle)
    from shapely.geometry import Polygon
    try:
        area_ratio = abs(Polygon(pts).area) / (np.pi * r * r)
    except Exception:
        area_ratio = 0
    if gap > np.radians(45) or not (0.9 < area_ratio < 1.1):
        return None
    if rms <= max(0.12, 0.04 * r):
        return cx, cy, r, rms, False
    if allow_rough and rms <= min(0.4, 0.13 * r):
        poly, ell = _ring_harmonics(pts, cx, cy, r)
        if poly < 0.6 and ell < 0.2:          # not a polygon (hex socket, square pocket) nor a long slot
            return cx, cy, r, rms, True
    return None


def find_circles(m: trimesh.Trimesh, step_mm: float = 0.6, max_levels: int = 260, work_faces: int = 150000) -> list:
    """Holes (interior rings) and round outer features on sections along each axis."""
    if len(m.faces) > work_faces * 1.3:
        try:
            m = m.simplify_quadric_decimation(face_count=work_faces)
        except Exception:
            pass
    found = []
    lo, hi = m.bounds
    for name, ax in AXES.items():
        k = "XYZ".index(name)
        span = hi[k] - lo[k]
        if span < 1.0:
            continue
        n_levels = int(np.clip(span / step_mm, 6, max_levels))
        levels = np.linspace(lo[k] + 0.02 * span, hi[k] - 0.02 * span, n_levels)
        for lev in levels:
            origin = np.zeros(3)
            origin[k] = lev
            polys, T = _section_polys(m, origin, ax)
            for poly in polys:
                rings = [("hole", r) for r in poly.interiors] + [("outer", poly.exterior)]
                for kind, ring in rings:
                    c = _circle_from_ring(ring.coords, allow_rough=(kind == "hole"))
                    if c is None:
                        continue
                    cx, cy, r, rms, rough = c
                    p3 = (T @ np.array([cx, cy, 0, 1.0]))[:3]
                    found.append(dict(kind=kind, axis=name, level=float(lev), center=p3, d=2 * r, rms=rms,
                                      rough=rough))
    # merge detections of the same feature across levels
    feats = []
    for det in found:
        k = "XYZ".index(det["axis"])
        perp = [i for i in range(3) if i != k]
        for fe in feats:
            if fe["kind"] == det["kind"] and fe["axis"] == det["axis"] and \
                    np.linalg.norm(fe["center"][perp] - det["center"][perp]) < max(0.6, 0.12 * det["d"]) and \
                    abs(fe["d"] - det["d"]) < max(0.6, 0.15 * det["d"]):
                fe["ds"].append(det["d"])
                fe["levels"].append(det["level"])
                fe["centers"].append(det["center"])
                fe["rough"].append(det["rough"])
                fe["center"] = np.mean(fe["centers"], 0)
                fe["d"] = float(np.median(fe["ds"]))
                break
        else:
            feats.append(dict(kind=det["kind"], axis=det["axis"], center=det["center"].copy(), d=det["d"],
                              ds=[det["d"]], levels=[det["level"]], centers=[det["center"]], rough=[det["rough"]]))
    out = []
    for fe in feats:
        if len(fe["ds"]) < 3:
            continue  # seen on too few slices: likely a coincidence
        k = "XYZ".index(fe["axis"])
        rough = np.array(fe["rough"])
        lv_all = np.array(fe["levels"])
        # a clean bore: measure it on its clean slices only. Otherwise report it, flagged as rough
        use = ~rough if (~rough).sum() >= 3 else np.ones(len(rough), bool)
        is_rough = bool(rough[use].any())
        lv = lv_all[use]
        ds = np.array(fe["ds"])[use][np.argsort(lv)]
        lv = np.sort(lv)
        step = (hi[k] - lo[k]) * 0.96 / max(1, int(np.clip((hi[k] - lo[k]) / step_mm, 6, max_levels)) - 1)
        # rims are rounded by the scan; the middle of the bore is the reliable part
        core = ds[len(ds) // 5: len(ds) - len(ds) // 5] if len(ds) >= 5 else ds
        out.append(dict(kind=fe["kind"], axis=fe["axis"], diameter=float(np.median(core)),
                        diameter_spread=float(np.ptp(ds)), center=[float(c) for c in fe["center"]],
                        span=[float(lv_all.min()), float(lv_all.max())],
                        length=float(lv_all.max() - lv_all.min() + step), n_slices=len(ds), rough=is_rough))
    out.sort(key=lambda d: (d["kind"] != "hole", d.get("rough", False), -d["diameter"]))
    return out


# --------------------------------------------------------------------------- planes
def _sphere_grid(K=8000):
    i = np.arange(K) + 0.5
    phi = np.arccos(1 - 2 * i / K)
    th = np.pi * (1 + 5 ** 0.5) * i
    return np.stack([np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)], 1)


def find_planes(m: trimesh.Trimesh, angle_deg: float = 8.0, dist_mm: float = 0.4, min_area_frac: float = 0.006,
                max_planes: int = 30, work_faces: int = 25000, smooth_mm: float = 2.0) -> list:
    """Planar faces of a (noisy, scanned) mesh.
    1. decimate (averages out triangle-level noise), 2. dominant normal directions on a sphere
    grid, 3. peaks of the offset histogram per direction, 4. refit each plane on the full mesh."""
    from scipy.ndimage import gaussian_filter1d
    from scipy.sparse import csr_matrix
    from scipy.spatial import cKDTree

    ms = m
    if len(m.faces) > work_faces * 1.5:
        try:
            ms = m.simplify_quadric_decimation(face_count=work_faces)
        except Exception:
            ms = m
    n0, a, c = ms.face_normals, ms.area_faces, ms.triangles_center
    total = a.sum()
    # scanned faces are bumpy at the ~1 mm scale: average normals over a 2 mm neighbourhood
    tc = cKDTree(c)
    prs = tc.query_pairs(r=smooth_mm, output_type="ndarray")
    Adj = csr_matrix((np.ones(2 * len(prs)), (np.r_[prs[:, 0], prs[:, 1]], np.r_[prs[:, 1], prs[:, 0]])),
                     shape=(len(c), len(c)))
    acc = n0 * a[:, None]
    acc = acc + Adj @ acc
    n = acc / np.maximum(np.linalg.norm(acc, axis=1, keepdims=True), 1e-12)
    G = _sphere_grid()
    tree = cKDTree(G)
    _, idx = tree.query(n)
    hist = np.bincount(idx, a, minlength=len(G))
    pairs = tree.query_pairs(r=2 * np.sin(np.radians(5) / 2), output_type="ndarray")
    A = csr_matrix((np.ones(2 * len(pairs)), (np.r_[pairs[:, 0], pairs[:, 1]], np.r_[pairs[:, 1], pairs[:, 0]])),
                   shape=(len(G), len(G)))
    sm = hist + A @ hist
    cos_t = np.cos(np.radians(angle_deg))
    dirs = []
    for j in np.argsort(-sm):
        if sm[j] < min_area_frac * total:
            break
        if any(G[j] @ d > np.cos(np.radians(1.5 * angle_deg)) for d in dirs):
            continue
        sel = n @ G[j] > cos_t
        d = (n[sel] * a[sel, None]).sum(0)
        dirs.append(d / np.linalg.norm(d))
    # full-resolution data for the refit
    Vf, Nf = m.vertices, m.vertex_normals
    planes = []
    used = np.zeros(len(n), bool)
    for d in dirs:
        sel = (n @ d > cos_t) & ~used
        if a[sel].sum() < min_area_frac * total:
            continue
        off = c[sel] @ d
        bw = 0.05
        lo = off.min() - 1
        nb_ = int((off.max() + 1 - lo) / bw) + 1
        hs = gaussian_filter1d(np.bincount(((off - lo) / bw).astype(int), a[sel], minlength=nb_), 2.0)
        cand = sorted((k for k in range(1, nb_ - 1) if hs[k] >= hs[k - 1] and hs[k] >= hs[k + 1]), key=lambda k: -hs[k])
        for k in cand:
            o0 = lo + (k + 0.5) * bw
            mem = sel & ~used & (np.abs(c @ d - o0) < dist_mm)
            if a[mem].sum() < min_area_frac * total:
                continue
            nn, dd = fit_plane(c[mem], a[mem])
            if nn @ d < 0:
                nn, dd = -nn, -dd
            mem = sel & ~used & (np.abs(c @ nn - dd) < dist_mm)
            if a[mem].sum() < min_area_frac * total:
                continue
            used |= mem
            # refit on full-resolution vertices close to the plane with roughly matching normals
            vm = np.zeros(len(Vf), bool)
            for _ in range(3):
                vm = (np.abs(Vf @ nn - dd) < 0.6 * dist_mm) & (Nf @ nn > np.cos(np.radians(12)))
                if vm.sum() <= 30:
                    break
                nn2, dd2 = fit_plane(Vf[vm])
                if nn2 @ nn < 0:
                    nn2, dd2 = -nn2, -dd2
                if nn2 @ nn < 0.999:
                    break
                nn, dd = nn2, dd2
            if vm.sum() > 30:
                res = Vf[vm] @ nn - dd
            else:
                res = c[mem] @ nn - dd
            planes.append(dict(normal=nn, offset=float(dd), area=float(a[mem].sum()),
                               flatness_rms=float(np.sqrt(np.mean(res ** 2))), pts=c[mem]))
            if len(planes) >= max_planes:
                break
    planes.sort(key=lambda p: -p["area"])
    return planes


def _plane_separation(p, q) -> float:
    """Distance between two (nearly) parallel planes, measured where the faces actually are.
    Faces of similar size: mean distance of each face's points to the other plane (robust to small tilts).
    Otherwise the larger face gives the better-defined plane (a narrow face's fitted tilt is uncertain, and a
    tilt error grows with distance): mean distance of the smaller face's points to the larger face's plane."""
    if max(p["area"], q["area"]) >= 1.5 * min(p["area"], q["area"]):
        big, small = (p, q) if p["area"] >= q["area"] else (q, p)
        return float(np.abs(small["pts"] @ big["normal"] - big["offset"]).mean())
    dpq = np.abs(p["pts"] @ q["normal"] - q["offset"]).mean()
    dqp = np.abs(q["pts"] @ p["normal"] - p["offset"]).mean()
    return float(0.5 * (dpq + dqp))


def plane_relations(planes: list, max_items: int = 12) -> list:
    rel = []
    for i in range(len(planes)):
        for j in range(i + 1, len(planes)):
            p, q = planes[i], planes[j]
            cd = float(p["normal"] @ q["normal"])
            if cd < -0.995:      # opposite faces -> thickness / width
                kind = "thickness"
            elif cd > 0.995:     # same facing -> step
                kind = "step"
            else:
                continue
            dist = _plane_separation(p, q)
            if dist < 0.3:
                continue
            # the two faces must overlap when projected along the normal (otherwise unrelated)
            nrm = p["normal"]
            basis = np.linalg.svd(np.eye(3) - np.outer(nrm, nrm))[0][:, :2]
            pa, qa = p["pts"] @ basis, q["pts"] @ basis
            ov = np.all(np.minimum(pa.max(0), qa.max(0)) - np.maximum(pa.min(0), qa.min(0)) > -1.0)
            if kind == "thickness" and not ov:
                continue
            rel.append(dict(kind=kind, distance=dist, planes=(i, j), weight=min(p["area"], q["area"]),
                            normal=[float(v) for v in nrm]))
    rel.sort(key=lambda r: -r["weight"])
    return rel[:max_items]


def face_dims(planes: list, ov: dict, total_area: float, mesh=None) -> dict:
    """Length/width/height between opposite planar faces (noise-free alternative to the envelope).
    Only reported for a pair of large opposite faces that are the part's outermost ones: spanning (almost)
    the whole envelope, or - when thin noise spikes inflate the envelope - with (almost) nothing of the part
    lying beyond either face."""
    out = {}
    ext = {"X": ov["length_x"], "Y": ov["width_y"], "Z": ov["height_z"]}
    if mesh is not None:
        tc, ta = mesh.triangles_center, mesh.area_faces
    for k, ax in enumerate("XYZ"):
        e = np.eye(3)[k]
        pos = [p for p in planes if p["normal"] @ e > 0.998 and p["area"] > 0.004 * total_area]
        neg = [p for p in planes if p["normal"] @ e < -0.998 and p["area"] > 0.004 * total_area]
        best = None
        for p in pos:
            for q in neg:
                d = _plane_separation(p, q)
                if d > ext[ax] + 0.3:
                    continue
                ok = d >= ext[ax] - 0.8
                if not ok and mesh is not None and d >= ext[ax] - max(3.0, 0.02 * ext[ax]):
                    beyond = ta[(tc @ p["normal"] - p["offset"] > 0.5) | (tc @ q["normal"] - q["offset"] > 0.5)]
                    ok = beyond.sum() < 0.001 * ta.sum()
                if ok and (best is None or d > best[0]):
                    best = (d, min(p["area"], q["area"]))
        if best is not None:
            out[ax] = float(best[0])
    return out


def best_dims(meas: dict):
    """(L, W, H, how) - between-faces values where available, else the envelope."""
    o = meas["overall"]
    bf = o.get("between_faces", {})
    L = bf.get("X", o["length_x"])
    W = bf.get("Y", o["width_y"])
    H = bf.get("Z", o["height_z"])
    how = "between faces" if len(bf) == 3 else ("envelope" if not bf else "faces/envelope")
    return float(L), float(W), float(H), how


def axis_label(nrm) -> str:
    k = int(np.argmax(np.abs(nrm)))
    if abs(nrm[k]) > 0.97:
        return "XYZ"[k]
    return "oblique"


def measure_all(m_mm: trimesh.Trimesh, material: str = "Steel", sdf_volume_mm3=None) -> dict:
    res = dict(overall=overall(m_mm), mass=mass_props(m_mm, material, sdf_volume_mm3))
    try:
        res["circles"] = find_circles(m_mm)
    except Exception as ex:
        log.warning("circle detection failed: %s", ex)
        res["circles"] = []
    try:
        planes = find_planes(m_mm)
        res["planes"] = [dict(normal=[float(v) for v in p["normal"]], offset=p["offset"], area=p["area"],
                              flatness_rms=p["flatness_rms"], axis=axis_label(p["normal"])) for p in planes]
        res["plane_relations"] = plane_relations(planes)
        res["overall"]["between_faces"] = face_dims(planes, res["overall"], float(m_mm.area), m_mm)
    except Exception as ex:
        log.warning("plane detection failed: %s", ex)
        res["planes"], res["plane_relations"] = [], []
    return res


In [ ]:
%%writefile /content/partscan/drawings.py
"""Dimensioned orthographic drawings and QA images (matplotlib, no GPU / OpenGL needed)."""
from __future__ import annotations

import io
import logging

import cv2
import numpy as np

log = logging.getLogger("partscan")

INK = "#1d2733"
DIM = "#c2410c"
HOLE = "#1d4ed8"


def _silhouette(V2, F, px_per_mm, pad_mm=2.0):
    lo = V2.min(0) - pad_mm
    hi = V2.max(0) + pad_mm
    W = int(np.ceil((hi[0] - lo[0]) * px_per_mm)) + 1
    H = int(np.ceil((hi[1] - lo[1]) * px_per_mm)) + 1
    img = np.zeros((H, W), np.uint8)
    pix = np.round((V2 - lo) * px_per_mm * 8).astype(np.int32)       # 3 bits of sub-pixel precision
    pix[:, 1] = H * 8 - 1 - pix[:, 1]                                  # v up
    from .geometry import fill_triangles
    fill_triangles(img, pix[F], 1, shift=3)                            # union of all faces
    return img, lo, H


def _contours(img, lo, H, px_per_mm):
    cnts, hier = cv2.findContours(img, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    out = []
    for c, h in zip(cnts, hier[0] if hier is not None else []):
        if len(c) < 6:
            continue
        p = c[:, 0, :].astype(float)
        mm = np.stack([p[:, 0] / px_per_mm + lo[0], (H - 1 - p[:, 1]) / px_per_mm + lo[1]], 1)
        out.append((mm, h[3] >= 0))   # (points, is_hole)
    return out


def _feature_edges(mesh, angle_deg=40.0, work_faces=20000):
    """Sharp edges (crease lines) for the drawing, found on a decimated + smoothed copy so that
    scan noise does not show up as speckle."""
    try:
        import trimesh
        m = mesh
        if len(m.faces) > work_faces * 1.5:
            m = m.simplify_quadric_decimation(face_count=work_faces)
        m = m.copy()
        trimesh.smoothing.filter_taubin(m, lamb=0.5, nu=0.53, iterations=10)
        ang = m.face_adjacency_angles
        e = m.face_adjacency_edges[ang > np.radians(angle_deg)]
        return m.vertices[e]
    except Exception:
        return np.zeros((0, 2, 3))


def _dim(ax, p0, p1, offset, text, horizontal=True):
    """Dimension line between p0 and p1 (2D), pushed out by offset (mm)."""
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    if horizontal:
        y = offset
        a, b = np.array([p0[0], y]), np.array([p1[0], y])
        ax.plot([p0[0], p0[0]], [p0[1], y], color=DIM, lw=0.6)
        ax.plot([p1[0], p1[0]], [p1[1], y], color=DIM, lw=0.6)
        ax.annotate("", a, b, arrowprops=dict(arrowstyle="<->", color=DIM, lw=0.9, shrinkA=0, shrinkB=0))
        ax.text((a[0] + b[0]) / 2, y, text, color=DIM, ha="center", va="bottom", fontsize=9, fontweight="bold",
                bbox=dict(fc="white", ec="none", pad=0.6))
    else:
        x = offset
        a, b = np.array([x, p0[1]]), np.array([x, p1[1]])
        ax.plot([p0[0], x], [p0[1], p0[1]], color=DIM, lw=0.6)
        ax.plot([p1[0], x], [p1[1], p1[1]], color=DIM, lw=0.6)
        ax.annotate("", a, b, arrowprops=dict(arrowstyle="<->", color=DIM, lw=0.9, shrinkA=0, shrinkB=0))
        ax.text(x, (a[1] + b[1]) / 2, text, color=DIM, ha="right", va="center", fontsize=9, fontweight="bold",
                rotation=90, bbox=dict(fc="white", ec="none", pad=0.6))


def _view(ax, mesh, ui, vi, title, circles, view_axis, px_per_mm, dims, mesh_small=None):
    V = mesh.vertices
    V2 = V[:, [ui, vi]]
    ms = mesh_small if mesh_small is not None else mesh
    img, lo, H = _silhouette(ms.vertices[:, [ui, vi]], ms.faces, px_per_mm)
    for pts, hole in _contours(img, lo, H, px_per_mm):
        ax.fill(pts[:, 0], pts[:, 1], color="white" if hole else "#e8edf3", zorder=1 if not hole else 2)
        ax.plot(np.r_[pts[:, 0], pts[0, 0]], np.r_[pts[:, 1], pts[0, 1]], color=INK, lw=1.1, zorder=3)
    E = _feature_edges(mesh)
    if len(E):
        seg = E[:, :, [ui, vi]]
        from matplotlib.collections import LineCollection
        ax.add_collection(LineCollection(seg, colors=INK, linewidths=0.35, alpha=0.55, zorder=3))
    umin, umax = V2[:, 0].min(), V2[:, 0].max()
    vmin, vmax = V2[:, 1].min(), V2[:, 1].max()
    span = max(umax - umin, vmax - vmin)
    off = 0.10 * span + 3
    _dim(ax, (umin, vmin), (umax, vmin), vmin - off, f"{dims[0]:.2f}", horizontal=True)
    _dim(ax, (umin, vmin), (umin, vmax), umin - off, f"{dims[1]:.2f}", horizontal=False)
    for c in circles:
        if c["axis"] != view_axis:
            continue
        cu, cv = c["center"][ui], c["center"][vi]
        r = c["diameter"] / 2
        t = np.linspace(0, 2 * np.pi, 120)
        ax.plot(cu + r * np.cos(t), cv + r * np.sin(t), color=HOLE, lw=1.0, zorder=4)
        ax.plot([cu - r * 1.25, cu + r * 1.25], [cv, cv], color=HOLE, lw=0.5, ls="-.", zorder=4)
        ax.plot([cu, cu], [cv - r * 1.25, cv + r * 1.25], color=HOLE, lw=0.5, ls="-.", zorder=4)
        lab = f"≈Ø{c['diameter']:.1f} (rough)" if c.get("rough") else f"Ø{c['diameter']:.2f}"
        ax.annotate(lab, (cu + r * 0.7, cv + r * 0.7), (cu + r + 0.06 * span, cv + r + 0.06 * span),
                    color=HOLE, fontsize=8.5, fontweight="bold",
                    arrowprops=dict(arrowstyle="-", color=HOLE, lw=0.7), zorder=5,
                    bbox=dict(fc="white", ec="none", pad=0.5))
    ax.set_title(title, fontsize=10.5, color=INK, loc="left", fontweight="bold")
    ax.set_aspect("equal")
    ax.set_xlim(umin - off - 0.12 * span - 4, umax + 0.22 * span + 4)
    ax.set_ylim(vmin - off - 0.08 * span - 4, vmax + 0.12 * span + 4)
    ax.axis("off")


def _iso(ax, mesh, faces_max=8000):
    m = mesh
    if len(m.faces) > faces_max:
        try:
            m = m.simplify_quadric_decimation(face_count=faces_max)
        except Exception:
            pass
    az, el = np.radians(-50), np.radians(28)
    Rz = np.array([[np.cos(az), -np.sin(az), 0], [np.sin(az), np.cos(az), 0], [0, 0, 1]])
    Rx = np.array([[1, 0, 0], [0, np.cos(el), -np.sin(el)], [0, np.sin(el), np.cos(el)]])
    R0 = np.array([[1, 0, 0], [0, 0, 1], [0, -1, 0]], float)   # world Z -> screen up
    R = Rx @ R0 @ Rz
    V = m.vertices @ R.T
    T = V[m.faces]
    n = m.face_normals @ R.T
    light = np.array([0.3, 0.5, 0.8])
    light /= np.linalg.norm(light)
    vis = n[:, 2] > -0.05
    shade = 0.35 + 0.65 * np.clip(n @ light, 0, 1)
    if m.visual.kind == "vertex":
        col = m.visual.vertex_colors[m.faces][:, :, :3].mean(1) / 255.0
    else:
        col = np.tile([0.72, 0.75, 0.8], (len(m.faces), 1))
    col = np.clip(col * shade[:, None], 0, 1)
    order = np.argsort(T[:, :, 2].mean(1))
    order = order[vis[order]]
    from matplotlib.collections import PolyCollection
    pc = PolyCollection(T[order][:, :, :2], facecolors=col[order], edgecolors=col[order], linewidths=0.2)
    ax.add_collection(pc)
    ax.set_xlim(V[:, 0].min() - 3, V[:, 0].max() + 3)
    ax.set_ylim(V[:, 1].min() - 3, V[:, 1].max() + 3)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("Isometric", fontsize=10.5, color=INK, loc="left", fontweight="bold")


def ortho_drawing(mesh_mm, meas: dict, path: str, title: str = "") -> str:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    from .measure import best_dims
    o = meas["overall"]
    L, Wd, Hd, how = best_dims(meas)
    circles = meas.get("circles", [])
    ext = np.ptp(mesh_mm.vertices, 0)
    ppm = float(np.clip(900 / max(ext.max(), 1e-3), 4, 40))
    try:   # silhouettes are rasterised triangle by triangle: use a lighter copy of the mesh
        ms = mesh_mm.simplify_quadric_decimation(face_count=120000) if len(mesh_mm.faces) > 150000 else mesh_mm
    except Exception:
        ms = mesh_mm
    fig, axs = plt.subplots(2, 2, figsize=(11, 8.5), dpi=150)
    _view(axs[0, 0], mesh_mm, 0, 1, "Top  (X right, Y up)", circles, "Z", ppm, (L, Wd), ms)
    _iso(axs[0, 1], mesh_mm)
    _view(axs[1, 0], mesh_mm, 0, 2, "Front  (X right, Z up)", circles, "Y", ppm, (L, Hd), ms)
    _view(axs[1, 1], mesh_mm, 1, 2, "Side  (Y right, Z up)", circles, "X", ppm, (Wd, Hd), ms)
    fig.suptitle(title or f"PartScan  -  {L:.2f} x {Wd:.2f} x {Hd:.2f} mm ({how})   all dimensions in mm",
                 fontsize=12, color=INK, x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(path, facecolor="white")
    plt.close(fig)
    return path


def qa_montage(kf, masks, T_wc, mesh, path, n: int = 8) -> str:
    """Keyframes with the part silhouette (orange) and the reprojected final mesh (cyan)."""
    idx = np.linspace(0, len(kf) - 1, min(n, len(kf))).astype(int)
    try:
        ms = mesh.simplify_quadric_decimation(face_count=40000) if len(mesh.faces) > 60000 else mesh
    except Exception:
        ms = mesh
    tiles = []
    for i in idx:
        f = kf[i]
        rgb = f.rgb.copy()
        H, W = rgb.shape[:2]
        if masks[i] is not None:
            cnts, _ = cv2.findContours(masks[i].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(rgb, cnts, -1, (255, 140, 0), max(2, W // 640))
        # reprojected mesh silhouette
        from .segment import mesh_silhouette
        sil = mesh_silhouette(ms.vertices, ms.faces, f.K, np.linalg.inv(T_wc[i]), (W, H)).astype(np.uint8)
        cnts, _ = cv2.findContours(sil, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(rgb, cnts, -1, (0, 200, 255), max(1, W // 960))
        ys, xs = np.nonzero(sil)
        if len(xs):
            cx, cy = int(xs.mean()), int(ys.mean())
            s = int(max(np.ptp(xs), np.ptp(ys)) * 0.9 + 60)
            x0, y0 = max(0, cx - s), max(0, cy - s)
            rgb = rgb[y0:min(H, cy + s), x0:min(W, cx + s)]
        tiles.append(cv2.resize(rgb, (360, 360), interpolation=cv2.INTER_AREA))
    while len(tiles) % 4:
        tiles.append(np.full((360, 360, 3), 255, np.uint8))
    rows = [np.concatenate(tiles[k:k + 4], 1) for k in range(0, len(tiles), 4)]
    img = np.concatenate(rows, 0)
    cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 88])
    return path


In [ ]:
%%writefile /content/partscan/viewer.py
"""Self-contained interactive 3D viewer (three.js inlined; works offline, phone friendly).

Tap/click two points on the part to measure the distance between them (mm, with dX/dY/dZ).
Buttons toggle the bounding-box dimensions, detected holes, colour and wireframe.
"""
from __future__ import annotations

import base64
import io
import json
import logging
import os
import tarfile
import urllib.request

import numpy as np

log = logging.getLogger("partscan")
THREE_VERSION = "0.170.0"
_FILES = {"three": "build/three.module.min.js", "orbit": "examples/jsm/controls/OrbitControls.js"}
CACHE = os.path.join(os.path.expanduser("~"), ".cache", "partscan", f"three-{THREE_VERSION}")


def _fetch(url, timeout=20):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()


def three_sources() -> dict | None:
    """three.js module sources (cached). Tries jsDelivr, then the npm registry tarball."""
    os.makedirs(CACHE, exist_ok=True)
    paths = {k: os.path.join(CACHE, os.path.basename(v)) for k, v in _FILES.items()}
    if all(os.path.exists(p) for p in paths.values()):
        return {k: open(p, "rb").read() for k, p in paths.items()}
    try:
        for k, rel in _FILES.items():
            data = _fetch(f"https://cdn.jsdelivr.net/npm/three@{THREE_VERSION}/{rel}")
            open(paths[k], "wb").write(data)
        return {k: open(p, "rb").read() for k, p in paths.items()}
    except Exception as ex:
        log.info("jsDelivr unavailable (%s); trying the npm registry", ex)
    try:
        tgz = _fetch(f"https://registry.npmjs.org/three/-/three-{THREE_VERSION}.tgz", timeout=60)
        with tarfile.open(fileobj=io.BytesIO(tgz)) as tf:
            for k, rel in _FILES.items():
                open(paths[k], "wb").write(tf.extractfile("package/" + rel).read())
        return {k: open(p, "rb").read() for k, p in paths.items()}
    except Exception as ex:
        log.warning("could not download three.js (%s); the viewer will load it from the CDN when opened", ex)
        return None


def _b64(a: np.ndarray) -> str:
    return base64.b64encode(np.ascontiguousarray(a).tobytes()).decode("ascii")


def viewer_payload(mesh_m, meas: dict, max_faces: int = 160_000) -> dict:
    m = mesh_m.copy()
    m.apply_scale(1000.0)
    src_V = np.asarray(m.vertices)
    src_C = (np.asarray(m.visual.vertex_colors[:, :3], np.uint8)
             if getattr(m.visual, "kind", None) == "vertex" and len(m.visual.vertex_colors) == len(src_V) else None)
    if len(m.faces) > max_faces:
        try:
            m = m.simplify_quadric_decimation(face_count=max_faces)
        except Exception:
            pass
    V = np.asarray(m.vertices, np.float32)
    F = np.asarray(m.faces, np.uint32)
    if src_C is not None:   # decimation drops vertex colours: take them from the nearest original vertex
        from scipy.spatial import cKDTree
        C = src_C[cKDTree(src_V).query(V)[1]]
    else:
        C = np.full((len(V), 3), 180, np.uint8)
    from .measure import best_dims
    o = meas["overall"]
    L, W, H, _ = best_dims(meas)
    circles = [dict(kind=c["kind"], axis=c["axis"], d=round(c["diameter"], 3), c=[round(v, 3) for v in c["center"]],
                    length=round(c["length"], 2), rough=bool(c.get("rough"))) for c in meas.get("circles", [])]
    return dict(V=_b64(V), F=_b64(F), C=_b64(C), nv=len(V), nf=len(F),
                bbox=dict(min=o["bbox_min"], max=o["bbox_max"], L=L, W=W, H=H),
                circles=circles)


CSS = """
:root{--bg:#f4f6f9;--panel:rgba(255,255,255,.92);--ink:#16202b;--muted:#5b6876;--accent:#c2410c;--blue:#1d4ed8;--line:#d6dde5}
@media (prefers-color-scheme:dark){:root{--bg:#11161c;--panel:rgba(25,31,39,.92);--ink:#e8edf2;--muted:#9aa7b4;--accent:#fb923c;--blue:#7aa2ff;--line:#2c3540}}
html,body{margin:0;height:100%;background:var(--bg);color:var(--ink);font:14px/1.35 system-ui,-apple-system,Segoe UI,Roboto,sans-serif;overflow:hidden}
#app{position:fixed;inset:0;touch-action:none}
#panel{position:fixed;left:12px;top:12px;right:12px;max-width:430px;background:var(--panel);border:1px solid var(--line);border-radius:12px;padding:10px 12px;backdrop-filter:blur(6px);box-shadow:0 4px 18px rgba(0,0,0,.08)}
#panel h1{font-size:14px;margin:0 0 6px;font-weight:650}
.row{display:flex;flex-wrap:wrap;gap:6px;margin:6px 0}
button{font:inherit;font-size:13px;padding:6px 10px;border-radius:8px;border:1px solid var(--line);background:transparent;color:var(--ink);cursor:pointer}
button.on{background:var(--ink);color:var(--bg);border-color:var(--ink)}
#readout{font-variant-numeric:tabular-nums;max-height:30vh;overflow:auto}
#readout div{padding:3px 0;border-top:1px solid var(--line)}
.hint{color:var(--muted);font-size:12.5px}
.lbl{position:fixed;transform:translate(-50%,-50%);pointer-events:none;font:600 12px system-ui,sans-serif;padding:2px 6px;border-radius:6px;background:var(--panel);border:1px solid var(--line);white-space:nowrap;font-variant-numeric:tabular-nums}
.lbl.dim{color:var(--accent)} .lbl.hole{color:var(--blue)} .lbl.meas{color:var(--ink);background:#fde68a;color:#3b2a00;border-color:#f59e0b}
"""

JS = r"""
import * as THREE from 'three';
import { OrbitControls } from 'three/addons/controls/OrbitControls.js';
const D = JSON.parse(document.getElementById('partscan-data').textContent);
function dec(s, T){const b=atob(s);const u=new Uint8Array(b.length);for(let i=0;i<b.length;i++)u[i]=b.charCodeAt(i);return new T(u.buffer);}
const V=dec(D.V,Float32Array), F=dec(D.F,Uint32Array), C=dec(D.C,Uint8Array);
const app=document.getElementById('app');
const renderer=new THREE.WebGLRenderer({antialias:true,alpha:true,preserveDrawingBuffer:true});
renderer.setPixelRatio(Math.min(2,window.devicePixelRatio||1));
app.appendChild(renderer.domElement);
const scene=new THREE.Scene();
const cam=new THREE.PerspectiveCamera(35,1,0.1,100000); cam.up.set(0,0,1);
const ctl=new OrbitControls(cam,renderer.domElement); ctl.enableDamping=true;
scene.add(new THREE.HemisphereLight(0xffffff,0x667788,1.6));
const sun=new THREE.DirectionalLight(0xffffff,1.6); sun.position.set(1,-1.5,2.5); scene.add(sun);
const g=new THREE.BufferGeometry();
g.setAttribute('position',new THREE.BufferAttribute(V,3));
g.setAttribute('color',new THREE.BufferAttribute(C,3,true));
g.setIndex(new THREE.BufferAttribute(F,1)); g.computeVertexNormals(); g.computeBoundingSphere();
const matC=new THREE.MeshStandardMaterial({vertexColors:true,roughness:.75,metalness:.05});
const matG=new THREE.MeshStandardMaterial({color:0xb9c3cf,roughness:.55,metalness:.1});
const mesh=new THREE.Mesh(g,matC); scene.add(mesh);
const wire=new THREE.LineSegments(new THREE.WireframeGeometry(g),new THREE.LineBasicMaterial({color:0x334155,transparent:true,opacity:.25})); wire.visible=false; scene.add(wire);
const bb=D.bbox, mn=new THREE.Vector3(...bb.min), mx=new THREE.Vector3(...bb.max);
const ctr=mn.clone().add(mx).multiplyScalar(.5), size=mx.clone().sub(mn), R=size.length();
const grid=new THREE.GridHelper(Math.ceil(R*1.6/10)*10,Math.ceil(R*1.6/10),0x94a3b8,0xcbd5e1);
grid.rotation.x=Math.PI/2; grid.position.set(ctr.x,ctr.y,0); grid.material.transparent=true; grid.material.opacity=.45; scene.add(grid);
function resetView(){const fv=cam.fov*Math.PI/180, fh=2*Math.atan(Math.tan(fv/2)*cam.aspect);
  const dist=(R/2)/Math.sin(Math.min(fv,fh)/2)*1.12;
  ctl.target.copy(ctr);cam.position.copy(ctr).add(new THREE.Vector3(1.1,-1.5,1.0).normalize().multiplyScalar(dist));ctl.update();}
// ---- labels
const labels=[];
function label(text,pos,cls,group){const el=document.createElement('div');el.className='lbl '+cls;el.textContent=text;document.body.appendChild(el);const L={el,pos:pos.clone(),group};labels.push(L);return L;}
function lineMat(c){return new THREE.LineBasicMaterial({color:c,depthTest:false,transparent:true});}
// ---- bounding box dims
const boxG=new THREE.Group(); scene.add(boxG);
{const b=new THREE.Box3(mn,mx); const h=new THREE.Box3Helper(b,0xc2410c); h.material.depthTest=false; h.material.transparent=true; h.material.opacity=.6; boxG.add(h);
 label('L '+bb.L.toFixed(2),new THREE.Vector3(ctr.x,mn.y,mn.z),'dim','box');
 label('W '+bb.W.toFixed(2),new THREE.Vector3(mx.x,ctr.y,mn.z),'dim','box');
 label('H '+bb.H.toFixed(2),new THREE.Vector3(mx.x,mn.y,ctr.z),'dim','box');}
// ---- holes
const holeG=new THREE.Group(); scene.add(holeG);
for(const c of D.circles){
  const pts=[];const r=c.d/2;const ax=c.axis;
  for(let k=0;k<=64;k++){const t=k/64*Math.PI*2;const a=r*Math.cos(t),b=r*Math.sin(t);
    pts.push(ax==='Z'?new THREE.Vector3(c.c[0]+a,c.c[1]+b,c.c[2]):ax==='Y'?new THREE.Vector3(c.c[0]+a,c.c[1],c.c[2]+b):new THREE.Vector3(c.c[0],c.c[1]+a,c.c[2]+b));}
  const ln=new THREE.Line(new THREE.BufferGeometry().setFromPoints(pts),lineMat(0x1d4ed8)); ln.renderOrder=2; holeG.add(ln);
  label(c.rough?'≈Ø'+c.d.toFixed(1)+' (rough)':'Ø'+c.d.toFixed(2),new THREE.Vector3(...c.c),'hole','holes');
}
// ---- point-to-point measuring
const measG=new THREE.Group(); scene.add(measG);
const ray=new THREE.Raycaster(); let pending=null; const results=[];
const dotGeo=new THREE.SphereGeometry(Math.max(R*0.006,0.25),16,12), dotMat=new THREE.MeshBasicMaterial({color:0xf59e0b,depthTest:false});
function addDot(p){const s=new THREE.Mesh(dotGeo,dotMat);s.position.copy(p);s.renderOrder=3;measG.add(s);}
function readout(){const el=document.getElementById('readout');el.innerHTML=results.map((m,i)=>`<div><b>#${i+1}</b> ${m.d.toFixed(3)} mm <span class="hint">dX ${m.dx.toFixed(2)} · dY ${m.dy.toFixed(2)} · dZ ${m.dz.toFixed(2)}</span></div>`).join('')||'<span class="hint">Tap two points on the part to measure.</span>';}
let down=null;
renderer.domElement.addEventListener('pointerdown',e=>{down=[e.clientX,e.clientY];});
renderer.domElement.addEventListener('pointerup',e=>{
  if(!down||Math.hypot(e.clientX-down[0],e.clientY-down[1])>6)return;
  const r=renderer.domElement.getBoundingClientRect();
  ray.setFromCamera(new THREE.Vector2((e.clientX-r.left)/r.width*2-1,-(e.clientY-r.top)/r.height*2+1),cam);
  const hit=ray.intersectObject(mesh,false)[0]; if(!hit)return;
  addDot(hit.point);
  if(!pending){pending=hit.point.clone();return;}
  const a=pending,b=hit.point.clone(); pending=null;
  const ln=new THREE.Line(new THREE.BufferGeometry().setFromPoints([a,b]),lineMat(0xf59e0b)); ln.renderOrder=3; measG.add(ln);
  const m={d:a.distanceTo(b),dx:Math.abs(b.x-a.x),dy:Math.abs(b.y-a.y),dz:Math.abs(b.z-a.z)}; results.push(m);
  label(m.d.toFixed(2)+' mm',a.clone().add(b).multiplyScalar(.5),'meas','meas'); readout();
});
// ---- buttons
const state={box:true,holes:true,color:true,wire:false};
function sync(){boxG.visible=state.box;holeG.visible=state.holes;mesh.material=state.color?matC:matG;wire.visible=state.wire;
  for(const k of Object.keys(state)){const b=document.getElementById('b-'+k);if(b)b.classList.toggle('on',state[k]);}}
for(const k of Object.keys(state)){const b=document.getElementById('b-'+k);if(b)b.onclick=()=>{state[k]=!state[k];sync();};}
document.getElementById('b-clear').onclick=()=>{measG.clear();pending=null;results.length=0;for(let i=labels.length-1;i>=0;i--)if(labels[i].group==='meas'){labels[i].el.remove();labels.splice(i,1);}readout();};
document.getElementById('b-reset').onclick=resetView;
sync(); readout();
// ---- loop
const v=new THREE.Vector3();
function resize(){const w=app.clientWidth,h=app.clientHeight;renderer.setSize(w,h,false);renderer.domElement.style.width=w+'px';renderer.domElement.style.height=h+'px';cam.aspect=w/h;cam.updateProjectionMatrix();}
window.addEventListener('resize',resize); resize(); resetView();
renderer.setAnimationLoop(()=>{ctl.update();renderer.render(scene,cam);
  const w=app.clientWidth,h=app.clientHeight;
  for(const L of labels){const vis=(L.group==='box'&&state.box)||(L.group==='holes'&&state.holes)||L.group==='meas';
    v.copy(L.pos).project(cam); const show=vis&&v.z<1;
    L.el.style.display=show?'block':'none'; if(show){L.el.style.left=((v.x+1)/2*w)+'px';L.el.style.top=((1-v.y)/2*h)+'px';}}
});
window.__partscan_ready=true;
"""

HTML = """<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=1">
<title>PartScan 3D viewer</title><style>{css}</style>
<script type="importmap">{importmap}</script></head><body>
<div id="app"></div>
<div id="panel"><h1>{title}</h1>
<div class="row"><button id="b-box">Dimensions</button><button id="b-holes">Holes</button><button id="b-color">Colour</button><button id="b-wire">Wireframe</button><button id="b-clear">Clear</button><button id="b-reset">Reset view</button></div>
<div id="readout"></div></div>
<script type="application/json" id="partscan-data">{data}</script>
<script type="module">{js}</script></body></html>"""


def write_viewer(path: str, mesh_m, meas: dict, title: str | None = None) -> str:
    src = three_sources()
    if src is not None:
        im = {"three": "data:text/javascript;base64," + base64.b64encode(src["three"]).decode(),
              "three/addons/controls/OrbitControls.js": "data:text/javascript;base64," + base64.b64encode(src["orbit"]).decode()}
    else:
        base = f"https://cdn.jsdelivr.net/npm/three@{THREE_VERSION}/"
        im = {"three": base + _FILES["three"], "three/addons/controls/OrbitControls.js": base + _FILES["orbit"]}
    from .measure import best_dims
    L, W, H, _ = best_dims(meas)
    title = title or f"Part {L:.2f} × {W:.2f} × {H:.2f} mm"
    data = json.dumps(viewer_payload(mesh_m, meas)).replace("</", "<\\/")
    html = HTML.format(css=CSS, importmap=json.dumps({"imports": im}), title=title, data=data, js=JS)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(html)
    return path


In [ ]:
%%writefile /content/partscan/report.py
"""Self-contained HTML measurement report (images embedded, prints cleanly)."""
from __future__ import annotations

import base64
import datetime as _dt
import html
import os

import numpy as np


def _img(path):
    if not path or not os.path.exists(path):
        return ""
    ext = os.path.splitext(path)[1].lower().strip(".")
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext
    return f"data:image/{mime};base64," + base64.b64encode(open(path, "rb").read()).decode()


def quality_rows(q: dict):
    """(label, value, status) with status in good / ok / poor / info."""
    def st(v, good, ok, higher_better=False):
        if v is None or (isinstance(v, float) and not np.isfinite(v)):
            return "info"
        if higher_better:
            return "good" if v >= good else ("ok" if v >= ok else "poor")
        return "good" if v <= good else ("ok" if v <= ok else "poor")
    rows = []
    ref = q.get("scale_reference", "")
    rows.append(("Scale reference", ref, "good" if "board" in ref.lower() else "ok"))
    if "board_frames" in q:
        rows.append(("Board seen in keyframes", f"{q['board_frames']} / {q.get('keyframes', '?')}",
                     st(q["board_frames"] / max(1, q.get("keyframes", 1)), 0.6, 0.3, True)))
    if "board_ba_rms_px" in q:
        rows.append(("Board reprojection error", f"{q['board_ba_rms_px']:.2f} px", st(q["board_ba_rms_px"], 0.5, 1.0)))
    if "arkit_scale_error_pct" in q:
        rows.append(("Phone tracking scale error (corrected)", f"{q['arkit_scale_error_pct']:+.2f} %", "info"))
    if "k1" in q:
        rows.append(("Lens distortion k1 (corrected)", f"{q['k1']:+.4f}", "info"))
    if "feature_points" in q:
        rows.append(("Photogrammetry tie points", f"{q['feature_points']:,}", st(q["feature_points"], 2000, 300, True)))
    if "ba_rms_tracks_px" in q:
        rows.append(("Tie-point reprojection error", f"{q['ba_rms_tracks_px']:.2f} px", st(q["ba_rms_tracks_px"], 0.8, 1.5)))
    if "silhouette_iou_median" in q:
        rows.append(("Model vs photo outlines (IoU)", f"{q['silhouette_iou_median']:.3f}",
                     st(q["silhouette_iou_median"], 0.95, 0.9, True)))
    if "outline_to_model_area_ratio" in q:
        rv = q["outline_to_model_area_ratio"]
        rows.append(("Photo outline area / model area", f"{rv:.3f}", st(abs(rv - 1), 0.02, 0.05)))
    if "depth_fit_sigma_mm" in q:
        rows.append(("Depth vs photogrammetry (per frame)", f"{q['depth_fit_sigma_mm']:.2f} mm", st(q["depth_fit_sigma_mm"], 1.0, 2.5)))
    if "depth_to_surface_rms_mm" in q:
        rows.append(("Depth vs final surface (RMS)", f"{q['depth_to_surface_rms_mm']:.2f} mm",
                     st(q["depth_to_surface_rms_mm"], 0.5, 1.0)))
    if "fusion_voxel_mm" in q:
        rows.append(("Voxel size", f"{q['fusion_voxel_mm']:.2f} mm", "info"))
    rows.append(("Depth model / segmentation", f"{q.get('depth_model', '?')} / {q.get('segmenter', '?')}", "info"))
    rows.append(("Frames used", f"{q.get('keyframes', '?')} of {q.get('frames_total', '?')}", "info"))
    return rows


CSS = """
:root{--ink:#16202b;--muted:#5b6876;--line:#dde3ea;--bg:#ffffff;--card:#f6f8fa;--good:#15803d;--ok:#b45309;--poor:#b91c1c;--accent:#c2410c}
@media (prefers-color-scheme:dark){:root:not([data-theme=light]){--ink:#e8edf2;--muted:#9aa7b4;--line:#2c3540;--bg:#11161c;--card:#19202a;--good:#4ade80;--ok:#fbbf24;--poor:#f87171;--accent:#fb923c}}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.5 system-ui,-apple-system,Segoe UI,Roboto,sans-serif}
main{max-width:1060px;margin:0 auto;padding:24px 16px 48px}
h1{font-size:24px;margin:0 0 4px} h2{font-size:17px;margin:28px 0 10px}
.sub{color:var(--muted)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:18px 0}
.kpi{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:10px 12px}
.kpi b{display:block;font-size:22px;font-variant-numeric:tabular-nums} .kpi span{color:var(--muted);font-size:13px}
table{width:100%;border-collapse:collapse;font-variant-numeric:tabular-nums}
td,th{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top} th{font-size:13px;color:var(--muted);font-weight:600}
td.v{white-space:nowrap;font-weight:600} td.n{color:var(--muted);font-size:13px}
.dot{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:7px}
.good{background:var(--good)} .ok{background:var(--ok)} .poor{background:var(--poor)} .info{background:var(--line)}
img{max-width:100%;border:1px solid var(--line);border-radius:10px;background:#fff}
.warn{border-left:4px solid var(--ok);background:var(--card);padding:8px 12px;margin:6px 0;border-radius:6px}
.wrap{overflow-x:auto}
footer{color:var(--muted);font-size:12.5px;margin-top:30px}
@media print{.noprint{display:none}}
"""


def write_report(path, meas, q, warnings, cfg, rows, files) -> str:
    from .measure import best_dims
    o, m = meas["overall"], meas["mass"]
    holes = [c for c in meas.get("circles", []) if c["kind"] == "hole"]
    L, W, H, how = best_dims(meas)
    kpis = [(f"{L:.2f}", "Length X, mm"), (f"{W:.2f}", "Width Y, mm"), (f"{H:.2f}", f"Height Z, mm ({how})")]
    if m["volume_mm3"] == m["volume_mm3"]:
        kpis.append((f"{m['volume_mm3'] / 1000:.2f}", "Volume, cm³"))
    if m.get("mass_g") == m.get("mass_g") and m.get("mass_g"):
        kpis.append((f"{m['mass_g']:.0f}", f"Mass ({m['material']}), g"))
    n_rough = sum(1 for c in holes if c.get("rough"))
    kpis.append((str(len(holes)), "Holes found" + (f" ({n_rough} rough)" if n_rough else "")))
    e = html.escape
    kp = "".join(f'<div class="kpi"><b>{a}</b><span>{e(b)}</span></div>' for a, b in kpis)
    mt = "".join(f"<tr><td>{e(a)}</td><td class='v'>{e(b)}</td><td class='n'>{e(c)}</td></tr>" for a, b, c in rows)
    qt = "".join(f"<tr><td><span class='dot {s}'></span>{e(a)}</td><td class='v'>{e(str(b))}</td></tr>"
                 for a, b, s in quality_rows(q))
    wr = "".join(f"<div class='warn'>{e(w)}</div>" for w in warnings)
    draw = _img(files.get("drawing"))
    qa = _img(files.get("qa"))
    ts = _dt.datetime.now().strftime("%Y-%m-%d %H:%M")
    doc = f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>PartScan report</title><style>{CSS}</style></head><body><main>
<h1>Part measurement report</h1>
<div class="sub">{e(q.get('source', ''))} · {q.get('keyframes', '?')} keyframes · {e(q.get('scale_reference', ''))} · {ts}</div>
<div class="kpis">{kp}</div>
{wr}
<h2>Dimensioned views</h2>
{'<img src="' + draw + '" alt="Dimensioned drawing">' if draw else '<p class="sub">drawing unavailable</p>'}
<h2>Measurements</h2>
<p class="sub">Frame: the part as it rested on the board/table. Z is up from the support plane, X runs along the
longest side of the footprint, and the origin is the centre of the footprint. Values in mm unless noted.</p>
<div class="wrap"><table><tr><th>Feature</th><th>Value</th><th>Note</th></tr>{mt}</table></div>
<h2>Capture quality</h2>
<div class="wrap"><table>{qt}</table></div>
<p class="sub">Green/amber/red are rules of thumb. The only proof of accuracy for a given part is a caliper check:
measure one feature by hand, enter it as the reference and re-run - the report then shows the error.</p>
<h2>Silhouette check</h2>
<p class="sub">Orange = part outline found in the photo; cyan = outline of the final 3D model projected back. They should coincide.</p>
{'<img src="' + qa + '" alt="Silhouette QA">' if qa else ''}
<footer>Generated by PartScan (Colab). Meshes: part_mm.stl / .obj / .ply in millimetres, part.glb in metres.</footer>
</main></body></html>"""
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(doc)
    return path


In [ ]:
%%writefile /content/partscan/export.py
"""Write every deliverable: meshes, drawings, reports, interactive viewer, zip."""
from __future__ import annotations

import csv
import json
import logging
import os
import zipfile

import numpy as np

log = logging.getLogger("partscan")


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return jsonable(o.tolist())
    if isinstance(o, (np.floating,)):
        v = float(o)
        return None if not np.isfinite(v) else v
    if isinstance(o, float):
        return None if not np.isfinite(o) else o
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o


def measurement_rows(meas: dict) -> list:
    """Flat (feature, value_mm, note) rows used by the CSV, the UI table and the report."""
    o, m = meas["overall"], meas["mass"]
    bf = o.get("between_faces", {})
    rows = []
    for ax, key, what in (("X", "length_x", "Length X"), ("Y", "width_y", "Width Y"), ("Z", "height_z", "Height Z")):
        if ax in bf:
            rows.append((f"{what} (between faces)", bf[ax], f"plane fit; envelope incl. scan noise {o[key]:.2f}"))
        else:
            rows.append((f"{what} (envelope)", o[key], "max extent" + (" above the support plane" if ax == "Z" else
                                                                         ", footprint rectangle")))
    rows.append(("Oriented box (min volume)", " x ".join(f"{v:.2f}" for v in o["obb_extents"]), "any orientation"))
    if m["volume_mm3"] == m["volume_mm3"]:
        rows.append(("Volume", f"{m['volume_mm3'] / 1000:.3f} cm³", "" if m["watertight"] else "from the fused volume"))
    rows.append(("Surface area", f"{m['area_mm2'] / 100:.2f} cm²", ""))
    if m.get("mass_g") == m.get("mass_g") and m.get("mass_g"):
        rows.append((f"Mass ({m['material']})", f"{m['mass_g']:.1f} g", f"{m['density_g_cm3']} g/cm³"))
    for k, c in enumerate(meas.get("circles", []), 1):
        what = "Hole" if c["kind"] == "hole" else "Round feature"
        ctr = ", ".join(f"{v:.2f}" for v in c["center"])
        if c.get("rough"):
            rows.append((f"{what} {k} Ø (axis {c['axis']}) - ROUGH, Ø uncertain", c["diameter"],
                         f"bore only partly captured, probably undersized: scan straight into it or measure it "
                         f"directly; centre ({ctr}), length {c['length']:.1f}"))
            continue
        rows.append((f"{what} {k} Ø (axis {c['axis']})", c["diameter"],
                     f"centre ({ctr}), length {c['length']:.1f}, spread {c['diameter_spread']:.2f}"))
    for r in meas.get("plane_relations", []):
        name = "Wall thickness / width" if r["kind"] == "thickness" else "Step height"
        from .measure import axis_label
        rows.append((f"{name} (along {axis_label(r['normal'])})", r["distance"], "between parallel faces"))
    ref = meas.get("reference")
    if ref and ref.get("measured_mm") is not None:
        rows.append((f"Caliper check: {ref['name']}", ref["measured_mm"],
                     f"you measured {ref['reference_mm']:.3f} -> error {ref['error_mm']:+.3f} mm "
                     f"({ref['error_pct']:+.2f}%)" + (" - scale corrected" if ref.get("applied") else "")))
    out = []
    for name, val, note in rows:
        out.append((name, f"{val:.3f}" if isinstance(val, (float, np.floating)) else str(val), note))
    return out


def export_all(out_dir, mesh_m, mesh_mm, meas, quality, warnings, cfg, qa=None, progress=None,
               extra_files=None) -> dict:
    from . import drawings, report, viewer
    files = dict(extra_files or {})
    p = lambda k, frac: progress(frac, k) if progress else None
    p("meshes", 0.05)
    files["stl"] = os.path.join(out_dir, "part_mm.stl")
    mesh_mm.export(files["stl"])
    files["obj"] = os.path.join(out_dir, "part_mm.obj")
    mesh_mm.export(files["obj"])
    files["ply"] = os.path.join(out_dir, "part_mm.ply")
    mesh_mm.export(files["ply"])
    files["glb"] = os.path.join(out_dir, "part.glb")
    mesh_m.export(files["glb"])
    p("drawings", 0.3)
    try:
        files["drawing"] = drawings.ortho_drawing(mesh_mm, meas, os.path.join(out_dir, "drawing.png"))
    except Exception as ex:
        log.warning("drawing failed: %s", ex)
    if qa is not None:
        try:
            files["qa"] = drawings.qa_montage(qa["kf"], qa["masks"], qa["T_wc"], mesh_m,
                                              os.path.join(out_dir, "qa_silhouettes.jpg"))
        except Exception as ex:
            log.warning("QA montage failed: %s", ex)
    p("tables", 0.6)
    rows = measurement_rows(meas)
    files["csv"] = os.path.join(out_dir, "measurements.csv")
    with open(files["csv"], "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["feature", "value (mm unless noted)", "note"])
        w.writerows(rows)
    p("viewer", 0.7)
    try:
        files["viewer"] = viewer.write_viewer(os.path.join(out_dir, "viewer.html"), mesh_m, meas)
    except Exception as ex:
        log.warning("viewer failed: %s", ex)
    p("report", 0.85)
    files["report"] = report.write_report(os.path.join(out_dir, "report.html"), meas, quality, warnings, cfg,
                                          rows, files)
    files["json"] = os.path.join(out_dir, "report.json")
    with open(files["json"], "w") as fh:
        json.dump(jsonable(dict(measurements=meas, quality=quality, warnings=warnings, config=cfg.to_dict())), fh, indent=1)
    files["zip"] = os.path.join(out_dir, "partscan_results.zip")
    with zipfile.ZipFile(files["zip"], "w", zipfile.ZIP_DEFLATED) as z:
        for k, f in files.items():
            if k == "zip" or not f or not os.path.exists(f):
                continue
            z.write(f, os.path.basename(f))
    return files


In [ ]:
%%writefile /content/partscan/pipeline.py
"""End-to-end pipeline: capture -> poses -> silhouettes -> depth -> fusion -> mesh -> measurements."""
from __future__ import annotations

import gc
import json
import logging
import os
import threading
import time
from dataclasses import dataclass, field

import cv2
import numpy as np

from . import depth as depthmod
from . import segment as segmod
from .board import board_extent_part
from .config import Config
from .fusion import FusionFrame, auto_voxel, extract_mesh, fuse, sample_sdf
from .geometry import K_for_size, backproject_depth, inv_T, ransac_plane, transform_points
from .io_scans import load_scan, select_keyframes
from .measure import measure_all
from .meshproc import canonical_frame, clean_mesh, colorize
from .poses import feature_refine, init_poses, refine_with_board, undistort_keyframes

RUN_LOCK = threading.Lock()      # one reconstruction at a time: the GPU (and RAM) cannot take two

log = logging.getLogger("partscan")

STAGES = [("load", 0.10), ("poses", 0.14), ("photogrammetry", 0.16), ("silhouettes", 0.14),
          ("depth", 0.16), ("fusion", 0.14), ("mesh", 0.08), ("export", 0.08)]


class Progress:
    def __init__(self, cb=None):
        self.cb = cb
        self.base = 0.0
        self.span = 0.0
        self.t0 = time.time()

    def stage(self, name):
        acc = 0.0
        for n, w in STAGES:
            if n == name:
                self.base, self.span = acc, w
                break
            acc += w
        self(0.0, name)

    def __call__(self, frac, msg=""):
        if self.cb is not None:
            self.cb(min(0.999, self.base + self.span * float(frac)), msg)

    def sub(self, lo, hi):
        return lambda p, m="": self(lo + (hi - lo) * p, m)


@dataclass
class Result:
    out_dir: str
    files: dict = field(default_factory=dict)
    measurements: dict = field(default_factory=dict)
    quality: dict = field(default_factory=dict)
    warnings: list = field(default_factory=list)
    seconds: float = 0.0


def resolve_device(dev: str) -> str:
    if dev != "auto":
        return dev
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        return "cpu"


_MODEL_CACHE = {}


def get_depth_model(cfg: Config, device: str):
    if cfg.depth_model == "lidar":
        return depthmod.LidarUpsampler()
    key = ("promptda", cfg.promptda_checkpoint, device)
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = depthmod.PromptDADepth(cfg.promptda_checkpoint, cfg.promptda_size, device)
    _MODEL_CACHE[key].size = int(cfg.promptda_size)      # one model, any processing size
    return _MODEL_CACHE[key]


def get_segmenter(cfg: Config, device: str):
    if cfg.segmenter == "geometry":
        return segmod.GeometrySegmenter()
    key = ("sam2", cfg.sam2_checkpoint, device)
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = segmod.Sam2Segmenter(cfg.sam2_checkpoint, device)
    return _MODEL_CACHE[key]


def _depth_edges(D, rel=0.012):
    """Pixels on/next to depth discontinuities (mixed 'flying' pixels): the depth range inside a
    3x3 window exceeds rel*depth. Smooth surfaces stay in up to ~80 deg grazing."""
    valid = D > 0
    big = np.where(valid, D, 0).astype(np.float32)
    small = np.where(valid, D, 1e3).astype(np.float32)
    k = np.ones((3, 3), np.uint8)
    rng_ = cv2.dilate(big, k) - cv2.erode(small, k)
    e = (rng_ > rel * np.where(valid, D, 1)) | ~valid
    return cv2.dilate(e.astype(np.uint8), k) > 0


def _occluding_contours(D, mask, band_px, thr=0.003):
    """Band around the silhouette where the part's depth differs from the background's (an
    occluding contour). Contact lines (part meets the table) are NOT flagged."""
    m = mask.astype(np.float32)
    r = 2 * band_px + 1
    Dv = np.where(D > 0, D, 0).astype(np.float32)
    w_in = cv2.blur(m * (D > 0), (r, r))
    w_out = cv2.blur((1 - m) * (D > 0), (r, r))
    d_in = cv2.blur(Dv * m, (r, r)) / np.maximum(w_in, 1e-6)
    d_out = cv2.blur(Dv * (1 - m), (r, r)) / np.maximum(w_out, 1e-6)
    near = (w_in > 0.05) & (w_out > 0.05)
    return near & (np.abs(d_out - d_in) > thr)


def _edge_free_depth(D, keep, max_step: int = 10, gap: int = 3, margin: float = 0.0002):
    """For pixels excluded at depth edges: a safe 'nothing is closer than this' depth.
    Walk from the pixel towards the near side of the edge (minus the depth gradient) to the first
    clean pixel, read a second clean pixel `gap` further, and extrapolate that near surface back
    to the pixel. Whichever side of the edge the pixel's ray really hits, everything in front of
    this depth is empty."""
    h, w = D.shape
    Ds = cv2.GaussianBlur(np.where(D > 0, D, 0).astype(np.float32), (5, 5), 1.0)
    gx = cv2.Sobel(Ds, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(Ds, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.hypot(gx, gy) + 1e-12
    vv, uu = np.nonzero(~keep & (D > 0))
    free = np.zeros((h, w), np.float32)
    if len(uu) == 0:
        return free
    dx, dy = -gx[vv, uu] / mag[vv, uu], -gy[vv, uu] / mag[vv, uu]
    s1 = np.zeros(len(uu), np.int32)
    for s in range(1, max_step + 1):          # first clean pixel along the direction
        u1 = np.clip(np.round(uu + s * dx).astype(int), 0, w - 1)
        v1 = np.clip(np.round(vv + s * dy).astype(int), 0, h - 1)
        hit = (s1 == 0) & keep[v1, u1] & (D[v1, u1] > 0)
        s1[hit] = s
    found = s1 > 0
    s1f = np.where(found, s1, 1).astype(np.float32)
    u1 = np.clip(np.round(uu + s1f * dx).astype(int), 0, w - 1)
    v1 = np.clip(np.round(vv + s1f * dy).astype(int), 0, h - 1)
    u2 = np.clip(np.round(uu + (s1f + gap) * dx).astype(int), 0, w - 1)
    v2 = np.clip(np.round(vv + (s1f + gap) * dy).astype(int), 0, h - 1)
    d1, d2 = D[v1, u1], D[v2, u2]
    ok = found & keep[v2, u2] & (d2 > 0)
    # the actual step lengths after rounding
    l1 = np.hypot(u1 - uu, v1 - vv)
    l2 = np.hypot(u2 - uu, v2 - vv)
    slope = (d2 - d1) / np.maximum(l2 - l1, 1e-6)
    dfg = d1 - slope * l1
    dmin = cv2.erode(np.where(D > 0, D, 1e3).astype(np.float32), np.ones((5, 5), np.uint8))[vv, uu]
    dfg = np.where(ok, dfg, np.where(found, np.minimum(d1, D[vv, uu]), dmin))
    dfg = np.minimum(dfg, D[vv, uu]) - margin
    free[vv, uu] = np.where(dfg > 0.05, dfg, 0)
    return free


def _sdt(mask, f):
    """Silhouette signed distance; pixels without image data (undistortion border) never vote 'outside'."""
    sdt = segmod.mask_sdt(mask)
    if "valid" in f.extra:
        sdt[~f.extra["valid"]] = -1000.0
    return sdt


def _plane_pixels(D, Kd, T_wc, tol=0.0025):
    """Pixels whose depth puts them on the support plane z = 0 (part frame)."""
    h, w = D.shape
    v, u = np.mgrid[0:h, 0:w]
    z = D
    X = np.stack([(u - Kd[0, 2]) / Kd[0, 0] * z, (v - Kd[1, 2]) / Kd[1, 1] * z, z], -1)
    zw = X @ T_wc[2, :3] + T_wc[2, 3]
    return (D > 0) & (np.abs(zw) < tol)


def _angle_weight(D, Kd):
    """cos^2 of the angle between viewing ray and surface (from depth gradients)."""
    z = np.where(D > 0, D, np.nan).astype(np.float32)
    gx = cv2.Sobel(np.nan_to_num(z), cv2.CV_32F, 1, 0, ksize=3) / 8.0
    gy = cv2.Sobel(np.nan_to_num(z), cv2.CV_32F, 0, 1, ksize=3) / 8.0
    sx = gx * Kd[0, 0] / np.where(D > 0, D, 1)
    sy = gy * Kd[1, 1] / np.where(D > 0, D, 1)
    c2 = 1.0 / (1.0 + sx * sx + sy * sy)
    return np.clip(c2, 0.03, 1.0).astype(np.float32)


def run(scan_path: str, out_dir: str, cfg: Config, progress=None, depth_model=None, segmenter=None,
        hooks=None) -> Result:
    """Run everything. depth_model / segmenter may be injected (tests)."""
    t_start = time.time()
    os.makedirs(out_dir, exist_ok=True)
    work = os.path.join(out_dir, "_work")
    os.makedirs(work, exist_ok=True)
    P = Progress(progress)
    res = Result(out_dir)
    q = res.quality
    device = resolve_device(cfg.device)
    q["device"] = device
    hooks = hooks or {}

    # ------------------------------------------------------------------ 1. load
    P.stage("load")
    scan = load_scan(scan_path, work, cfg.max_candidates, progress=P.sub(0, 0.8))
    res.warnings += scan.notes
    kf = select_keyframes(scan.frames, cfg.n_keyframes, cfg.min_sharpness_ratio)
    if len(kf) < 12:
        raise RuntimeError(f"Only {len(kf)} usable frames. Record a longer, slower scan (30-60 s) around the part.")
    for f in kf:
        f.load_rgb()
    if not cfg.debug:
        import shutil
        shutil.rmtree(os.path.join(work, "extracted"), ignore_errors=True)   # everything needed is in memory
    keep_ids = {id(f) for f in kf}
    for f in scan.frames:
        if id(f) not in keep_ids:
            f.rgb_bytes = None
    gc.collect()
    q.update(source=scan.source, frames_total=scan.n_total, keyframes=len(kf), rgb_size=list(kf[0].rgb_size))
    log.info("%s: %d frames, %d keyframes, %dx%d", scan.source, scan.n_total, len(kf), *kf[0].rgb_size)

    # ------------------------------------------------------------------ 2. poses
    P.stage("poses")
    st = init_poses(kf, cfg, progress=P.sub(0, 0.6))
    board_ext = None
    if st.world == "board":
        st = refine_with_board(kf, st, cfg)
        undistort_keyframes(kf, st)
        board_ext = board_extent_part(cfg.board)
        lidar_align = segmod.plane_lidar_alignment(kf, st.T_wc, region=board_ext)
        q["scale_reference"] = "ChArUco board"
    else:
        lidar_align = None
        q["scale_reference"] = "iPhone LiDAR/ARKit (no board found)"
        res.warnings.append("No calibration board detected: absolute scale comes from the phone's tracking "
                            "(typically 0.5-2% error). Use the printed board or enter a caliper reference.")
    q.update({k: v for k, v in st.stats.items()})
    if cfg.use_board and st.world != "board" and st.stats.get("board_frames", 0) > 0:
        res.warnings.append(f"The board was recognised in only {st.stats['board_frames']} of {len(kf)} keyframes, so "
                            "it was not used. Keep the whole board in view and avoid glare on the paper.")
    P(0.8, "locating the part")
    info = segmod.coarse_part(kf, st.T_wc, st.world, board_ext, lidar_align)
    if st.world != "board":
        st.T_wc = np.array([info["T_pw"] @ T for T in st.T_wc])
    roi_min, roi_max = info["roi_min"], info["roi_max"]
    if "roi" in hooks:
        hooks["roi"](info)

    # ------------------------------------------------------------------ 3. photogrammetric refinement
    P.stage("photogrammetry")
    if cfg.feature_ba:
        st = feature_refine(kf, st, roi_min, roi_max, cfg, progress=P.sub(0, 0.95))
        q.update({k: v for k, v in st.stats.items() if k.startswith(("feature", "ba_", "poses_"))})
        if st.tracks is None and st.world != "board":
            res.warnings.append("Very few image features matched between frames, so the phone's own tracking was "
                                "used without photogrammetric refinement. More light, slower motion or the board help.")
    plane = (np.array([0, 0, 1.0]), 0.0)
    if st.world != "board" and st.tracks is not None and st.tracks.n_points > 100:
        # re-estimate the support plane from triangulated points (more precise than LiDAR), ignoring
        # everything over the part's footprint (a flat part's top must not be taken for the table)
        X = st.tracks.X
        cand = (np.abs(X[:, 2]) < 0.006) & (np.hypot(X[:, 0], X[:, 1]) < 0.25)
        fp_lo, fp_hi = info["roi_min"][:2] - 0.005, info["roi_max"][:2] + 0.005
        cand &= ~np.all((X[:, :2] > fp_lo) & (X[:, :2] < fp_hi), 1)
        cand = X[cand]
        n = None
        if len(cand) > 50:
            n, d, inl = ransac_plane(cand, 0.0008, 500, normal_hint=np.array([0, 0, 1.0]), max_angle_deg=10)
            if np.degrees(np.arccos(min(1.0, n[2]))) > 3.0 or abs(d) > 0.004:
                log.warning("photogrammetric table plane disagrees with LiDAR (%.1f deg, %.1f mm) - keeping LiDAR's",
                            np.degrees(np.arccos(min(1.0, n[2]))), d * 1000)
                n = None
        if n is not None:
            T_fix = np.eye(4)
            zax = n
            xax = np.cross([0, 1.0, 0], zax)
            xax /= np.linalg.norm(xax)
            yax = np.cross(zax, xax)
            T_fix[:3, :3] = np.stack([xax, yax, zax], 0)
            T_fix[:3, 3] = -T_fix[:3, :3] @ (n * d)
            st.T_wc = np.array([T_fix @ T for T in st.T_wc])
            st.tracks.X = transform_points(T_fix, X)
            if len(info["points"]):
                info["points"] = transform_points(T_fix, info["points"])
            info["look_at"] = transform_points(T_fix, info["look_at"][None])[0]
            q["support_plane_points"] = int(inl.sum())
    if "poses" in hooks:
        hooks["poses"](kf, st)
    if len(info["points"]) < 30:
        # LiDAR barely saw the part (small or flat): seed the silhouette prompts from tie points above
        # the support plane, or from a small disc where the cameras were pointing
        X = st.tracks.X if st.tracks is not None else np.zeros((0, 3))
        up = X[(X[:, 2] > 0.0008) & (X[:, 2] < 0.3) & np.all((X >= roi_min) & (X <= roi_max), 1)] if len(X) else X
        if len(up) >= 20:
            info["points"] = up
        else:
            c = info["look_at"]
            a = np.linspace(0, 2 * np.pi, 24, endpoint=False)
            info["points"] = np.concatenate([np.stack([c[0] + r * np.cos(a), c[1] + r * np.sin(a), np.full(24, z)], 1)
                                             for r in (0.004, 0.009) for z in (0.001, 0.004)])
        res.warnings.append("The LiDAR barely saw this part (it is small or flat); its outline was found from the "
                            "photos instead. Check the silhouette panel.")

    # ------------------------------------------------------------------ 4. silhouettes
    P.stage("silhouettes")
    from .geometry import upright_k
    for i, f in enumerate(kf):   # gravity-up direction in each image (the AI models prefer upright images)
        f.extra["upright_k"] = upright_k(inv_T(st.T_wc[i])[:3, :3] @ np.array([0, 0, 1.0]))
    seg = segmenter or get_segmenter(cfg, device)
    masks, scores = [], []
    for i, f in enumerate(kf):
        pr = segmod.make_prompts(f, st.T_wc[i], info["points"], info["look_at"])
        if pr is None:
            masks.append(None)
            scores.append(0.0)
            continue
        m, s = seg.segment(f, pr["coarse"], pr["box"], pr["pos"], pr["neg"])
        masks.append(m)
        scores.append(s)
        P(0.9 * i / len(kf), f"segmenting {i + 1}/{len(kf)}")
    trusted = getattr(seg, "trusted", False)
    q["segmenter"] = getattr(seg, "name", "custom")
    # quick silhouette-only hull to tighten the ROI and re-check the masks
    if trusted:
        hull_frames = (FusionFrame(inv_T(st.T_wc[i]), None, None, None, _sdt(m, kf[i]).astype(np.float16), kf[i].K)
                       for i, m in enumerate(masks) if m is not None)      # generator: one SDT at a time
        v_c = float(np.clip(np.max(roi_max - roi_min) / 70.0, 0.0005, 0.0015))   # coarse hull, scaled to the part
        hvol, _ = fuse(hull_frames, roi_min, roi_max, v_c, trunc=3 * v_c, k_carve=cfg.carve_min_views,
                       tol_px=2.0, close_bottom=True, device=device)
        del hull_frames
        solid = hvol < 0
        del hvol
        gc.collect()
        if solid.sum() > 20:
            idx = np.argwhere(solid)
            lo = roi_min + idx.min(0) * v_c
            hi = roi_min + (idx.max(0) + 1) * v_c
            pad = np.array([0.004, 0.004, 0.004])
            roi_min = np.maximum(roi_min, lo - pad)
            roi_max = np.minimum(roi_max, hi + pad)
            roi_min[2] = -0.002
            # IoU of every mask with the hull projection: flag silhouettes that disagree
            pts = (np.argwhere(solid)[:: max(1, int(solid.sum() // 20000))] + 0.5) * v_c + info["roi_min"]
            for i, m in enumerate(masks):
                if m is None:
                    continue
                hm = segmod.splat_mask(pts, kf[i].K, inv_T(st.T_wc[i]), kf[i].rgb_size, 0.9 * v_c)
                hm = cv2.morphologyEx(hm, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)) > 0
                iou = (hm & m).sum() / max(1, (hm | m).sum())
                scores[i] = float(iou)
        bad = [i for i, s in enumerate(scores) if masks[i] is not None and s < 0.7]
        for i in bad:
            masks[i] = None
        q["silhouettes_rejected"] = len(bad)
        if len(bad) > 0.2 * len(kf):
            res.warnings.append(f"{len(bad)} of {len(kf)} part outlines disagreed with the others and were ignored "
                                "(busy background, part touching other objects, or poor light?).")
        q["silhouette_iou_median"] = float(np.median([s for s in scores if s > 0])) if any(scores) else float("nan")
    q["roi_mm"] = [[float(v * 1000) for v in roi_min], [float(v * 1000) for v in roi_max]]
    if "masks" in hooks:
        hooks["masks"](kf, masks, st)

    # ------------------------------------------------------------------ 5. depth
    if "depth_inputs" in hooks:
        hooks["depth_inputs"](kf, st, masks, info)
    P.stage("depth")
    dm = depth_model or get_depth_model(cfg, device)
    q["depth_model"] = getattr(dm, "name", "custom")
    region = board_ext if st.world == "board" else (-0.15, 0.15, -0.15, 0.15)
    depths, align_stats = [], []
    for i, f in enumerate(kf):
        D, Kd = dm.predict(f)
        if cfg.align_depth:
            pm = masks[i]
            if pm is None:
                pr = segmod.make_prompts(f, st.T_wc[i], info["points"], info["look_at"])
                pm = None if pr is None else cv2.dilate(pr["coarse"], np.ones((25, 25), np.uint8)) > 0
            D, a = depthmod.align_depth(D, Kd, st.T_wc[i], f.K, i, st.tracks, plane, pm, region)
        else:
            a = dict(a=1.0, b=0.0, n=0, sigma_mm=float("nan"), ok=False)
        depths.append((D, Kd))
        align_stats.append(a)
        P(i / len(kf), f"depth {i + 1}/{len(kf)}")
    ok = [a for a in align_stats if a["ok"]]
    if ok:
        q["depth_scale_median"] = float(np.median([a["a"] for a in ok]))
        q["depth_offset_median_mm"] = float(np.median([a["b"] for a in ok]) * 1000)
        q["depth_fit_sigma_mm"] = float(np.median([a["sigma_mm"] for a in ok]))
        q["depth_relief_correction_median"] = float(np.median([a.get("c", 0.0) for a in ok]))
    q["depth_frames_corrected"] = len(ok)
    if cfg.align_depth and len(ok) < 0.8 * len(kf):
        res.warnings.append(f"Only {len(ok)} of {len(kf)} depth maps could be checked against the photogrammetry; "
                            "the others count less in the 3D model.")

    # ------------------------------------------------------------------ 6. fusion
    P.stage("fusion")
    voxel = auto_voxel(roi_min, roi_max, cfg.voxel_mm)
    def fusion_frames(keep_masks):
        """Build each frame's fusion inputs on demand (memory: only one frame's buffers at a time)."""
        for i, f in enumerate(kf):
            D, Kd = depths[i]
            h, w = D.shape
            # part AND background are fused (background = evidence of free space); skip mixed pixels
            mk = ~_depth_edges(D)
            mr = None
            if masks[i] is not None:
                mr = cv2.resize(masks[i].astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                mk &= ~_occluding_contours(D, mr, max(2, int(round(4 * w / 1920))))
            sdt = _sdt(masks[i], f).astype(np.float16) if (trusted and masks[i] is not None) else None
            # excluded pixels still prove that space in front of the near surface is empty
            free = _edge_free_depth(D, mk)
            # support-plane pixels away from the part: the plane is known exactly, so they only count as
            # free space (otherwise depth noise on the table becomes small solid blobs / a skirt)
            on_plane = _plane_pixels(D, Kd, st.T_wc[i], tol=0.0025)
            if mr is not None:
                on_plane &= cv2.dilate(mr, np.ones((7, 7), np.uint8)) == 0
            free = np.where(on_plane, np.maximum(D - 0.0003, 0), free).astype(np.float32)
            mk &= ~on_plane
            keep_masks.append(mk)
            # frames whose depth could not be checked against the photogrammetry count less
            fw = 1.0 if (not cfg.align_depth or align_stats[i]["ok"]) else 0.25
            yield FusionFrame(inv_T(st.T_wc[i]), D, Kd, mk, sdt, f.K, weight=fw,
                              weight_map=_angle_weight(D, Kd).astype(np.float16), free_depth=free)

    depth_masks = []
    fit_sig = q.get("depth_fit_sigma_mm", 0.5) / 1000.0
    trunc = float(np.clip(max(cfg.trunc_voxels * voxel, 3.0 * fit_sig), 0.001, 0.004))
    q["fusion_trunc_mm"] = trunc * 1000
    vol, fst = fuse(fusion_frames(depth_masks), roi_min, roi_max, voxel, trunc, cfg.carve_min_views,
                    cfg.carve_tolerance_px, cfg.close_bottom, device, progress=P.sub(0, 1),
                    min_views=cfg.fusion_min_views, trunc_back=max(3 * voxel, cfg.trunc_back_ratio * trunc))
    gc.collect()
    if device.startswith("cuda"):
        import torch
        torch.cuda.empty_cache()
    q.update({f"fusion_{k}": v for k, v in fst.items()})
    edge_solid = [(vol[s] < 0).mean() for s in ((0,), (-1,), (slice(None), 0), (slice(None), -1),
                                                (slice(None), slice(None), -1))]
    if max(edge_solid) > 0.002:
        res.warnings.append("The model touches the edge of the reconstruction box - part of it may be cut off. "
                            "Is the part much larger than the board, or not on the board?")

    # ------------------------------------------------------------------ 7. mesh + measurements
    P.stage("mesh")
    mesh = extract_mesh(vol, roi_min, voxel)
    if cfg.debug:
        mesh.export(os.path.join(work, "raw_marching_cubes.ply"))
        np.save(os.path.join(work, "sdf_volume.npy"), vol)
    # consistency of the depth data with the final surface
    errs = []
    for i in range(0, len(kf), max(1, len(kf) // 15)):
        D, Kd = depths[i]
        sel = depth_masks[i]
        if masks[i] is not None:
            sel = sel & (cv2.resize(masks[i].astype(np.uint8), (D.shape[1], D.shape[0]), interpolation=cv2.INTER_NEAREST) > 0)
        Pw, _ = backproject_depth(D, Kd, st.T_wc[i], sel, stride=3)
        if len(Pw):
            s = sample_sdf(vol, roi_min, voxel, Pw)
            errs.append(s[np.isfinite(s) & (np.abs(s) < 0.9 * trunc)])
    if errs:
        e = np.concatenate(errs)
        q["depth_to_surface_rms_mm"] = float(np.sqrt(np.mean(e ** 2)) * 1000) if len(e) else float("nan")
    mesh = clean_mesh(mesh, cfg.smooth_iterations, cfg.max_faces)
    ious = []
    try:
        m_small = mesh.simplify_quadric_decimation(face_count=40000) if len(mesh.faces) > 60000 else mesh
    except Exception:
        m_small = mesh
    ratios = []
    for i in range(len(kf)):
        if masks[i] is None:
            continue
        sil = segmod.mesh_silhouette(m_small.vertices, m_small.faces, kf[i].K, inv_T(st.T_wc[i]), kf[i].rgb_size)
        ious.append((sil & masks[i]).sum() / max(1, (sil | masks[i]).sum()))
        ratios.append(masks[i].sum() / max(1, sil.sum()))
    if ious:
        q["silhouette_iou_median"] = float(np.median(ious))
        q["silhouette_iou_p10"] = float(np.percentile(ious, 10))
        q["outline_to_model_area_ratio"] = float(np.median(ratios))
        if q["silhouette_iou_median"] < 0.9 or abs(q["outline_to_model_area_ratio"] - 1) > 0.05:
            what = ("smaller" if q["outline_to_model_area_ratio"] > 1 else "larger")
            res.warnings.append(f"The 3D model is noticeably {what} than the part's outline in the photos "
                                f"(agreement {q['silhouette_iou_median']:.2f}). The depth data was probably biased "
                                "(shiny, dark or very small part?). Do not trust the dimensions without a caliper "
                                "check; scanning spray and softer light usually fix this.")
    T_cp, rect = canonical_frame(mesh)
    mesh.apply_transform(T_cp)
    T_wc_c = np.array([T_cp @ T for T in st.T_wc])
    P(0.4, "colouring")
    views = [(kf[i].rgb, kf[i].K, inv_T(T_wc_c[i]), depths[i][0], depths[i][1]) for i in range(len(kf))]
    q["color_coverage"] = colorize(mesh, views)
    del views
    mesh_mm = mesh.copy()
    mesh_mm.apply_scale(1000.0)
    P(0.6, "measuring")
    sdf_vol_mm3 = _sdf_volume(vol, voxel)
    meas = measure_all(mesh_mm, cfg.material, sdf_volume_mm3=sdf_vol_mm3)
    # optional caliper reference
    ref = reference_check(meas, cfg)
    if ref is not None:
        meas["reference"] = ref
        if cfg.apply_reference_scale and ref.get("ratio") and not ref["name"].endswith("diameter") \
                and abs(ref["ratio"] - 1) <= 0.03:
            s = ref["ratio"]
            mesh.apply_scale(s)
            mesh_mm.apply_scale(s)
            meas = measure_all(mesh_mm, cfg.material, sdf_volume_mm3=sdf_vol_mm3 * s ** 3)
            ref["applied"] = True
            meas["reference"] = ref
            T_wc_c = T_wc_c.copy()
            T_wc_c[:, :3, 3] *= s          # keep the QA overlays consistent with the rescaled model
        elif cfg.apply_reference_scale:
            res.warnings.append("Rescaling skipped: only length/width/height can be used, and only for "
                                "corrections up to 3 % (larger differences mean a capture or entry problem).")
    res.measurements = meas
    rough = [c for c in meas.get("circles", []) if c["kind"] == "hole" and c.get("rough")]
    if rough:
        res.warnings.append(f"{len(rough)} hole(s) (axis {', '.join(sorted({c['axis'] for c in rough}))}) were only "
                            "partly captured: their outlines are ragged and the diameters (marked 'rough') are "
                            "uncertain and probably too small. Add views looking straight into them, or measure "
                            "them directly.")
    if "mesh" in hooks:
        hooks["mesh"](mesh_mm, meas)

    # ------------------------------------------------------------------ 8. export
    P.stage("export")
    from . import drawings
    from . import export as exportmod
    pre = {}
    try:   # the silhouette check needs the photos and masks: make it, then free all per-frame data
        pre["qa"] = drawings.qa_montage(kf, masks, T_wc_c, mesh, os.path.join(out_dir, "qa_silhouettes.jpg"))
    except Exception as ex:
        log.warning("QA montage failed: %s", ex)
    del depths, depth_masks, vol
    masks = None
    for f in scan.frames:
        f.rgb = f.depth = f.conf = f.rgb_bytes = None
        f.extra.clear()
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    q["seconds"] = round(time.time() - t_start, 1)
    res.files = exportmod.export_all(out_dir, mesh, mesh_mm, meas, q, res.warnings, cfg, qa=None,
                                     progress=P.sub(0, 1), extra_files=pre)
    res.seconds = time.time() - t_start
    if progress is not None:
        progress(1.0, "done")
    return res


def _sdf_volume(vol, voxel):
    """Solid volume [mm^3] straight from the fused SDF (partial-voxel occupancy)."""
    occ = np.clip(0.5 - vol / voxel, 0.0, 1.0)
    return float(occ.sum() * (voxel * 1000.0) ** 3)


REFERENCE_CHOICES = ["Length (X)", "Width (Y)", "Height (Z)", "Largest hole diameter", "Smallest hole diameter"]


def reference_check(meas: dict, cfg: Config):
    if not cfg.reference_mm or cfg.reference_mm <= 0 or not cfg.reference_name:
        return None
    from .measure import best_dims
    L, W, H, _ = best_dims(meas)
    holes = [c for c in meas.get("circles", []) if c["kind"] == "hole"]
    holes = [c for c in holes if not c.get("rough")] or holes     # compare with cleanly measured holes first
    name = cfg.reference_name
    val = {"Length (X)": L, "Width (Y)": W, "Height (Z)": H}.get(name)
    if name == "Largest hole diameter" and holes:
        val = max(h["diameter"] for h in holes)
    if name == "Smallest hole diameter" and holes:
        val = min(h["diameter"] for h in holes)
    if val is None:
        return dict(name=name, reference_mm=cfg.reference_mm, measured_mm=None, error_mm=None, ratio=None)
    return dict(name=name, reference_mm=float(cfg.reference_mm), measured_mm=float(val),
                error_mm=float(val - cfg.reference_mm), error_pct=float((val / cfg.reference_mm - 1) * 100),
                ratio=float(cfg.reference_mm / val), applied=False)


In [ ]:
%%writefile /content/partscan/app.py
"""Gradio interface (works inline in Colab and on a phone browser)."""
from __future__ import annotations

import datetime as _dt
import glob
import html
import logging
import os
import traceback

from .config import BOARDS, MATERIAL_DENSITY_G_CM3, PRESETS, SCALE_CHECKS, BoardSpec, board_from_check, make_config

log = logging.getLogger("partscan")

GUIDE = """
### What you need
* **iPhone 17 Pro / Pro Max** (or any iPhone / iPad **Pro** with LiDAR). The regular iPhone 17 and the iPhone Air have no LiDAR.
* The free app **Stray Scanner** (App Store). Record3D `.r3d`, Polycam *raw data* and 3D Scanner App *All Data* exports also work.
* The **PartScan board** (tab *2 · Board*): print at **100 %**, glue it flat to something rigid, then
  measure the printed **100 mm ruler** with a ruler or caliper and type what you measured into the *Print check* box.
  That single number sets the scale of everything.

### Prepare the part
* Place the part **near the middle of the board**, resting on its most stable face. Nothing else on the board.
* Shiny, black or transparent parts confuse every optical sensor: dust them lightly with **3D-scanning spray** or
  developer spray (or dry shampoo). Spray outdoors or with good ventilation, away from flames; wear a mask with powders.
  Coatings add roughly 10-50 µm - clean them off precision surfaces afterwards.
* Even, diffuse light (no hard shadows, no direct sun, no glare on the board). Don't move the part or the board while scanning.

### Record (40-90 s, slowly)
1. Keep the phone **30-40 cm** from the part (the LiDAR does not work much closer than ~25 cm).
2. **Low ring**: go all the way around at about table height + 10 cm, camera aimed at the part.
3. **Middle ring** all the way around at ~45°.
4. **High ring** at ~70°, then a slow **pass right over the top**, looking straight down (needed to measure holes).
5. Holes in the *sides*? Finish with a few slow views looking straight into each one, the phone level with the
   hole (rest it on the table if needed) - otherwise those holes come out too small and are marked *rough*.
6. Move slowly and smoothly - sharp frames matter more than many frames. Keep the part roughly centred and the board in view.

### Get the scan into this notebook
Stray Scanner → **Library** → your scan → **Share** → *Save to Files* → **Google Drive › PartScan**
(a zip or a folder both work). Then press **↻**, pick it under *…or a scan from Google Drive*, and **Build 3D model**.
On the free Colab tier the phone is for capturing; run the app here in the notebook (on a computer or the phone's browser).

### Reading the results
* **Length × width × height** are measured with the part as it rested on the board: Z is up from the board,
  X runs along the longest side of the footprint. *Between faces* values come from fitted planar faces (most precise);
  the *envelope* is the outer extent including scan noise.
* *Wall thickness / width* and *step* rows are distances between parallel faces.
* **Holes** are measured on cross-sections; the Ø is the median over the middle of the bore. A hole marked
  **rough** was only partly seen (typically a side hole filmed at an angle): its Ø is uncertain and usually too small.
* Always check one number with a caliper (*Caliper check*): the report then shows the real error of this scan.
"""

CSS = """
#ps-title h1{margin-bottom:0}
.ps-small{font-size:.9em;opacity:.8}
footer{display:none !important}
"""


def _is_scan_dir(d):
    try:
        names = set(os.listdir(d))
    except OSError:
        return False
    return ({"odometry.csv", "rgb.mp4"} <= names or ("metadata" in names and "rgbd" in names)
            or "keyframes" in names or any(n.startswith("frame_") and n.endswith(".json") for n in names))


def _list_drive(folder):
    """(label, path) for every scan in the Drive folder: zips, .r3d files and exported folders."""
    if not folder or not os.path.isdir(folder):
        return []
    items = []
    for root, dirs, files in os.walk(folder):
        if os.path.relpath(root, folder).count(os.sep) > 2 or "results" in os.path.relpath(root, folder).split(os.sep):
            dirs[:] = []
            continue
        for fn in files:
            if fn.lower().endswith((".zip", ".r3d")):
                items.append(os.path.join(root, fn))
        for dn in list(dirs):
            if _is_scan_dir(os.path.join(root, dn)):
                items.append(os.path.join(root, dn))
                dirs.remove(dn)
    items.sort(key=os.path.getmtime, reverse=True)
    out = []
    for p in items:
        when = _dt.datetime.fromtimestamp(os.path.getmtime(p)).strftime("%b %d %H:%M")
        out.append((f"{os.path.relpath(p, folder)}  ·  {when}", p))
    return out


BOARD_FILES = (("PartScan_board_Letter.pdf", "Letter", "Standard 8x6 (Letter / A4)", True),
               ("PartScan_board_Letter_no_text.pdf", "Letter", "Standard 8x6 (Letter / A4)", False),
               ("PartScan_board_A4.pdf", "A4", "Standard 8x6 (Letter / A4)", True),
               ("PartScan_board_A3.pdf", "A3", "Large 11x7 (A3 / Tabloid)", True),
               ("PartScan_board_Tabloid.pdf", "Tabloid", "Large 11x7 (A3 / Tabloid)", True))


def export_boards(folder: str) -> list:
    """Write every printable board PDF into `folder` (skipping the ones already there). Returns the paths."""
    from .board import render_board_pdf
    os.makedirs(folder, exist_ok=True)
    out = []
    for name, paper, board_name, text in BOARD_FILES:
        path = os.path.join(folder, name)
        if not os.path.exists(path):
            try:
                render_board_pdf(BoardSpec(**BOARDS[board_name]), path, paper=paper, text=text)
            except Exception as ex:
                log.warning("could not write %s: %s", name, ex)
                continue
        out.append(path)
    return out


def boards_folder_for(drive_dir: str) -> str:
    """Where the boards go on Google Drive: MyDrive/ShapeMeasure/Boards if that project folder exists,
    otherwise next to the scans in MyDrive/PartScan/Boards."""
    my_drive = os.path.dirname(os.path.abspath(drive_dir.rstrip("/")))
    shape = os.path.join(my_drive, "ShapeMeasure")
    return os.path.join(shape if os.path.isdir(shape) else drive_dir, "Boards")


def _viewer_iframe(path, height=560):
    if not path:
        return ""
    src = "/gradio_api/file=" + html.escape(os.path.abspath(path))
    return (f'<iframe src="{src}" style="width:100%;height:{height}px;border:1px solid rgba(127,127,127,.25);'
            f'border-radius:12px" allow="fullscreen"></iframe>'
            f'<div class="ps-small">Tap two points to measure. <a href="{src}" target="_blank">Open full screen</a></div>')


def _summary(res):
    from .export import measurement_rows  # noqa
    m = res.measurements
    o = m["overall"]
    q = res.quality
    holes = [c for c in m.get("circles", []) if c["kind"] == "hole"]
    from .measure import best_dims
    L, W, H, how = best_dims(m)
    lines = [f"### {L:.2f} × {W:.2f} × {H:.2f} mm  <span style='font-size:.6em;opacity:.7'>({how})</span>",
             f"Scale from **{q.get('scale_reference', '?')}** · {q.get('keyframes', '?')} keyframes · "
             f"{len(holes)} hole(s)" + (f" ({sum(1 for c in holes if c.get('rough'))} rough)"
                                        if any(c.get("rough") for c in holes) else "")
             + f" · {res.seconds / 60:.1f} min"]
    ref = m.get("reference")
    if ref and ref.get("measured_mm") is not None:
        lines.append(f"Caliper check ({ref['name']}): scan {ref['measured_mm']:.3f} mm vs yours "
                     f"{ref['reference_mm']:.3f} mm → **{ref['error_mm']:+.3f} mm** ({ref['error_pct']:+.2f} %)"
                     + (" - model rescaled to match." if ref.get("applied") else ""))
    for w in res.warnings:
        lines.append(f"> ⚠️ {w}")
    return "\n\n".join(lines)


def build_app(out_root: str = "/content/partscan_runs", drive_dir: str | None = None):
    import gradio as gr

    from . import pipeline
    from .board import render_board_pdf
    from .export import measurement_rows
    from .report import quality_rows

    os.makedirs(out_root, exist_ok=True)

    def run(upload, drive_file, preset, use_board, board_type, check_mm, check_on, material, ref_name, ref_val,
            ref_apply, progress=gr.Progress()):
        path = upload if upload else drive_file
        if not path:
            raise gr.Error("Upload a scan (.zip / .r3d) or pick one from Google Drive first.")
        try:
            board, note = board_from_check(board_type, float(check_mm or 0), check_on)
            cfg = make_config(preset, use_board=bool(use_board), board=board, material=material,
                              reference_name=ref_name or "", reference_mm=float(ref_val or 0),
                              apply_reference_scale=bool(ref_apply))
            stamp = _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
            run_dir = os.path.join(out_root, stamp)
            progress(0, desc="starting (the first run downloads the AI models, about 2 GB)")
            if not pipeline.RUN_LOCK.acquire(timeout=1.0):
                progress(0, desc="waiting for another scan to finish")
                pipeline.RUN_LOCK.acquire()
            try:
                res = pipeline.run(path, run_dir, cfg, progress=lambda p, m: progress(p, desc=m))
            finally:
                pipeline.RUN_LOCK.release()
            if note and use_board:
                res.warnings.insert(0, note)
            if drive_dir and res.files.get("zip"):
                try:   # keep a copy of the results next to the scans
                    import shutil
                    os.makedirs(os.path.join(drive_dir, "results"), exist_ok=True)
                    base = os.path.splitext(os.path.basename(str(path).rstrip("/")))[0]
                    shutil.copy(res.files["zip"], os.path.join(drive_dir, "results", f"{base}_{stamp}.zip"))
                except Exception as ex:
                    log.warning("could not copy results to Drive: %s", ex)
        except Exception as ex:
            log.error(traceback.format_exc())
            raise gr.Error(f"{type(ex).__name__}: {ex}", duration=None)
        f = res.files
        downloads = [f[k] for k in ("zip", "stl", "obj", "ply", "glb", "report", "viewer", "csv", "json")
                     if f.get(k) and os.path.exists(f[k])]
        rows = [list(r) for r in measurement_rows(res.measurements)]
        qrows = [[("🟢" if s == "good" else "🟠" if s == "ok" else "🔴" if s == "poor" else "⚪") + " " + a, str(b)]
                 for a, b, s in quality_rows(res.quality)]
        return (_summary(res), _viewer_iframe(f.get("viewer")), rows, f.get("drawing"), downloads, f.get("glb"),
                qrows, f.get("qa"))

    def refresh():
        return gr.update(choices=_list_drive(drive_dir), value=None)

    def make_board(paper):
        name = "Large 11x7 (A3 / Tabloid)" if paper in ("A3", "Tabloid") else "Standard 8x6 (Letter / A4)"
        spec, _ = board_from_check(name, 100.0)
        p = os.path.join(out_root, f"PartScan_board_{paper}.pdf")
        render_board_pdf(spec, p, paper)
        return p

    if drive_dir:
        try:
            folder = boards_folder_for(drive_dir)
            if export_boards(folder):
                log.info("printable boards are in %s", folder)
        except Exception as ex:
            log.warning("could not export the boards to Drive: %s", ex)

    with gr.Blocks(title="PartScan") as demo:
        gr.Markdown("# PartScan  ·  iPhone LiDAR → measured 3D model", elem_id="ps-title")
        gr.Markdown("Upload a scan of **one small part** on the printed board. You get a true-scale mesh "
                    "(STL/OBJ/PLY/GLB), dimensioned views, holes, wall thicknesses and an interactive measuring viewer.",
                    elem_classes="ps-small")
        with gr.Tab("1 · Scan"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=300):
                    upload = gr.File(label="Scan (.zip from Stray Scanner, .r3d, Polycam / 3D Scanner App zip)",
                                     file_types=[".zip", ".r3d"], type="filepath")
                    with gr.Row():
                        drive = gr.Dropdown(_list_drive(drive_dir), label="…or a scan from Google Drive"
                                            + (f" ({drive_dir})" if drive_dir else ""), interactive=True,
                                            visible=drive_dir is not None, scale=4)
                        rf = gr.Button("↻", visible=drive_dir is not None, scale=1, min_width=50)
                    preset = gr.Radio(list(PRESETS), value="Accurate (recommended)", label="Quality")
                    with gr.Accordion("Calibration board", open=True):
                        use_board = gr.Checkbox(True, label="Part is on the printed PartScan board")
                        board_type = gr.Radio(list(BOARDS), value="Standard 8x6 (Letter / A4)", label="Board")
                        check_mm = gr.Number(100.0, label="Print check: what you measured (mm)",
                                             info="Printers scale by up to ~1 %. Measure the printed 100 mm ruler.")
                        check_on = gr.Dropdown(list(SCALE_CHECKS), value="printed 100 mm ruler",
                                               label="Measured on")
                    with gr.Accordion("Caliper check (optional, recommended)", open=False):
                        ref_name = gr.Dropdown(pipeline.REFERENCE_CHOICES, label="Feature you measured by hand")
                        ref_val = gr.Number(0, label="Your measurement (mm)")
                        ref_apply = gr.Checkbox(False, label="Rescale the model so this feature matches exactly",
                                                info="Only length/width/height, and only for corrections up to 3 %.")
                    material = gr.Dropdown(list(MATERIAL_DENSITY_G_CM3), value="Steel", label="Material (for mass)")
                    go = gr.Button("Build 3D model", variant="primary", size="lg")
                with gr.Column(scale=2, min_width=320):
                    status = gr.Markdown("Results appear here.")
                    viewer = gr.HTML()
                    table = gr.Dataframe(headers=["Feature", "Value (mm)", "Note"], label="Measurements",
                                         wrap=True, interactive=False, column_widths=["34%", "16%", "50%"])
                    drawing = gr.Image(label="Dimensioned views", type="filepath", interactive=False)
                    files = gr.File(label="Downloads (zip = everything)", file_count="multiple", interactive=False)
                    with gr.Accordion("Model preview (Gradio viewer) & capture quality", open=False):
                        m3d = gr.Model3D(label="Mesh", clear_color=(0.96, 0.97, 0.98, 1.0), height=420)
                        quality = gr.Dataframe(headers=["Check", "Value"], label="Capture quality", interactive=False)
                        qa = gr.Image(label="Silhouette check (orange = photo outline, cyan = 3D model)",
                                      type="filepath", interactive=False)
            go.click(run, [upload, drive, preset, use_board, board_type, check_mm, check_on, material, ref_name,
                           ref_val, ref_apply], [status, viewer, table, drawing, files, m3d, quality, qa])
            rf.click(refresh, None, drive)
        with gr.Tab("2 · Board"):
            gr.Markdown("Print **at 100 % / Actual size** (not *Fit to page*) on matte paper, glue it flat to card, "
                        "glass or a clipboard, then measure the printed 100 mm ruler and enter the value on the first tab. "
                        "Letter / A4 give the standard 8×6 board (parts up to ~120 mm); A3 / Tabloid give the "
                        "large 11×7 board (parts up to ~200 mm) - select the matching board on the first tab.")
            paper = gr.Radio(["Letter", "A4", "A3", "Tabloid"], value="Letter", label="Paper")
            pb = gr.Button("Create board PDF")
            pdf = gr.File(label="Board PDF", interactive=False)
            pb.click(make_board, [paper], pdf)
        with gr.Tab("3 · Guide"):
            gr.Markdown(GUIDE)
    return demo


def launch(demo, out_root: str, share: bool = False, debug: bool = False, height: int = 1100, **kw):
    """Start the app. pwa=True makes it installable on a phone (Safari: Share -> Add to Home Screen)."""
    import gradio as gr
    theme = gr.themes.Soft(primary_hue="orange", neutral_hue="slate")
    return demo.launch(share=share, debug=debug, allowed_paths=[out_root], theme=theme, css=CSS,
                       height=height, show_error=True, pwa=True, **kw)


PHONE_STEPS = """**PartScan iPhone app (TestFlight):** open the *Server* tab, tap **Scan QR code** and point it at the code
below - that connects the app to this notebook. Then *Scan → Record → Upload & build*.

**Without the app (Safari):** scan the code with the Camera app, open the link, then *Share → Add to Home Screen*.
Record with **Stray Scanner** (the scan is a folder under *Files → On My iPhone → Stray Scanner*): move it to
*Google Drive › PartScan* and press ↻ in PartScan, or long-press it → **Compress** and upload the .zip (use Wi-Fi).

Either way keep this notebook tab open on the computer; a scan takes ~10 min on a T4.
"""


def phone_link(demo, out_dir: str, api_key: str | None = None):
    """QR code + instructions for using the running app from a phone. Returns (png_path or None, url or None).
    With an access key the QR carries it (?k=...): the PartScan iPhone app reads both from one scan."""
    url = getattr(demo, "share_url", None) or getattr(demo, "local_url", None)
    if not url:
        return None, None
    if api_key:
        url = url.rstrip("/") + "/?k=" + api_key
    png = None
    try:
        import qrcode
        os.makedirs(out_dir, exist_ok=True)
        png = os.path.join(out_dir, "phone_link_qr.png")
        qrcode.make(url).save(png)
    except Exception as ex:                               # qrcode not installed: the link alone still works
        log.info("no QR code (%s)", ex)
    return png, url


In [ ]:
%%writefile /content/partscan/api.py
"""Small REST API for the PartScan phone app, attached to the running Gradio server.

    POST /api/scans                     multipart: file (scan .zip / .r3d) + options  -> {jobId}
    GET  /api/jobs                      recent jobs
    GET  /api/jobs/{id}                 status, progress, and (when done) summary, rows, warnings, files
    GET  /api/jobs/{id}/files/{key}     a result file (zip, stl, obj, ply, glb, drawing, qa, csv, json, report, viewer)
    GET  /api/jobs/{id}/raw/{name}      any file of the result folder by name (lets report.html find its images)
    GET  /api/ping                      {ok, app, version, auth, device}

Authentication (optional): an access key checked on every /api route, given as the X-PartScan-Key
header, a ?k= query parameter, or the partscan_key cookie (set after the first keyed request so an
HTML report's images load too).
"""
from __future__ import annotations

import datetime
import json
import logging
import os
import queue
import secrets
import shutil
import threading
import time
import traceback
import uuid

# module level on purpose: with postponed annotations FastAPI resolves the endpoint
# signatures against this module's globals
from fastapi import File, Form, HTTPException, Request, UploadFile  # noqa: E402
from fastapi.responses import FileResponse, JSONResponse  # noqa: E402

log = logging.getLogger("partscan")
FILE_KEYS = ("zip", "stl", "obj", "ply", "glb", "drawing", "qa", "csv", "json", "report", "viewer")


class Job:
    def __init__(self, job_id: str, out_dir: str, options: dict):
        self.id = job_id
        self.dir = out_dir
        self.options = options
        self.status = "queued"          # queued | running | done | error
        self.progress = 0.0
        self.message = "waiting"
        self.error = None
        self.created = time.time()
        self.finished = None
        self.result = None              # dict, see _build_result

    def to_dict(self, full: bool = True) -> dict:
        d = dict(jobId=self.id, status=self.status, progress=round(float(self.progress), 4), message=self.message,
                 error=self.error, createdAt=datetime.datetime.fromtimestamp(self.created).isoformat(timespec="seconds"),
                 options=self.options)
        if self.result:
            d.update(self.result if full else dict(summary=self.result.get("summary")))
        return d


def _build_result(res, job_dir: str) -> dict:
    from .export import measurement_rows
    from .measure import best_dims
    from .report import quality_rows
    L, W, H, how = best_dims(res.measurements)
    holes = [c for c in res.measurements.get("circles", []) if c["kind"] == "hole"]
    files = {k: os.path.basename(v) for k, v in res.files.items() if k in FILE_KEYS and v and os.path.exists(v)}
    return dict(
        summary=dict(length=round(L, 3), width=round(W, 3), height=round(H, 3), how=how, holes=len(holes),
                     roughHoles=sum(1 for c in holes if c.get("rough")),
                     scaleReference=res.quality.get("scale_reference", ""), keyframes=res.quality.get("keyframes"),
                     seconds=round(float(res.seconds), 1)),
        rows=[[str(a), str(b), str(c)] for a, b, c in measurement_rows(res.measurements)],
        quality=[[str(a), str(b), str(s)] for a, b, s in quality_rows(res.quality)],
        warnings=list(res.warnings),
        files=files,
    )


class JobManager:
    """Runs reconstructions one after another on a background thread."""

    def __init__(self, out_root: str, run_fn=None):
        self.out_root = out_root
        self.jobs: dict[str, Job] = {}
        self.q: queue.Queue = queue.Queue()
        self.run_fn = run_fn                 # for tests: replaces pipeline.run
        os.makedirs(out_root, exist_ok=True)
        self._load_finished()
        self.thread = threading.Thread(target=self._worker, daemon=True, name="partscan-jobs")
        self.thread.start()

    # -- persistence of finished jobs (so the phone can list earlier results after a restart)
    def _load_finished(self):
        for name in sorted(os.listdir(self.out_root)):
            jf = os.path.join(self.out_root, name, "job.json")
            if os.path.exists(jf):
                try:
                    d = json.load(open(jf))
                    j = Job(d["jobId"], os.path.join(self.out_root, name), d.get("options", {}))
                    j.status, j.progress, j.message, j.error = d["status"], 1.0, d.get("message", ""), d.get("error")
                    j.created = d.get("created", os.path.getmtime(jf))
                    j.result = d.get("result")
                    self.jobs[j.id] = j
                except Exception as ex:
                    log.warning("could not load %s: %s", jf, ex)

    def _save(self, job: Job):
        with open(os.path.join(job.dir, "job.json"), "w") as fh:
            json.dump(dict(jobId=job.id, status=job.status, message=job.message, error=job.error, created=job.created,
                           options=job.options, result=job.result), fh)

    def submit(self, scan_path: str, options: dict) -> Job:
        job_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
        job_dir = os.path.join(self.out_root, job_id)
        os.makedirs(job_dir, exist_ok=True)
        job = Job(job_id, job_dir, options)
        job.scan_path = scan_path
        self.jobs[job_id] = job
        self.q.put(job)
        return job

    def _worker(self):
        while True:
            job = self.q.get()
            try:
                self._run(job)
            except Exception as ex:
                log.error(traceback.format_exc())
                job.status, job.error, job.message = "error", f"{type(ex).__name__}: {ex}", "failed"
            finally:
                job.finished = time.time()
                try:
                    self._save(job)
                except Exception as ex:
                    log.warning("could not save job: %s", ex)
                self.q.task_done()

    def _run(self, job: Job):
        from . import pipeline
        from .config import board_from_check, make_config
        o = job.options
        board, note = board_from_check(o.get("board_type", ""), float(o.get("check_mm") or 0), o.get("check_on", ""))
        cfg = make_config(o.get("preset", "Accurate (recommended)"), use_board=bool(o.get("use_board", True)),
                          board=board, material=o.get("material", "Steel"), reference_name=o.get("ref_name", "") or "",
                          reference_mm=float(o.get("ref_mm") or 0), apply_reference_scale=bool(o.get("ref_apply", False)))
        job.status, job.message = "running", "waiting for the GPU"

        def prog(p, m):
            job.progress, job.message = float(p), str(m)

        with pipeline.RUN_LOCK:
            job.message = "starting"
            res = (self.run_fn or pipeline.run)(job.scan_path, job.dir, cfg, progress=prog)
        if note and cfg.use_board:
            res.warnings.insert(0, note)
        job.result = _build_result(res, job.dir)
        job.status, job.progress, job.message = "done", 1.0, "done"
        try:
            os.remove(job.scan_path)        # the upload is big; the results folder keeps everything that matters
        except OSError:
            pass

    def list(self, limit: int = 50) -> list:
        jobs = sorted(self.jobs.values(), key=lambda j: j.created, reverse=True)[:limit]
        return [j.to_dict(full=False) for j in jobs]

    def delete(self, job_id: str) -> bool:
        job = self.jobs.pop(job_id, None)
        if job is None:
            return False
        shutil.rmtree(job.dir, ignore_errors=True)
        return True


def new_key() -> str:
    return secrets.token_urlsafe(9)


def attach_api(app, out_root: str, api_key: str | None = None, manager: JobManager | None = None) -> JobManager:
    """Add the /api routes to the FastAPI app that Gradio's launch() returned."""
    from . import __version__
    manager = manager or JobManager(out_root)
    api_root = os.path.join(out_root, "api")
    os.makedirs(api_root, exist_ok=True)

    def check(request: Request):
        if not api_key:
            return False
        got = (request.headers.get("x-partscan-key") or request.query_params.get("k")
               or request.cookies.get("partscan_key"))
        if got != api_key:
            raise HTTPException(status_code=401, detail="missing or wrong access key")
        return request.cookies.get("partscan_key") != api_key   # True -> set the cookie on the response

    def with_cookie(response, set_cookie: bool):
        if set_cookie and api_key:
            response.set_cookie("partscan_key", api_key, httponly=True, samesite="lax", max_age=30 * 86400)
        return response

    def get_job(job_id: str) -> Job:
        job = manager.jobs.get(job_id)
        if job is None:
            raise HTTPException(status_code=404, detail="no such job")
        return job

    @app.get("/api/ping")
    def ping(request: Request):
        sc = check(request)
        from . import pipeline
        try:
            device = pipeline.resolve_device("auto")
        except Exception:
            device = "?"
        return with_cookie(JSONResponse(dict(ok=True, app="PartScan", version=__version__, auth=bool(api_key),
                                             device=device, jobsQueued=manager.q.qsize())), sc)

    @app.post("/api/scans")
    async def create_scan(request: Request, file: UploadFile = File(...), preset: str = Form("Accurate (recommended)"),
                          use_board: str = Form("1"), board_type: str = Form("Standard 8x6 (Letter / A4)"),
                          check_mm: str = Form("100"), check_on: str = Form("printed 100 mm ruler"),
                          material: str = Form("Steel"), ref_name: str = Form(""), ref_mm: str = Form("0"),
                          ref_apply: str = Form("0"), client: str = Form("")):
        sc = check(request)
        name = os.path.basename(file.filename or "scan.zip")
        ext = os.path.splitext(name)[1].lower() or ".zip"
        if ext not in (".zip", ".r3d"):
            raise HTTPException(status_code=400, detail="upload a .zip (Stray Scanner / PartScan / Polycam / 3DSA) or .r3d")
        tmp = os.path.join(api_root, f"upload-{uuid.uuid4().hex}{ext}")
        size = 0
        with open(tmp, "wb") as out:
            while True:
                chunk = await file.read(1 << 20)
                if not chunk:
                    break
                out.write(chunk)
                size += len(chunk)
        if size < 1000:
            os.remove(tmp)
            raise HTTPException(status_code=400, detail="the upload is empty")
        truthy = lambda v: str(v).strip().lower() in ("1", "true", "yes", "on")
        options = dict(preset=preset, use_board=truthy(use_board), board_type=board_type, check_mm=check_mm,
                       check_on=check_on, material=material, ref_name=ref_name, ref_mm=ref_mm, ref_apply=truthy(ref_apply),
                       client=client, upload=name, uploadBytes=size)
        job = manager.submit(tmp, options)
        log.info("job %s queued: %s (%.1f MB)", job.id, name, size / 1e6)
        return with_cookie(JSONResponse(job.to_dict(), status_code=202), sc)

    @app.get("/api/jobs")
    def list_jobs(request: Request):
        sc = check(request)
        return with_cookie(JSONResponse(dict(jobs=manager.list())), sc)

    @app.get("/api/jobs/{job_id}")
    def job_status(job_id: str, request: Request):
        sc = check(request)
        return with_cookie(JSONResponse(get_job(job_id).to_dict()), sc)

    @app.delete("/api/jobs/{job_id}")
    def job_delete(job_id: str, request: Request):
        sc = check(request)
        job = get_job(job_id)
        if job.status in ("queued", "running"):
            raise HTTPException(status_code=409, detail="the job is still running")
        manager.delete(job_id)
        return with_cookie(JSONResponse(dict(ok=True)), sc)

    def serve(job: Job, name: str, set_cookie: bool):
        path = os.path.join(job.dir, name)
        if not name or "/" in name or "\\" in name or name.startswith(".") or not os.path.isfile(path):
            raise HTTPException(status_code=404, detail="no such file")
        media = {".html": "text/html", ".png": "image/png", ".jpg": "image/jpeg", ".json": "application/json",
                 ".csv": "text/csv", ".zip": "application/zip", ".glb": "model/gltf-binary",
                 ".stl": "model/stl", ".obj": "text/plain", ".ply": "application/octet-stream"}
        return with_cookie(FileResponse(path, media_type=media.get(os.path.splitext(name)[1].lower(),
                                                                   "application/octet-stream"),
                                        filename=None if name.endswith(".html") else name), set_cookie)

    @app.get("/api/jobs/{job_id}/files/{key}")
    def job_file(job_id: str, key: str, request: Request):
        sc = check(request)
        job = get_job(job_id)
        if not job.result or key not in job.result.get("files", {}):
            raise HTTPException(status_code=404, detail="no such file")
        return serve(job, job.result["files"][key], sc)

    @app.get("/api/jobs/{job_id}/raw/{name}")
    def job_raw(job_id: str, name: str, request: Request):
        sc = check(request)
        return serve(get_job(job_id), name, sc)

    log.info("PartScan API attached (%s)", "access key required" if api_key else "no access key")
    return manager


## 3 · Start PartScan
* **Google Drive** (recommended for phone scans): the cell mounts your Drive and creates `MyDrive/PartScan`.
  On the iPhone: Stray Scanner → Library → scan → **Share → Save to Files → Google Drive › PartScan**.
  Then press **↻** in the app and pick the scan.
* The app runs **inside this notebook**. `PUBLIC_LINK = True` additionally creates a temporary public link
  (`gradio.live`, lasts up to a week) and shows a **QR code** to open the app on the iPhone and add it to the home
  screen. Google's Colab FAQ disallows *"bypassing the notebook UI to interact primarily via a web UI"* on **free**
  runtimes (allowed with a positive compute-unit balance), so it is off by default. The runtime must stay alive:
  keep this notebook tab open on a computer (Colab disconnects idle free sessions after a while).


In [ ]:
#@title 3 · Start the app { display-mode: "form" }
USE_GOOGLE_DRIVE = True   #@param {type:"boolean"}
PUBLIC_LINK = False       #@param {type:"boolean"}
PRELOAD_AI_MODELS = True  #@param {type:"boolean"}
ACCESS_KEY = ""           #@param {type:"string"}
import logging, os, sys
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
for k in [k for k in sys.modules if k == "partscan" or k.startswith("partscan.")]:
    del sys.modules[k]                                   # pick up edits to the engine cells
OUT = "/content/partscan_runs"
drive_dir = None
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_dir = "/content/drive/MyDrive/PartScan"
        os.makedirs(drive_dir, exist_ok=True)
        print("Put scans in Google Drive > PartScan")
    except Exception as ex:
        print("Google Drive not mounted:", ex)
from partscan import pipeline
from partscan.config import make_config
if PRELOAD_AI_MODELS:
    dev = pipeline.resolve_device("auto")
    cfg0 = make_config()
    print("loading Prompt Depth Anything + SAM 2.1 on", dev, "(first time ≈1-2 min)...")
    pipeline.get_depth_model(cfg0, dev)
    pipeline.get_segmenter(cfg0, dev)
    print("models ready")
from partscan.app import build_app, launch, phone_link, PHONE_STEPS
from partscan.api import attach_api, new_key
demo = build_app(OUT, drive_dir)
app, local_url, share_url = launch(demo, OUT, share=PUBLIC_LINK)
key = ACCESS_KEY.strip() or new_key()          # protects the phone-app API; blank = a new random key each start
attach_api(app, OUT, key)
if PUBLIC_LINK:
    from IPython.display import Image, Markdown, display
    png, url = phone_link(demo, OUT, key)
    if url:
        display(Markdown(f"### Phone link: `{url}`  \n(server `{share_url}`, access key `{key}`)  \n" + PHONE_STEPS))
        if png:
            display(Image(png, width=220))
    else:
        print("No public link was created (see the messages above).")

## 4 · (Optional) Run without the app
Process a scan from a path and show the results right here. Handy for batches or for scripting.

In [ ]:
#@title 4 · Run on a file { display-mode: "form" }
SCAN_PATH = "/content/drive/MyDrive/PartScan/my_scan.zip"  #@param {type:"string"}
QUALITY = "Accurate (recommended)"  #@param ["Fast (fewer frames)", "Accurate (recommended)", "Max detail (slow)"]
BOARD = "Standard 8x6 (Letter / A4)"  #@param ["Standard 8x6 (Letter / A4)", "Large 11x7 (A3 / Tabloid)"]
SCALE_CHECK_MM = 100.0   #@param {type:"number"}
SCALE_CHECK_OF = "printed 100 mm ruler"  #@param ["printed 100 mm ruler", "6 squares between inner corners"]
ON_BOARD = True          #@param {type:"boolean"}
MATERIAL = "Steel"       #@param ["Steel", "Stainless steel", "Aluminium 6061", "Titanium Ti-6Al-4V", "Brass", "Copper", "Cast iron", "PLA", "ABS", "Nylon (PA12)", "POM / Delrin", "None"]
import os, datetime, pandas as pd
from IPython.display import display, Image, Markdown
from partscan import pipeline
from partscan.config import make_config, board_from_check
from partscan.export import measurement_rows
from partscan.measure import best_dims

def run_file():
    board, note = board_from_check(BOARD, SCALE_CHECK_MM, SCALE_CHECK_OF)
    if note:
        print("NOTE:", note)
    cfg = make_config(QUALITY, use_board=ON_BOARD, board=board, material=MATERIAL)
    run_dir = os.path.join("/content/partscan_runs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    last = [-1]
    def prog(p, m):
        if int(p * 20) != last[0]:
            last[0] = int(p * 20); print(f"{p:5.0%}  {m}")
    res = pipeline.run(SCAN_PATH, run_dir, cfg, progress=prog)
    L, W, H, how = best_dims(res.measurements)
    display(Markdown(f"### {L:.2f} × {W:.2f} × {H:.2f} mm ({how})"))
    for w in res.warnings:
        print("WARNING:", w)
    display(pd.DataFrame(measurement_rows(res.measurements), columns=["feature", "value (mm)", "note"]))
    display(Image(res.files["drawing"]))
    print("All files:", run_dir)
    try:
        from google.colab import files
        files.download(res.files["zip"])
    except Exception:
        pass
    return res

if os.path.exists(SCAN_PATH):
    res = run_file()
else:
    print(f"Set SCAN_PATH to your scan first (not found: {SCAN_PATH}). The app above does the same job.")

## Troubleshooting
**"No calibration board detected"** — keep the whole board in view in most frames and avoid glare (matte paper, soft
light). Without the board, scale comes from the phone's motion tracking (typically 0.5–2 % off): enter a caliper
reference and tick *Rescale*.

**Holes come out too small or closed** — the camera has to look *into* a hole to see its walls: add the slow pass
straight above the part, and look straight into holes on the sides. Holes under ~4 mm are at the limit of a phone.

**Bumpy, missing or cut-down surfaces** — shiny, black or transparent surfaces: use scanning spray. Move slowly (motion
blur), add light. Check the *Silhouette check* image and the *Depth vs final surface* row in the report.

**Very thin walls and fins (under ~1.5 mm on a part up to ~100 mm)** can come out too thick: depth noise sets the
thinnest wall the fusion can resolve.

**A hole is marked *rough*** — it was only seen at an angle (typical for holes in side walls), so its outline is
ragged and its Ø is too small. Rescan with a few views looking straight into it, or measure it directly.

**CUDA / RAM out of memory** — choose *Fast*, or lower `n_keyframes` / `promptda_size` in `partscan/config.py`.

## What accuracy to expect — honestly
The pipeline was tested end-to-end on **synthetic iPhone scans** of a known 60 × 40 × 30 mm L-bracket: 1920×1440
video, ARKit-like tracking with drift and a 1.2 % scale error, lens distortion, blurred / noisy / biased LiDAR, and
the capture described in the guide. The two AI models could not be run in the test environment, so they were
**simulated** by stand-ins with deliberately injected errors (depth: ±1 % scale and ±3 mm offset per frame, a smooth
±0.6 mm error field, noise, blurred edges; outlines: ±1 px jitter plus occasional gross mistakes). With the board:

| feature | true (mm) | scanned (mm) | error (mm) |
|---|---|---|---|
| length × width × height (between faces) | 60 × 40 × 30 | 59.97 × 39.97 × 29.94 | −0.03 / −0.03 / −0.06 |
| outer envelope (includes scan noise) | 60 × 40 × 30 | 60.47 × 40.41 × 30.07 | +0.47 / +0.41 / +0.07 |
| wall / base thickness | 6 / 8 | 5.97 / 7.97 | −0.03 / −0.03 |
| inside length / wall height above base | 54 / 22 | 53.99 / 21.95 | −0.01 / −0.05 |
| 2 × Ø8 vertical holes | 8 | 7.95 / 7.94 | −0.05 / −0.06 |
| Ø6 side hole, filmed from the rings only | 6 | ≈5.15, marked *rough* | −0.85 |
| Ø6 side hole, second scan with a few low views straight into it | 6 | 5.74 | −0.26 |
| volume | 23.51 cm³ | 23.23 cm³ | −1.2 % |
| whole surface vs. truth | | mean 0.06, 95 % within 0.16 | |

Without the board, sizes came out about 1 % too large (the phone's own scale error): enter one caliper value and
tick *Rescale*. When the depth stand-in was made to flatten the part's relief (by up to 6 %, a typical failure
on small, dark or shiny parts), the height came out 1.3 mm short and the volume 8.6 % low — and the report caught
it: the photo outlines were 7 % larger than the model (flagged red) and the app showed a warning.

So the geometry engine can reach a few hundredths to a tenth of a millimetre **when the AI models behave like the
stand-ins**. Real parts add reflections, surface finish, lighting and the real models' own errors; nothing here has
been validated on real captures yet. Treat **±0.2–0.5 mm on a 50–100 mm part** as a plausible target with the board,
and **check one dimension with a caliper on every part** (*Caliper check* shows the real error of that scan).
This is a measurement aid, not a replacement for a CMM.

## Hosting it yourself (permanent phone app)
The same app runs unchanged on a **Hugging Face Space** with a GPU, which gives it a permanent address (no notebook
to keep open, no expiring link): create a Space (SDK *Gradio*), upload the contents of `PartScan_HF_Space.zip`
(delivered with this notebook; it is just `partscan/` + `app.py` + `requirements.txt` + `README.md`), and pick
**Nvidia T4 small** under *Settings → Hardware* ($0.40 / h as of Oct 2026, billed only while running; set a
*sleep time* so it stops when idle — waking up takes 1–2 min). Make the Space **private** unless you want the whole
internet to use your GPU. On the phone: open the Space's URL in Safari → *Share → Add to Home Screen*.

**Licences.** Prompt Depth Anything weights: Apache-2.0 (trained on Hypersim / ScanNet++ / ARKitScenes — check those
dataset terms before commercial use). SAM 2.1: Apache-2.0. Stray Scanner: MIT. three.js: MIT.
